# Day 11 - 1교시: Docker 환경 구성

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- Kafka + Spark + Python 통합 환경을 Docker로 구성할 수 있다
- KRaft 모드로 Zookeeper 없이 Kafka를 실행할 수 있다
- Spark Master/Worker 아키텍처를 이해하고 설정할 수 있다
- Kafka UI와 Spark UI로 클러스터 상태를 모니터링할 수 있다

---

## 오늘의 시나리오: API 게이트웨이 모니터링

### 배경

우리 회사의 API 게이트웨이에서 발생하는 모든 요청/응답 이벤트를 실시간으로 모니터링해야 합니다.

### API 게이트웨이란?

```
┌─────────────────────────────────────────────────────────────────────────┐
│                           API Gateway (게이트웨이)                        │
│                                                                         │
│  클라이언트의 모든 요청이 거쳐가는 "정문"                                   │
│                                                                         │
│  역할:                                                                   │
│  - 인증/인가: 사용자가 올바른 권한을 가지고 있는지 확인                      │
│  - 라우팅: 요청을 적절한 백엔드 서비스로 전달                               │
│  - 속도 제한: 초당 요청 수 제한 (Rate Limiting)                           │
│  - 로깅: 모든 요청/응답 기록 ← 오늘 우리가 처리할 데이터                    │
│  - 캐싱: 자주 요청되는 데이터 캐시                                         │
└─────────────────────────────────────────────────────────────────────────┘

![What is an API GateWay?](https://substackcdn.com/image/fetch/$s_!_CtQ!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F6e41325f-2b6b-4f2a-8ca6-d622413ad64c_1692x720.png)

클라이언트 ───────▶ API Gateway ───────▶ 백엔드 서비스들
   │                    │                   │
   │                    │                   ├── 사용자 서비스
   │                    │                   ├── 주문 서비스
   │                    ▼                   ├── 상품 서비스
   │              [모든 요청 로깅]           └── 결제 서비스
   │                    │
   │                    ▼
   │           오늘 처리할 데이터!
   │           (request_id, endpoint, status_code, response_time...)
```

**왜 API 게이트웨이 모니터링이 중요한가?**

1. **에러 감지**: 500 에러가 갑자기 늘어나면 즉시 알아야 함
2. **성능 분석**: 어떤 API가 느린지, 병목은 어디인지 파악
3. **트래픽 패턴**: 시간대별 요청량 분석, 용량 계획
4. **보안 감시**: 비정상적인 요청 패턴 탐지

- **문제**: 초당 수천 건의 API 호출이 발생
- **목표**: 실시간으로 에러 감지, 응답시간 분석, 트래픽 패턴 파악

### 아키텍처 개요

```
API Gateway  →  Kafka (이벤트 수집)  →  Spark Streaming (실시간 처리)
                       ↓                           ↓
                  Kafka UI                    Spark UI
                  (모니터링)                   (작업 모니터링)
```

### 오늘 만들 환경

| 컴포넌트 | 역할 | 포트 |
|---------|------|------|
| Kafka | 이벤트 스트림 저장소 | 9092 |
| Kafka UI | 토픽/메시지 모니터링 | 8080 |
| Spark Master | 분산 처리 관리 | 7077, 8081 |
| Spark Worker | 실제 처리 수행 | 8082 |
| Python | Producer/Consumer/Spark 코드 실행 | - |

---

## Part 1: compose.yml 이해하기 (20분)

### 핵심 개념 1: KRaft 모드란?

Kafka는 클러스터를 관리하기 위해 메타데이터(토픽 정보, 파티션 위치 등)를 저장해야 합니다.
이 메타데이터를 어떻게 관리하느냐에 따라 두 가지 모드가 있습니다:

```
┌─────────────────────────────────────────────────────────────────────────┐
│ 기존 방식: Zookeeper 필요                                                │
│                                                                         │
│   ┌─────────┐       ┌───────────┐                                       │
│   │  Kafka  │ ◀────▶│ Zookeeper │                                       │
│   │ Broker  │       │           │                                       │
│   └─────────┘       └───────────┘                                       │
│                                                                         │
│   문제점:                                                                │
│   - 별도의 Zookeeper 클러스터 운영 필요                                    │
│   - 두 시스템 간 동기화 복잡성                                             │
│   - 장애 포인트 증가                                                      │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│ KRaft 모드: Zookeeper 불필요 (Kafka 3.0+)                                │
│                                                                         │
│   ┌─────────────────────────────────┐                                   │
│   │            Kafka                │                                   │
│   │   ┌─────────┐  ┌────────────┐  │                                   │
│   │   │ Broker  │  │ Controller │  │  ← 한 프로세스에서 모두 수행         │
│   │   │ (메시지 ) │  │ (메타데이터) │  │                                   │
│   │   └─────────┘  └────────────┘  │                                   │
│   └─────────────────────────────────┘                                   │
│                                                                         │
│   장점:                                                                  │
│   - 아키텍처 단순화 (컨테이너 1개 감소)                                    │
│   - 설정 간소화                                                          │
│   - 성능 향상 (불필요한 네트워크 통신 제거)                                 │
│   - 더 빠른 장애 복구                                                     │
└─────────────────────────────────────────────────────────────────────────┘
```

**KRaft의 핵심: Raft 합의 알고리즘**

- 여러 컨트롤러가 "누가 리더인지" 투표로 결정
- 리더가 죽으면 자동으로 새 리더 선출
- 모든 메타데이터 변경은 과반수 동의 필요

### 핵심 개념 2: Spark 클러스터 아키텍처

![](https://image.samsungsds.com/kr/insights/cluster_img2.jpg?queryString=20250214030334)

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        Spark 클러스터 구조                               │
│                                                                         │
│   ┌─────────────────────────────────────────┐                          │
│   │           Driver Program                │                          │
│   │   (우리가 작성하는 PySpark 코드)             │                          │
│   │                                         │                          │
│   │   spark = SparkSession.builder...       │                          │
│   │   df = spark.read.csv(...)              │                          │
│   └────────────────┬────────────────────────┘                          │
│                    │ 작업 제출                                           │
│                    ▼                                                    │
│   ┌─────────────────────────────────────────┐                          │
│   │          Spark Master (:7077)           │                          │
│   │                                         │                          │
│   │   - Worker 등록 및 관리                    │                          │
│   │   - 리소스(CPU, 메모리) 할당                 │                          │
│   │   - 작업 스케줄링                          │                          │
│   │   - Web UI 제공 (:8081)                  │                          │
│   └────────────────┬────────────────────────┘                          │
│                    │ Task 분배                                           │
│                    ▼                                                    │
│   ┌─────────────────────────────────────────┐                          │
│   │          Spark Worker (:8082)           │                          │
│   │                                         │                          │
│   │   - 실제 데이터 처리 수행                 │                          │
│   │   - Executor 프로세스 실행               │                          │
│   │   - Task: 가장 작은 작업 단위             │                          │
│   │   - 결과를 Driver에게 반환               │                          │
│   └─────────────────────────────────────────┘                          │
└─────────────────────────────────────────────────────────────────────────┘
```

**핵심 용어 정리**

| 용어 | 설명 | 비유 |
|------|------|------|
| Driver | 우리 코드가 실행되는 곳 | 건축 설계사 |
| Master | 작업을 분배하는 관리자 | 현장 소장 |
| Worker | 실제 처리를 수행하는 노드 | 작업자 |
| Executor | Worker 안에서 실제 연산 수행 | 작업자의 손 |
| Task | 가장 작은 처리 단위 | 벽돌 하나 쌓기 |

### 핵심 개념 3: 포트 매핑 이해하기

```yaml
ports:
  - "8081:8080"
  # ↑ 호스트 포트 : ↑ 컨테이너 포트
```

```
┌───────────────────────────────────────────────────────────────┐
│ 호스트 (내 컴퓨터)                                              │
│                                                               │
│   브라우저에서 localhost:8081 접속                               │
│         │                                                     │
│         ▼                                                     │
│   ┌─────────────────────────────────────────────────────┐    │
│   │ Docker 네트워크                                       │    │
│   │                                                       │    │
│   │   8081 ─────────────▶ 8080                           │    │
│   │   (외부 접근)          (컨테이너 내부)                   │    │
│   │                       ┌─────────────────┐            │    │
│   │                       │  Spark Master    │            │    │
│   │                       │  (8080 포트에서   │            │    │
│   │                       │   UI 실행 중)    │            │    │
│   │                       └─────────────────┘            │    │
│   └─────────────────────────────────────────────────────┘    │
└───────────────────────────────────────────────────────────────┘
```

**왜 다른 포트를 쓰나요?**

- Kafka UI도 8080 포트 사용
- Spark Master UI도 8080 포트 사용
- 호스트에서는 같은 포트를 두 번 쓸 수 없음
- 해결: 호스트 포트를 다르게 매핑 (8080 → Kafka UI, 8081 → Spark Master)

---

### Step 1: 빈칸 채우기

`docker/compose-with-blanks.yml` 파일을 열어서 빈칸을 채워보세요.

**TODO 항목**:

1. **KAFKA_LISTENERS**: Kafka 리스너 설정
2. **KAFKA_ADVERTISED_LISTENERS**: 외부 접근용 주소

<details>
<summary>TODO 1 힌트: KAFKA_LISTENERS</summary>

리스너는 Kafka가 연결을 받아들이는 "문"입니다.

- PLAINTEXT: 클라이언트(Producer/Consumer)용 - 9092 포트
- CONTROLLER: KRaft 컨트롤러 통신용 - 9093 포트
- 0.0.0.0: 모든 네트워크 인터페이스에서 접근 허용

형식: `PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093`

</details>

<details>
<summary>TODO 1 정답</summary>

```yaml
KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
```

</details>

<details>
<summary>TODO 2 힌트: KAFKA_ADVERTISED_LISTENERS</summary>

클라이언트에게 "이 주소로 접속하세요"라고 알려주는 설정입니다.

- Docker 네트워크 내에서는 컨테이너 이름으로 접근
- 컨테이너 이름이 `kafka`이고, 클라이언트 포트가 9092

형식: `PLAINTEXT://kafka:9092`

</details>

<details>
<summary>TODO 2 정답</summary>

```yaml
KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:9092
```

</details>

### Step 2: 완성된 compose.yml 확인

빈칸을 채운 후, `docker/compose.yml`과 비교해보세요.

**주요 설정 설명**:

```yaml
# Kafka 리스너 설정
KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
#                ↑ 클라이언트용(9092)      ↑ 컨트롤러용(9093)

KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:9092
#                                      ↑ 다른 컨테이너가 접근할 주소

# Spark Worker가 Master에 연결
SPARK_MASTER_URL: spark://spark-master:7077
#                        ↑ 서비스 이름  ↑ 포트
```

---

## Part 2: 환경 구동 (20분)

### Step 1: Docker 이미지 빌드 및 컨테이너 시작

In [ ]:
# 터미널에서 실행할 명령어들 (이 셀은 설명용)

# 1. docker 디렉토리로 이동
cd docker

# 2. Python 이미지 빌드 (처음 한 번만)
# docker compose build

# 3. 모든 서비스 시작
# docker compose up -d

# 4. 컨테이너 상태 확인
# docker compose ps

**예상 출력**:

```
NAME          IMAGE                           STATUS                   PORTS
kafka         apache/kafka:4.1.1              Up (healthy)             9092-9093/tcp
kafka-ui      provectuslabs/kafka-ui:latest   Up                       0.0.0.0:8080->8080/tcp
spark-master  apache/spark:3.5.8              Up                       0.0.0.0:4040->4040/tcp, ...
spark-worker  apache/spark:3.5.8              Up                       0.0.0.0:8082->8081/tcp
python-dev    day11-python                    Up
```

### 문제 해결

**Kafka가 unhealthy 상태라면?**
```bash
# Kafka 로그 확인
docker compose logs kafka

# 재시작
docker compose restart kafka
```

**포트 충돌 에러가 발생하면?**
```bash
# 사용 중인 포트 확인 (예: 8080)
lsof -i :8080

# 해당 프로세스 종료 또는 compose.yml에서 포트 변경
```

### Step 2: UI 접속 확인

#### Kafka UI 확인

1. 브라우저에서 http://localhost:8080 접속
2. 왼쪽 메뉴에서 `day11-cluster` 확인
3. `Brokers` 탭에서 브로커 1대 확인

**확인 포인트**:
- Broker ID: 1
- State: Active Controller
- Listeners: PLAINTEXT://kafka:9092

#### Spark UI 확인

1. 브라우저에서 http://localhost:8081 접속 (Master UI)
2. `Workers` 섹션에서 Worker 1대 확인
   - Cores: 2
   - Memory: 2.0 GB

3. http://localhost:8082 접속 (Worker UI)
   - Master URL: spark://spark-master:7077
   - 실행 중인 Executor 정보

---

## Part 3: 연결 테스트 (20분)

### Kafka 연결 테스트

In [ ]:
# Python 컨테이너에서 실행
# docker compose exec python python

# Kafka 연결 테스트
from confluent_kafka.admin import AdminClient

admin = AdminClient({"bootstrap.servers": "kafka:9092"})

# 클러스터 메타데이터 확인
metadata = admin.list_topics(timeout=10)

print("Kafka 클러스터 연결 성공!")
print(f"브로커 수: {len(metadata.brokers)}")
print(f"토픽 목록: {list(metadata.topics.keys())}")

**예상 출력**:

```
Kafka 클러스터 연결 성공!
브로커 수: 1
토픽 목록: [] # or ['__consumer_offsets']
```

### 테스트 토픽 생성

In [ ]:
# 테스트용 토픽 생성
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient({"bootstrap.servers": "kafka:9092"})

# 토픽 설정: 4개 파티션, 복제 1
topic = NewTopic(topic="api-events", num_partitions=4, replication_factor=1)

# 토픽 생성
fs = admin.create_topics([topic])

# 결과 확인
for topic_name, f in fs.items():
    try:
        f.result()  # 완료 대기
        print(f"토픽 '{topic_name}' 생성 완료!")
    except Exception as e:
        print(f"토픽 '{topic_name}' 생성 실패: {e}")

# 토픽 목록 확인
metadata = admin.list_topics(timeout=10)
print(
    f"\n현재 토픽 목록: {[t for t in metadata.topics.keys() if not t.startswith('_')]}"
)

**예상 출력**:

```
토픽 'api-events' 생성 완료!

현재 토픽 목록: ['api-events']
```

**Kafka UI에서 확인**:
1. http://localhost:8080 → Topics
2. `api-events` 클릭
3. Partitions: 4개 확인

### Spark 연결 테스트

**Standalone 클러스터란?**

Spark가 자체적으로 제공하는 가장 간단한 클러스터 매니저입니다.

| 클러스터 매니저 | 설명 |
|----------------|------|
| **Standalone** | Spark 자체 내장 (별도 설치 불필요) |
| YARN | Hadoop 클러스터 사용 |
| Kubernetes | K8s 클러스터 사용 |
| Mesos | Apache Mesos 사용 |

- Spark에 기본 포함되어 있어 **별도 설치 없이** 바로 사용 가능
- Master + Worker 구조로 단순함
- 학습/개발/소규모 환경에 적합

In [ ]:
# PySpark 연결 테스트
from pyspark.sql import SparkSession

# SparkSession: Spark의 모든 기능을 사용하기 위한 진입점
spark = (
    SparkSession.builder.appName(
        "ConnectionTest"
    )  # Spark UI에 표시될 애플리케이션 이름
    .master("spark://spark-master:7077")  # Standalone 클러스터 Master 주소
    .config("spark.executor.memory", "1g")  # Executor당 메모리
    .config("spark.executor.cores", "1")  # Executor당 CPU 코어
    .getOrCreate()  # 기존 세션이 있으면 재사용, 없으면 생성
)

# sudo sh -c 'echo "127.0.0.1 python-dev" >> /etc/hosts'로 가능

# 연결 확인
print("Spark 클러스터 연결 성공!")
print(f"Spark 버전: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"App ID: {spark.sparkContext.applicationId}")

# 간단한 테스트: 0~999까지 숫자 DataFrame 생성
df = spark.range(1000)
print(f"\n테스트 DataFrame 생성: {df.count()} rows")

# 세션 종료 (리소스 반환)
spark.stop()
print("\nSpark 세션 종료")

**예상 출력**:

```
Spark 클러스터 연결 성공!
Spark 버전: 3.5.8
Master: spark://spark-master:7077
App ID: app-20260118-...

테스트 DataFrame 생성: 1000 rows

Spark 세션 종료
```

**Spark UI에서 확인** (http://localhost:8081):
- Running Applications 또는 Completed Applications 섹션
- `ConnectionTest` 애플리케이션 확인

---

## Step 3: 지시사항 - Kafka UI 서비스 직접 추가하기

### 과제

`compose-with-blanks.yml`에는 Kafka UI 서비스가 빠져 있습니다.
다음 조건에 맞게 Kafka UI 서비스를 추가해보세요.

**요구사항**:
1. 이미지: `provectuslabs/kafka-ui:latest`
2. 포트: 8080
3. 환경변수:
   - `KAFKA_CLUSTERS_0_NAME`: 클러스터 이름
   - `KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS`: Kafka 주소
4. depends_on: kafka가 healthy일 때 시작

<details>
<summary>힌트 보기</summary>

```yaml
kafka-ui:
  image: _____
  container_name: kafka-ui
  ports:
    - "_____:8080"
  environment:
    KAFKA_CLUSTERS_0_NAME: _____
    KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: _____
  depends_on:
    kafka:
      condition: _____
```

- 이미지: `provectuslabs/kafka-ui:latest`
- 포트: 호스트 8080 → 컨테이너 8080
- 클러스터 이름: 원하는 이름 (예: day11-cluster)
- 브로커 주소: kafka:9092
- condition: service_healthy

</details>

<details>
<summary>정답 보기</summary>

```yaml
kafka-ui:
  image: provectuslabs/kafka-ui:latest
  container_name: kafka-ui
  ports:
    - "8080:8080"
  environment:
    KAFKA_CLUSTERS_0_NAME: day11-cluster
    KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092
  depends_on:
    kafka:
      condition: service_healthy
```

**설명**:
- `KAFKA_CLUSTERS_0_NAME`: UI에서 표시될 클러스터 이름
- `KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS`: Kafka 브로커 주소 (Docker 네트워크 내부 주소)
- `service_healthy`: Kafka의 healthcheck가 통과한 후에 시작

</details>

---

## 핵심 요약

### 오늘 구성한 환경

| 컴포넌트 | 이미지 | 포트 | 역할 |
|---------|-------|------|------|
| Kafka | apache/kafka:4.1.1 | 9092 | 이벤트 스트림 |
| Kafka UI | provectuslabs/kafka-ui | 8080 | 모니터링 |
| Spark Master | apache/spark:3.5.8 | 7077, 8081 | 작업 관리 |
| Spark Worker | apache/spark:3.5.8 | 8082 | 작업 실행 |
| Python | custom | - | 코드 실행 |

### 핵심 명령어

```bash
# 환경 시작
docker compose up -d

# 상태 확인
docker compose ps

# 로그 확인
docker compose logs -f [서비스명]

# Python 컨테이너 접속
docker compose exec python bash

# 환경 종료
docker compose down

# 볼륨까지 삭제 (데이터 초기화)
docker compose down -v
```

### UI 접속 주소

- **Kafka UI**: http://localhost:8080
- **Spark Master UI**: http://localhost:8081
- **Spark Worker UI**: http://localhost:8082
- **Spark App UI**: http://localhost:4040 (앱 실행 중에만)

---

## 다음 시간 예고

**2교시: Kafka Producer/Consumer**

- API 이벤트 데이터 생성 및 전송
- 파티션별 처리량 측정
- Consumer Group으로 병렬 처리

---


# Day 11 - 2교시: Kafka Producer/Consumer

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- API 이벤트 데이터를 Kafka로 전송하는 Producer를 구현할 수 있다
- Consumer로 메시지를 수신하고 처리할 수 있다
- 파티션 수에 따른 처리량 변화를 측정할 수 있다
- Consumer Group을 활용한 병렬 처리를 이해할 수 있다

---

## 핵심 개념: 메시지 브로커란?

![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Producers_1.png?w=2500&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=1131c8739311c9061a659d275f070f43)
![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Consumers_1.png?w=2500&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=cbac2f51d00a470957afa28b2a0c3d03)

```

### 직접 연결 vs Kafka

```
┌─────────────────────────────────────────────────────────────────────────┐
│ 직접 연결 방식의 문제점                                                   │
│                                                                         │
│   Producer ─────────────────▶ Consumer                                  │
│                                                                         │
│   문제:                                                                  │
│   - Consumer가 죽으면? → 데이터 유실!                                    │
│   - Consumer가 느리면? → Producer가 대기해야 함                           │
│   - Consumer가 여러 개 필요하면? → 복잡해짐                               │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│ Kafka를 사용하면                                                        │
│                                                                         │
│   Producer ───▶ Kafka ───▶ Consumer 1                                   │
│                   │                                                     │
│                   └───────▶ Consumer 2                                   │
│                   │                                                     │
│                   └───────▶ Consumer 3                                   │
│                                                                         │
│   해결:                                                                  │
│   - Consumer가 죽어도 Kafka가 데이터 보관 (영속성)                        │
│   - Producer/Consumer 독립적 확장 (디커플링)                             │
│   - 여러 Consumer가 병렬로 처리 가능 (확장성)                             │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## 핵심 개념: 파티션(Partition)이란?

### 파티션 = 데이터를 나누는 상자

```

![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Producers_1.png?w=2500&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=1131c8739311c9061a659d275f070f43)


---

## 핵심 개념: Consumer Group

![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Consumer_Groups_1.png?w=2500&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=d42f57dd9b2e23b53a1c337323caef95)

```

---

## 핵심 개념: 오프셋(Offset)

### 오프셋 = "어디까지 읽었는지" 기록

```
![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/Kafka_Topics_1.png?w=2500&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=8b71698f46a5dd2a5bcdbce66414e0f1)

```

---

## 핵심 개념: 직렬화/역직렬화

### 왜 필요한가?

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    직렬화 (Serialization)                                │
│                                                                         │
│   Python 객체 (딕셔너리)          →      바이트 (bytes)                  │
│   {"user_id": "U001",             →      b'{"user_id":"U001",...}'       │
│    "status": 200}                                                       │
│                                                                         │
│   Kafka는 바이트만 저장/전송!                                             │
│   Python 객체를 그대로 보낼 수 없음                                       │
│                                                                         │
│   Producer에서:                                                          │
│   json.dumps(event).encode("utf-8")  # 딕셔너리 → JSON 문자열 → 바이트   │
│                                                                         │
│   Consumer에서:                                                          │
│   json.loads(msg.value().decode("utf-8"))  # 바이트 → JSON 문자열 → 딕셔너리 │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## 시나리오: API 게이트웨이 이벤트 스트림

### 데이터 구조

API 게이트웨이에서 발생하는 모든 요청을 다음 형식으로 기록합니다:

```json
{
  "request_id": "REQ_001",
  "user_id": "U123",
  "endpoint": "/api/products",
  "method": "GET",
  "status_code": 200,
  "response_time_ms": 45,
  "timestamp": "2026-01-18T10:30:00"
}
```

### 측정 항목

| 지표 | 설명 | 목표 |
|------|------|------|
| Throughput | 초당 처리 건수 (records/sec) | 파티션 수에 따른 변화 확인 |
| Latency | 메시지 처리 시간 | 1초 미만 |
| Consumer Lag | 처리 지연 건수 | Kafka UI에서 모니터링 |

---

## Part 1: Producer 구현 (20분)

### Step 1: 빈칸 채우기

다음 코드의 `_____` 부분을 채워보세요.

In [ ]:
# Step 1: Producer 빈칸 채우기
from confluent_kafka import Producer
import json
import random
from datetime import datetime

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
config = {
    # TODO 1: Kafka 브로커 주소를 설정하세요
    # 힌트: Docker 서비스 이름과 포트
    "bootstrap.servers": "_____",
    "client.id": "api-event-producer",
}

# TODO 2: Producer 객체 생성
producer = _____

# -----------------------------------------------------------------------------
# API 이벤트 생성 함수
# -----------------------------------------------------------------------------
ENDPOINTS = [
    "/api/products",
    "/api/users",
    "/api/orders",
    "/api/payments",
    "/api/search",
]
METHODS = ["GET", "POST", "PUT", "DELETE"]
STATUS_CODES = [200, 200, 200, 200, 201, 400, 404, 500]  # 200이 더 자주 발생


def generate_api_event():
    """API 이벤트 데이터 생성"""
    return {
        "request_id": f"REQ_{random.randint(1, 999999):06d}",
        "user_id": f"U{random.randint(1, 1000):04d}",
        "endpoint": random.choice(ENDPOINTS),
        "method": random.choice(METHODS),
        "status_code": random.choice(STATUS_CODES),
        "response_time_ms": random.randint(10, 500),
        "timestamp": datetime.now().isoformat(),
    }


# -----------------------------------------------------------------------------
# 메시지 전송
# -----------------------------------------------------------------------------
# TODO 3: 토픽 이름 설정
TOPIC = "_____"

for i in range(10):
    event = generate_api_event()

    # TODO 4: 메시지 전송 (topic, value 파라미터 사용)
    producer.produce(
        topic=_____,
        value=json.dumps(event).encode("utf-8"),
    )
    print(f"전송: {event['request_id']} - {event['endpoint']}")

# TODO 5: 버퍼의 모든 메시지를 전송하고 대기
producer._____()
print("전송 완료!")

<details>
<summary>TODO 1 힌트</summary>

Docker Compose에서 Kafka 서비스 이름이 `kafka`이고, 기본 포트는 9092입니다.
형식: `호스트:포트`

</details>

<details>
<summary>TODO 1 정답</summary>

```python
"bootstrap.servers": "kafka:9092",
```

</details>

<details>
<summary>TODO 2 힌트</summary>

confluent_kafka 라이브러리에서 Producer 클래스를 사용합니다.
config 딕셔너리를 인자로 전달합니다.

</details>

<details>
<summary>TODO 2 정답</summary>

```python
producer = Producer(config)
```

</details>

<details>
<summary>TODO 3 힌트</summary>

1교시에서 생성한 토픽 이름을 사용합니다.
API 이벤트를 저장하는 토픽입니다.

</details>

<details>
<summary>TODO 3 정답</summary>

```python
TOPIC = "api-events"
```

</details>

<details>
<summary>TODO 4 힌트</summary>

produce() 메서드의 topic 파라미터에 토픽 이름을 전달합니다.
위에서 정의한 TOPIC 변수를 사용하세요.

</details>

<details>
<summary>TODO 4 정답</summary>

```python
producer.produce(
    topic=TOPIC,
    value=json.dumps(event).encode("utf-8"),
)
```

</details>

<details>
<summary>TODO 5 힌트</summary>

Producer는 메시지를 버퍼에 모아서 일괄 전송합니다.
모든 메시지를 즉시 전송하고 완료를 대기하는 메서드입니다.

</details>

<details>
<summary>TODO 5 정답</summary>

```python
producer.flush()
```

</details>

### Step 2: 완성된 Producer

In [ ]:
# Step 2: 완성된 Producer
from confluent_kafka import Producer
import json
import random
from datetime import datetime
import time

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
config = {
    "bootstrap.servers": "kafka:9092",
    "client.id": "api-event-producer",
}

producer = Producer(config)

# -----------------------------------------------------------------------------
# API 이벤트 생성 함수
# -----------------------------------------------------------------------------
ENDPOINTS = [
    "/api/products",
    "/api/users",
    "/api/orders",
    "/api/payments",
    "/api/search",
]
METHODS = ["GET", "POST", "PUT", "DELETE"]
STATUS_CODES = [200, 200, 200, 200, 201, 400, 404, 500]


def generate_api_event():
    """API 이벤트 데이터 생성"""
    return {
        "request_id": f"REQ_{random.randint(1, 999999):06d}",
        "user_id": f"U{random.randint(1, 1000):04d}",
        "endpoint": random.choice(ENDPOINTS),
        "method": random.choice(METHODS),
        "status_code": random.choice(STATUS_CODES),
        "response_time_ms": random.randint(10, 500),
        "timestamp": datetime.now().isoformat(),
    }


# -----------------------------------------------------------------------------
# Delivery Callback (전송 결과 확인)
# -----------------------------------------------------------------------------
sent_count = 0


def delivery_callback(err, msg):
    global sent_count
    if err:
        print(f"전송 실패: {err}")
    else:
        sent_count += 1


# -----------------------------------------------------------------------------
# 메시지 대량 전송
# -----------------------------------------------------------------------------
TOPIC = "api-events"
NUM_MESSAGES = 1000

print(f"API 이벤트 {NUM_MESSAGES}건 전송 시작...")
start_time = time.time()

for i in range(NUM_MESSAGES):
    event = generate_api_event()

    producer.produce(
        topic=TOPIC,
        value=json.dumps(event).encode("utf-8"),
        callback=delivery_callback,
    )

    # 1000건마다 진행 상황 출력
    if (i + 1) % 1000 == 0:
        producer.flush()
        print(f"  {i + 1}건 전송 완료")

producer.flush()
elapsed = time.time() - start_time

print("\n전송 완료!")
print(f"  총 전송: {sent_count}건")
print(f"  소요 시간: {elapsed:.2f}초")
print(f"  처리량: {sent_count / elapsed:.0f} records/sec")

**예상 출력**:

```
API 이벤트 1000건 전송 시작...
  1000건 전송 완료

전송 완료!
  총 전송: 1000건
  소요 시간: 0.45초
  처리량: 2222 records/sec
```

**Kafka UI에서 확인** (http://localhost:8080):
1. Topics → api-events 선택
2. Messages 탭에서 전송된 메시지 확인
3. 파티션별 메시지 분포 확인

---

## Part 2: Consumer 구현 (20분)

### Step 1: Consumer 빈칸 채우기

In [ ]:
# Step 1: Consumer 빈칸 채우기
from confluent_kafka import Consumer
import json

# -----------------------------------------------------------------------------
# Consumer 설정
# -----------------------------------------------------------------------------
config = {
    # TODO 1: Kafka 브로커 주소
    "bootstrap.servers": "_____",
    # TODO 2: Consumer Group ID (같은 그룹의 Consumer는 파티션을 나눠 처리)
    "group.id": "_____",
    # TODO 3: 오프셋 설정 (earliest: 처음부터, latest: 최신부터)
    "auto.offset.reset": "_____",
    "enable.auto.commit": True,
}

# TODO 4: Consumer 객체 생성
consumer = _____

# TODO 5: 토픽 구독
consumer.subscribe(_____)

# -----------------------------------------------------------------------------
# 메시지 수신
# -----------------------------------------------------------------------------
print("메시지 수신 대기 중... (Ctrl+C로 종료)")

try:
    count = 0
    while count < 10:  # 10개만 수신하고 종료
        # TODO 6: 메시지 가져오기 (timeout=1.0)
        msg = consumer._____

        if msg is None:
            continue
        if msg.error():
            print(f"에러: {msg.error()}")
            continue

        # 메시지 처리
        event = json.loads(msg.value().decode("utf-8"))
        print(
            f"수신: {event['request_id']} - {event['endpoint']} ({event['status_code']})"
        )
        count += 1

finally:
    consumer.close()

<details>
<summary>TODO 1 힌트</summary>

Producer와 동일하게 Kafka 브로커 주소를 설정합니다.

</details>

<details>
<summary>TODO 1 정답</summary>

```python
"bootstrap.servers": "kafka:9092",
```

</details>

<details>
<summary>TODO 2 힌트: group.id</summary>

Consumer Group ID는 이 Consumer가 속할 그룹을 지정합니다.

- 같은 그룹의 Consumer들은 파티션을 나눠서 처리
- 다른 그룹의 Consumer들은 모든 메시지를 각각 수신

아무 문자열이나 사용 가능합니다 (예: "api-event-consumer-group")

</details>

<details>
<summary>TODO 2 정답</summary>

```python
"group.id": "api-event-consumer-group",
```

</details>

<details>
<summary>TODO 3 힌트: auto.offset.reset</summary>

새 Consumer Group이 처음 읽을 때 어디서부터 시작할지 결정합니다.

- `earliest`: 토픽의 가장 오래된 메시지부터 (과거 데이터 필요할 때)
- `latest`: 지금부터 새로 들어오는 메시지만 (실시간 처리할 때)

테스트할 때는 `earliest`를 사용해야 기존 메시지를 볼 수 있습니다.

</details>

<details>
<summary>TODO 3 정답</summary>

```python
"auto.offset.reset": "earliest",
```

</details>

<details>
<summary>TODO 4 힌트</summary>

confluent_kafka 라이브러리에서 Consumer 클래스를 사용합니다.

</details>

<details>
<summary>TODO 4 정답</summary>

```python
consumer = Consumer(config)
```

</details>

<details>
<summary>TODO 5 힌트: subscribe</summary>

subscribe() 메서드에 토픽 이름의 리스트를 전달합니다.
하나의 토픽만 구독해도 리스트 형태로 전달해야 합니다.

</details>

<details>
<summary>TODO 5 정답</summary>

```python
consumer.subscribe(["api-events"])
```

</details>

<details>
<summary>TODO 6 힌트: poll</summary>

poll() 메서드로 메시지를 가져옵니다.
timeout 파라미터로 대기 시간(초)을 지정합니다.

</details>

<details>
<summary>TODO 6 정답</summary>

```python
msg = consumer.poll(timeout=1.0)
```

</details>

### Step 2: 완성된 Consumer

In [ ]:
# Step 2: 완성된 Consumer
from confluent_kafka import Consumer
import json
import time

# -----------------------------------------------------------------------------
# Consumer 설정
# -----------------------------------------------------------------------------
config = {
    "bootstrap.servers": "kafka:9092",
    "group.id": "api-event-consumer-group",
    "auto.offset.reset": "earliest",  # 처음부터 읽기
    "enable.auto.commit": True,
}

consumer = Consumer(config)
consumer.subscribe(["api-events"])

# -----------------------------------------------------------------------------
# 메시지 수신 및 통계
# -----------------------------------------------------------------------------
print("메시지 수신 시작...\n")

stats = {"total": 0, "by_status": {}, "by_endpoint": {}}
start_time = time.time()
max_messages = 100  # 100개 메시지 수신 후 통계 출력

try:
    while stats["total"] < max_messages:
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            continue
        if msg.error():
            print(f"에러: {msg.error()}")
            continue

        # 메시지 파싱
        event = json.loads(msg.value().decode("utf-8"))

        # 통계 업데이트
        stats["total"] += 1

        status = event["status_code"]
        stats["by_status"][status] = stats["by_status"].get(status, 0) + 1

        endpoint = event["endpoint"]
        stats["by_endpoint"][endpoint] = stats["by_endpoint"].get(endpoint, 0) + 1

        # 10개마다 진행 상황
        if stats["total"] % 100 == 0:
            print(f"  수신: {stats['total']}건")

finally:
    consumer.close()

# -----------------------------------------------------------------------------
# 통계 출력
# -----------------------------------------------------------------------------
elapsed = time.time() - start_time

print(f"\n{'=' * 50}")
print(f"총 수신: {stats['total']}건 ({elapsed:.2f}초)")
print(f"처리량: {stats['total'] / elapsed:.0f} records/sec")

print(f"\n상태 코드별 분포:")
for status, count in sorted(stats["by_status"].items()):
    pct = count / stats["total"] * 100
    print(f"  {status}: {count}건 ({pct:.1f}%)")

print(f"\n엔드포인트별 분포:")
for endpoint, count in sorted(stats["by_endpoint"].items(), key=lambda x: -x[1]):
    pct = count / stats["total"] * 100
    print(f"  {endpoint}: {count}건 ({pct:.1f}%)")

**예상 출력**:

```
메시지 수신 시작...

  수신: 100건

==================================================
총 수신: 100건 (0.15초)
처리량: 666 records/sec

상태 코드별 분포:
  200: 52건 (52.0%)
  201: 13건 (13.0%)
  400: 12건 (12.0%)
  404: 11건 (11.0%)
  500: 12건 (12.0%)

엔드포인트별 분포:
  /api/products: 23건 (23.0%)
  /api/users: 21건 (21.0%)
  /api/orders: 20건 (20.0%)
  /api/payments: 19건 (19.0%)
  /api/search: 17건 (17.0%)
```

---

## Part 3: 파티션과 처리량 (20분)

### Step 3: 지시사항 - 파티션 수에 따른 처리량 측정

#### 과제 1: 파티션 1개 vs 4개 비교

1. 파티션 1개 토픽 생성
2. 메시지 10,000건 전송 및 시간 측정
3. 파티션 4개 토픽에 동일 테스트
4. 처리량 비교

In [ ]:
# 처리량 측정 실험
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer
import json
import time
import random
from datetime import datetime

admin = AdminClient({"bootstrap.servers": "kafka:9092"})


# -----------------------------------------------------------------------------
# 토픽 생성 함수
# -----------------------------------------------------------------------------
def create_topic(name, num_partitions):
    """토픽 생성 (이미 존재하면 삭제 후 생성)"""
    # 기존 토픽 삭제 시도
    try:
        admin.delete_topics([name]).get(name).result()
        time.sleep(2)  # 삭제 완료 대기
    except:
        pass

    # 새 토픽 생성
    topic = NewTopic(topic=name, num_partitions=num_partitions, replication_factor=1)
    fs = admin.create_topics([topic])
    fs[name].result()
    time.sleep(1)
    print(f"토픽 '{name}' 생성 (파티션: {num_partitions})")


# -----------------------------------------------------------------------------
# 처리량 측정 함수
# -----------------------------------------------------------------------------
def measure_throughput(topic_name, num_messages):
    """메시지 전송 처리량 측정"""
    producer = Producer({"bootstrap.servers": "kafka:9092"})

    sent_count = 0

    def callback(err, msg):
        nonlocal sent_count
        if not err:
            sent_count += 1

    start_time = time.time()

    for i in range(num_messages):
        event = {
            "request_id": f"REQ_{i:06d}",
            "user_id": f"U{random.randint(1, 1000):04d}",
            "endpoint": f"/api/test",
            "timestamp": datetime.now().isoformat(),
        }

        producer.produce(
            topic=topic_name,
            value=json.dumps(event).encode("utf-8"),
            callback=callback,
        )

        # 주기적으로 flush
        if (i + 1) % 10000 == 0:
            producer.flush()

    producer.flush()
    elapsed = time.time() - start_time

    return sent_count, elapsed


# -----------------------------------------------------------------------------
# 실험 실행
# -----------------------------------------------------------------------------
NUM_MESSAGES = 50000

print("=" * 60)
print("파티션 수에 따른 처리량 비교")
print("=" * 60)

# 실험 1: 파티션 1개
create_topic("throughput-test-1p", 1)
count1, time1 = measure_throughput("throughput-test-1p", NUM_MESSAGES)
throughput1 = count1 / time1
print(f"  파티션 1개: {throughput1:,.0f} records/sec ({time1:.2f}초)")

# 실험 2: 파티션 4개
create_topic("throughput-test-4p", 4)
count4, time4 = measure_throughput("throughput-test-4p", NUM_MESSAGES)
throughput4 = count4 / time4
print(f"  파티션 4개: {throughput4:,.0f} records/sec ({time4:.2f}초)")

# 비교
print(f"\n결과:")
print(f"  처리량 향상: {(throughput4 / throughput1 - 1) * 100:.1f}%")

**예상 출력**:

```
============================================================
파티션 수에 따른 처리량 비교
============================================================
토픽 'throughput-test-1p' 생성 (파티션: 1)
  파티션 1개: 45,000 records/sec (1.11초)
토픽 'throughput-test-4p' 생성 (파티션: 4)
  파티션 4개: 52,000 records/sec (0.96초)

결과:
  처리량 향상: 15.6%
```

> **참고**: 단일 Producer에서는 파티션 수 증가의 효과가 제한적입니다.
> 진정한 성능 향상은 **여러 Consumer가 병렬로 처리**할 때 나타납니다.

### 과제 2: Consumer 2개로 병렬 처리

**시나리오**: 같은 Consumer Group의 Consumer 2개가 파티션을 나눠 처리

```
api-events 토픽 (4개 파티션)
├── Partition 0 ──→ Consumer 1
├── Partition 1 ──→ Consumer 1
├── Partition 2 ──→ Consumer 2
└── Partition 3 ──→ Consumer 2
```

<details>
<summary>힌트 보기</summary>

터미널 2개를 열어서 같은 group.id로 Consumer를 실행하면,
Kafka가 자동으로 파티션을 분배합니다.

```python
config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'parallel-consumer-group',  # 같은 그룹 ID!
    'auto.offset.reset': 'earliest',
}
```

</details>

<details>
<summary>정답 보기</summary>

**터미널 1에서 실행:**

```python
from confluent_kafka import Consumer
import json
import time

config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'parallel-consumer-group',
    'auto.offset.reset': 'earliest',
}

consumer = Consumer(config)
consumer.subscribe(['api-events'])

print("Consumer 1 시작...")
count = 0
start = time.time()

try:
    while True:
        msg = consumer.poll(1.0)
        if msg is None:
            if count > 0:
                elapsed = time.time() - start
                print(f"Consumer 1: {count}건 처리 ({count/elapsed:.0f}/sec)")
            continue
        if msg.error():
            continue

        count += 1
        if count % 1000 == 0:
            print(f"Consumer 1: {count}건 처리중 (파티션 {msg.partition()})")
except KeyboardInterrupt:
    pass
finally:
    consumer.close()
```

**터미널 2에서 동일한 코드 실행** (Consumer 2로 변경)

결과: 각 Consumer가 2개의 파티션을 담당하여 병렬 처리

</details>

### Step 3: 병렬 처리 실습 - Producer (100만 건 전송)

**실습 방법**:
1. 먼저 아래 Producer 코드를 실행하여 100만 건의 메시지를 전송합니다.
2. 그 다음 Jupyter 노트북 2개를 열어 Consumer 코드를 동시에 실행합니다.

In [ ]:
# Producer: 파티션 4개 토픽에 100만 건 전송
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
import json
import random
from datetime import datetime
import time

# -----------------------------------------------------------------------------
# 토픽 생성 (파티션 4개)
# -----------------------------------------------------------------------------
TOPIC = "api-events"

admin = AdminClient({"bootstrap.servers": "kafka:9092"})

# 기존 토픽 삭제 후 재생성
try:
    admin.delete_topics([TOPIC])[TOPIC].result()
    time.sleep(2)
    print(f"기존 토픽 '{TOPIC}' 삭제")
except:
    pass

topic = NewTopic(topic=TOPIC, num_partitions=4, replication_factor=1)
admin.create_topics([topic])[TOPIC].result()
print(f"토픽 '{TOPIC}' 생성 (파티션: 4개)\n")

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
config = {
    "bootstrap.servers": "kafka:9092",
    "client.id": "api-event-producer",
}

producer = Producer(config)

# -----------------------------------------------------------------------------
# API 이벤트 생성 함수
# -----------------------------------------------------------------------------
ENDPOINTS = ["/api/products", "/api/users", "/api/orders", "/api/payments", "/api/search"]
METHODS = ["GET", "POST", "PUT", "DELETE"]
STATUS_CODES = [200, 200, 200, 200, 201, 400, 404, 500]


def generate_api_event():
    return {
        "request_id": f"REQ_{random.randint(1, 999999):06d}",
        "user_id": f"U{random.randint(1, 1000):04d}",
        "endpoint": random.choice(ENDPOINTS),
        "method": random.choice(METHODS),
        "status_code": random.choice(STATUS_CODES),
        "response_time_ms": random.randint(10, 500),
        "timestamp": datetime.now().isoformat(),
    }


# -----------------------------------------------------------------------------
# Delivery Callback
# -----------------------------------------------------------------------------
sent_count = 0


def delivery_callback(err, msg):
    global sent_count
    if err:
        print(f"전송 실패: {err}")
    else:
        sent_count += 1


# -----------------------------------------------------------------------------
# 메시지 100만 건 전송
# -----------------------------------------------------------------------------
NUM_MESSAGES = 1000000

print(f"API 이벤트 {NUM_MESSAGES:,}건 전송 시작...")
start_time = time.time()

for i in range(NUM_MESSAGES):
    event = generate_api_event()

    producer.produce(
        topic=TOPIC,
        value=json.dumps(event).encode("utf-8"),
        callback=delivery_callback,
    )

    if (i + 1) % 100000 == 0:
        producer.flush()
        print(f"  {i + 1:,}건 전송 완료")

producer.flush()
elapsed = time.time() - start_time

print("\n전송 완료!")
print(f"  총 전송: {sent_count:,}건")
print(f"  소요 시간: {elapsed:.2f}초")
print(f"  처리량: {sent_count / elapsed:,.0f} records/sec")

### Step 4: 병렬 처리 실습 - Consumer (노트북 2개에서 동시 실행)

**실습 방법**:
1. Jupyter 노트북 파일 2개를 엽니다.
2. 아래 Consumer 코드를 각 노트북에 복사합니다.
3. **동시에** 실행하면 Kafka가 자동으로 파티션을 분배합니다.

```
api-events 토픽 (4개 파티션)
├── Partition 0, 1 ──→ Consumer (노트북 1)
└── Partition 2, 3 ──→ Consumer (노트북 2)
```

In [ ]:
# Consumer: 병렬 처리 (노트북 2개에서 동시 실행)
from confluent_kafka import Consumer
import json
import time

config = {
    "bootstrap.servers": "kafka:9092",
    "group.id": "parallel-consumer-group",  # 같은 그룹 ID!
    "auto.offset.reset": "earliest",
}

consumer = Consumer(config)
consumer.subscribe(["api-events"])

print("Consumer 시작...")
count = 0
start = None
last_msg_time = None

empty_count = 0
max_empty = 3  # 1초 x 3번 = 3초 대기 후 종료
first_message = True

try:
    while True:
        # 첫 메시지는 10초 대기, 이후 1초 대기
        timeout = 10.0 if first_message else 1.0
        msg = consumer.poll(timeout)

        if msg is None:
            empty_count += 1
            if first_message:
                print("10초 동안 메시지 없음. 종료합니다.")
                break
            if empty_count >= max_empty:
                print("더 이상 메시지 없음. 종료합니다.")
                break
            continue

        if msg.error():
            continue

        empty_count = 0

        # 첫 메시지 수신 시 시간 측정 시작
        if first_message:
            start = time.time()
            first_message = False

        last_msg_time = time.time()
        count += 1

        if count % 100000 == 0:
            print(f"  {count:,}건 처리중 (파티션 {msg.partition()})")

except KeyboardInterrupt:
    pass
finally:
    consumer.close()

# 결과 출력 (대기 시간 제외, 실제 처리 시간만 측정)
print(f"\n{'=' * 50}")
if count > 0 and start and last_msg_time:
    elapsed = last_msg_time - start
    print(f"총 수신: {count:,}건")
    print(f"소요 시간: {elapsed:.2f}초 (대기 시간 제외)")
    print(f"처리량: {count / elapsed:,.0f} records/sec")
else:
    print("수신된 메시지 없음")

### Consumer Group 동작 확인

**Kafka UI에서 확인** (http://localhost:8080):

1. Consumers 메뉴 선택
2. `parallel-consumer-group` 그룹 클릭
3. 파티션 할당 확인:
   - Consumer 1: Partition 0, 1
   - Consumer 2: Partition 2, 3

**Consumer Lag 확인**:
- `Lag` 컬럼: 처리되지 않은 메시지 수
- 0에 가까울수록 실시간 처리 중

---

## 핵심 요약

### Producer 핵심 코드

```python
from confluent_kafka import Producer
import json

producer = Producer({'bootstrap.servers': 'kafka:9092'})

producer.produce(
    topic='api-events',
    value=json.dumps(event).encode('utf-8'),
    callback=delivery_callback
)

producer.flush()  # 전송 완료 대기
```

### Consumer 핵심 코드

```python
from confluent_kafka import Consumer

consumer = Consumer({
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'my-group',
    'auto.offset.reset': 'earliest'
})

consumer.subscribe(['api-events'])

while True:
    msg = consumer.poll(1.0)
    if msg:
        event = json.loads(msg.value())
        # 처리 로직
```

### 파티션과 Consumer Group

| 설정 | 효과 |
|------|------|
| 파티션 수 증가 | 병렬 처리 가능한 Consumer 수 증가 |
| 같은 Group ID | 파티션을 나눠서 처리 |
| 다른 Group ID | 모든 메시지를 각각 수신 |

---

## 다음 시간 예고

**3교시: Spark 기초**

- PySpark DataFrame 기본 연산
- Pandas vs Spark 성능 비교
- 10만 → 100만 → 500만 건 스케일업 테스트

---


# Day 11 - 3교시: Spark 기초

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- PySpark SparkSession을 생성하고 설정할 수 있다
- DataFrame 기본 연산(select, filter, groupBy, agg)을 수행할 수 있다
- Pandas와 Spark의 문법 차이를 이해할 수 있다
- Lazy Evaluation과 실행 계획을 이해할 수 있다

---

## Spark는 언제 쓰는가?

### 현실적인 비교

| 상황 | 권장 도구 | 이유 |
|------|-----------|------|
| 데이터 < 수GB | **Pandas** | 단일 머신에서 더 빠름 |
| 데이터 > 메모리 | **Spark** | Pandas는 OOM 발생 |
| 분산 클러스터 (수십~수백 노드) | **Spark** | 수평 확장 가능 |
| 복잡한 ETL 파이프라인 | **Spark** | Lazy Evaluation, 최적화 |
| 실시간 스트리밍 | **Spark** | Structured Streaming |

### 왜 지금 환경에서 Spark가 느릴까?

```
┌─────────────────────────────────────────────────────────────────────────┐
│ 현재 실습 환경 (Docker)                                                  │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   [Python 컨테이너]                                                      │
│      │                                                                  │
│      ├── Pandas: 메모리에서 직접 처리 → 빠름                             │
│      │                                                                  │
│      └── Spark Driver                                                   │
│             │                                                           │
│             │  네트워크 통신 (오버헤드)                                   │
│             ↓                                                           │
│      [Spark Master] → [Worker 1] [Worker 2] [Worker 3] [Worker 4]      │
│                                                                         │
│   문제:                                                                  │
│   1. JVM ↔ Python 직렬화/역직렬화 비용                                   │
│   2. 네트워크 통신 오버헤드                                               │
│   3. Task 스케줄링 오버헤드                                              │
│   4. 데이터가 메모리에 충분히 들어가는 크기                                │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### Spark가 진짜 빛나는 순간

```
┌─────────────────────────────────────────────────────────────────────────┐
│ 실제 프로덕션 환경                                                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   데이터: 1TB (Pandas로는 메모리 부족 → OOM!)                             │
│                                                                         │
│   Spark 클러스터:                                                        │
│   ┌──────────┐  ┌──────────┐  ┌──────────┐       ┌──────────┐          │
│   │ Worker 1 │  │ Worker 2 │  │ Worker 3 │  ...  │Worker 100│          │
│   │ 100GB    │  │ 100GB    │  │ 100GB    │       │ 100GB    │          │
│   └──────────┘  └──────────┘  └──────────┘       └──────────┘          │
│                                                                         │
│   → 각 Worker가 10GB씩 나눠서 병렬 처리                                  │
│   → 수평 확장으로 처리량 선형 증가                                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

> **결론**: 이번 실습에서는 **성능 비교**가 아닌 **Spark 문법과 개념**을 학습합니다.
> Spark의 진정한 가치는 대용량 분산 처리와 스트리밍에서 나타납니다.

---

## 핵심 개념 1: SparkSession

### SparkSession = Spark 애플리케이션의 "진입점"

```
┌─────────────────────────────────────────────────────────────────┐
│                      SparkSession                               │
│  "Spark의 모든 기능에 접근하는 단일 진입점"                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐       │
│   │ DataFrame API │   │   SQL API    │   │ Streaming    │       │
│   │              │   │              │   │              │       │
│   │ spark.read   │   │ spark.sql    │   │ spark.       │       │
│   │ df.select    │   │ ("SELECT..") │   │ readStream   │       │
│   └──────────────┘   └──────────────┘   └──────────────┘       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## 핵심 개념 2: Lazy Evaluation (지연 실행)

### Spark는 "게으르게" 실행한다

```
┌─────────────────────────────────────────────────────────────────┐
│  Pandas (Eager Evaluation - 즉시 실행)                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   df = pd.read_csv(...)     →  [즉시 실행: 파일 읽기]            │
│   df2 = df[df['x'] > 0]     →  [즉시 실행: 필터링]              │
│   df3 = df2.groupby(...)    →  [즉시 실행: 그룹화]              │
│   result = df3.sum()        →  [즉시 실행: 합계]                │
│                                                                 │
│   매 줄마다 실행 → 중간 결과를 메모리에 저장                      │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│  Spark (Lazy Evaluation - 지연 실행)                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   df = spark.read.csv(...)  →  [계획만 세움]                    │
│   df2 = df.filter(...)      →  [계획에 추가]                    │
│   df3 = df2.groupBy(...)    →  [계획에 추가]                    │
│   result = df3.count()      →  [실행! 최적화된 계획으로 한번에]   │
│                             ↑                                   │
│                          Action이 호출될 때만 실행               │
└─────────────────────────────────────────────────────────────────┘
```

### Transformation vs Action

| 구분 | 설명 | 예시 | 실행 시점 |
|------|------|------|----------|
| **Transformation** | 새 DataFrame 생성 | select, filter, groupBy, join | 실행 안 함 (계획만) |
| **Action** | 결과 반환/저장 | count, collect, show, write | **즉시 실행** |

---

## Part 1: SparkSession 생성

In [ ]:
# SparkSession 생성
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Day11-SparkBasics")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

print("SparkSession 생성 완료!")
print(f"  버전: {spark.version}")
print(f"  Master: {spark.sparkContext.master}")
print(f"  총 코어: {spark.sparkContext.defaultParallelism}")

---

## Part 2: 테스트 데이터 생성

In [ ]:
# 테스트 데이터 생성
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

np.random.seed(42)


def generate_api_events(num_records):
    """API 이벤트 데이터 생성"""
    endpoints = ["/api/products", "/api/users", "/api/orders", "/api/payments", "/api/search"]
    methods = ["GET", "POST", "PUT", "DELETE"]
    status_codes = [200, 200, 200, 200, 201, 400, 404, 500]
    base_time = datetime(2024, 1, 1, 0, 0, 0)

    return pd.DataFrame({
        "request_id": [f"REQ_{i:08d}" for i in range(num_records)],
        "user_id": [f"U{np.random.randint(1, 1001):04d}" for _ in range(num_records)],
        "endpoint": np.random.choice(endpoints, num_records),
        "method": np.random.choice(methods, num_records),
        "status_code": np.random.choice(status_codes, num_records),
        "response_time_ms": np.random.randint(10, 500, num_records),
        "timestamp": [base_time + timedelta(seconds=i * 0.1) for i in range(num_records)],
    })


# 10만 건 데이터 생성
filepath = "/data/api_events_100k.csv"
if not os.path.exists(filepath):
    df = generate_api_events(100_000)
    df.to_csv(filepath, index=False)
    print(f"{filepath} 생성 완료 (100,000건)")
else:
    print(f"{filepath} 이미 존재")

---

## Part 3: Pandas ↔ Spark 문법 비교

| 작업 | Pandas | Spark |
|------|--------|-------|
| 파일 읽기 | `pd.read_csv(...)` | `spark.read.csv(...)` |
| 컬럼 선택 | `df[["a", "b"]]` | `df.select("a", "b")` |
| 필터링 | `df[df["x"] > 10]` | `df.filter(col("x") > 10)` |
| 새 컬럼 | `df["new"] = ...` | `df.withColumn("new", ...)` |
| 그룹 집계 | `df.groupby("a").mean()` | `df.groupBy("a").agg(avg(...))` |
| 정렬 | `df.sort_values("x")` | `df.orderBy("x")` |

In [ ]:
# 데이터 로드
df = spark.read.csv("/data/api_events_100k.csv", header=True, inferSchema=True)

print(f"데이터: {df.count():,}건")
print("\n스키마:")
df.printSchema()

In [ ]:
# 데이터 미리보기
df.show(5)

### 1. 컬럼 선택

In [ ]:
# Pandas: df[["endpoint", "status_code", "response_time_ms"]]
# Spark:
selected = df.select("endpoint", "status_code", "response_time_ms")
selected.show(5)

### 2. 필터링

In [ ]:
from pyspark.sql.functions import col

# Pandas: df[df["status_code"] >= 400]
# Spark:
errors = df.filter(col("status_code") >= 400)
print(f"에러 건수: {errors.count():,}건")
errors.show(5)

### 3. 새 컬럼 추가

In [ ]:
from pyspark.sql.functions import when

# Pandas: df["is_error"] = df["status_code"] >= 400
# Spark:
df_with_flag = df.withColumn(
    "is_error",
    when(col("status_code") >= 400, True).otherwise(False)
)
df_with_flag.select("request_id", "status_code", "is_error").show(5)

### 4. 그룹별 집계

In [ ]:
from pyspark.sql.functions import count, avg, max as spark_max

# Pandas: df.groupby("endpoint").agg({"request_id": "count", "response_time_ms": "mean"})
# Spark:
endpoint_stats = df.groupBy("endpoint").agg(
    count("request_id").alias("요청수"),
    avg("response_time_ms").alias("평균응답시간"),
    spark_max("response_time_ms").alias("최대응답시간"),
)
endpoint_stats.show()

### 5. 정렬

In [ ]:
# Pandas: df.sort_values("response_time_ms", ascending=False)
# Spark:
sorted_df = df.orderBy(col("response_time_ms").desc())
sorted_df.select("request_id", "endpoint", "response_time_ms").show(5)

### 6. SQL 쿼리 사용

In [ ]:
# Spark의 강점: SQL로 데이터 처리 가능
df.createOrReplaceTempView("api_events")

result = spark.sql("""
    SELECT
        endpoint,
        COUNT(*) as 요청수,
        ROUND(AVG(response_time_ms), 2) as 평균응답시간,
        SUM(CASE WHEN status_code >= 400 THEN 1 ELSE 0 END) as 에러수
    FROM api_events
    GROUP BY endpoint
    ORDER BY 요청수 DESC
""")
result.show()

---

## Part 4: Lazy Evaluation 실습

In [ ]:
# Transformation은 실행되지 않음 (계획만 세움)
print("Transformation 정의 중... (아직 실행 안 됨)")

step1 = df.filter(col("status_code") >= 400)
step2 = step1.groupBy("endpoint").count()
step3 = step2.orderBy(col("count").desc())

print("여기까지 아무것도 실행되지 않았음!")
print("Action을 호출해야 실행됨")

In [ ]:
# Action 호출 → 모든 Transformation이 한번에 실행
print("\nAction 호출 (collect):")
result = step3.collect()
print(result)

### 실행 계획 확인

In [ ]:
# 실행 계획 보기
print("=== 실행 계획 ===")
step3.explain()

```
실행 계획 읽는 법 (아래에서 위로):

== Physical Plan ==
AdaptiveSparkPlan
+- Sort [count DESC]                    ← 4. 정렬
   +- Exchange                          ← 3. 셔플 (데이터 재분배)
      +- HashAggregate                  ← 2. 그룹별 집계
         +- Filter (status_code >= 400) ← 1. 필터링
            +- FileScan csv             ← 0. 파일 읽기
```

---

## Part 5: 복합 분석 예제

In [ ]:
from pyspark.sql.functions import sum as spark_sum

# 엔드포인트별 에러율 분석
error_analysis = (
    df.groupBy("endpoint")
    .agg(
        count("*").alias("총요청"),
        spark_sum(when(col("status_code") >= 400, 1).otherwise(0)).alias("에러수"),
        avg("response_time_ms").alias("평균응답시간"),
    )
    .withColumn("에러율", col("에러수") / col("총요청") * 100)
    .orderBy(col("에러율").desc())
)

print("엔드포인트별 에러율 분석:")
error_analysis.show()

In [ ]:
# 응답시간 구간별 분포
from pyspark.sql.functions import when

response_dist = (
    df.withColumn(
        "응답시간구간",
        when(col("response_time_ms") < 100, "빠름 (<100ms)")
        .when(col("response_time_ms") < 300, "보통 (100-300ms)")
        .otherwise("느림 (>300ms)")
    )
    .groupBy("응답시간구간")
    .count()
    .orderBy("count")
)

print("응답시간 구간별 분포:")
response_dist.show()

---

## 핵심 요약

### Pandas ↔ Spark 문법 비교

| Pandas | Spark |
|--------|-------|
| `df[["a", "b"]]` | `df.select("a", "b")` |
| `df[df["x"] > 10]` | `df.filter(col("x") > 10)` |
| `df["new"] = ...` | `df.withColumn("new", ...)` |
| `df.groupby("a").mean()` | `df.groupBy("a").agg(avg(...))` |
| 즉시 실행 | Lazy Evaluation |

### Transformation vs Action

| 구분 | 예시 | 실행 시점 |
|------|------|----------|
| **Transformation** | select, filter, groupBy, join, withColumn | Lazy (Action 호출 시) |
| **Action** | count, collect, show, first, write | 즉시 실행 |

### Spark를 배우는 이유

| 상황 | 설명 |
|------|------|
| **대용량 데이터** | TB~PB 규모 처리 (Pandas는 메모리 한계) |
| **분산 클러스터** | Worker 추가로 선형 확장 |
| **스트리밍** | Kafka 연동 실시간 처리 (다음 시간!) |
| **SQL 지원** | SQL로 데이터 처리 가능 |

---

## 다음 시간 예고

**4교시: Spark Structured Streaming**

- Kafka에서 실시간 데이터 수신
- 윈도우 기반 집계 (5분 단위)
- 결과를 콘솔/파일로 출력

In [ ]:
# 세션 정리 (필요시)
# spark.stop()

---


# PySpark 기초 API - 1교시: SparkSession과 DataFrame I/O

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- PySpark 필수 임포트 패턴을 이해하고 작성할 수 있다
- SparkSession을 생성하고 설정할 수 있다
- CSV, Parquet, JSON 파일을 읽고 쓸 수 있다
- DataFrame의 구조와 내용을 확인할 수 있다

---

## Part 1: 필수 임포트 (Boilerplate)

### PySpark를 시작할 때 거의 항상 필요한 임포트

```
┌─────────────────────────────────────────────────────────────────┐
│                    PySpark 임포트 구조                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  pyspark.sql                                                    │
│  ├── SparkSession        → Spark 진입점 (필수)                  │
│  ├── functions           → 데이터 변환 함수들 (col, lit, when..) │
│  └── types               → 데이터 타입 정의 (StringType..)      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 임포트 패턴별 사용 용도

| 임포트 | 용도 | 사용 빈도 |
|--------|------|----------|
| `SparkSession` | Spark 애플리케이션 시작점 | 항상 |
| `functions` | 컬럼 연산, 집계, 변환 | 거의 항상 |
| `types` | 스키마 직접 정의 시 | 자주 |

In [ ]:
# -----------------------------------------------------------------------------
# 기본 임포트: 모든 PySpark 프로젝트의 시작점
# -----------------------------------------------------------------------------

# SparkSession: Spark의 모든 기능에 접근하는 "진입점"
# - DataFrame 생성, 파일 읽기/쓰기, SQL 실행 등 모든 작업의 시작
from pyspark.sql import SparkSession

# functions: 데이터 변환에 사용하는 함수 모음
# - col(): 컬럼 참조
# - lit(): 상수값 삽입
# - when(): 조건문
# - count(), sum(), avg(): 집계 함수
from pyspark.sql.functions import col, lit, when, count, sum, avg

# types: 데이터 타입 정의 (스키마 직접 지정 시 필요)
# - StructType: 전체 스키마 (테이블 구조)
# - StructField: 개별 컬럼 정의
# - StringType, IntegerType 등: 데이터 타입
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

print("임포트 완료!")

### 실무에서 자주 쓰는 확장 임포트

프로젝트가 복잡해지면 더 많은 함수가 필요합니다.

In [ ]:
# -----------------------------------------------------------------------------
# 확장 임포트: 실무에서 자주 사용하는 함수들
# -----------------------------------------------------------------------------

# 날짜/시간 관련 함수
from pyspark.sql.functions import (
    current_date,      # 오늘 날짜 반환
    current_timestamp, # 현재 시간 반환
    to_date,           # 문자열 → 날짜 변환
    to_timestamp,      # 문자열 → 타임스탬프 변환
    date_format,       # 날짜 포맷 변경
    year, month, dayofmonth,  # 날짜 부분 추출
)

# 문자열 처리 함수
from pyspark.sql.functions import (
    concat,         # 문자열 합치기
    concat_ws,      # 구분자로 문자열 합치기
    substring,      # 부분 문자열 추출
    trim,           # 공백 제거
    upper, lower,   # 대소문자 변환
    split,          # 문자열 분리 → 배열
)

# 집계/윈도우 함수
from pyspark.sql.functions import (
    min, max,          # 최소/최대값
    countDistinct,     # 고유값 카운트
    first, last,       # 첫번째/마지막 값
    row_number, rank,  # 순위 함수
    lag, lead,         # 이전/다음 행 참조
)

# 기타 유용한 함수
from pyspark.sql.functions import (
    coalesce,       # 첫 번째 non-null 값 반환
    explode,        # 배열 → 여러 행으로 펼치기
    array,          # 여러 컬럼 → 배열로 합치기
    struct,         # 여러 컬럼 → 구조체로 합치기
    from_json,      # JSON 문자열 파싱
    to_json,        # 구조체 → JSON 문자열
)

print("확장 임포트 완료!")

---

## Part 2: SparkSession 생성

### SparkSession이란?

```
┌─────────────────────────────────────────────────────────────────┐
│                      SparkSession                               │
│  "Spark의 모든 기능에 접근하는 단일 진입점"                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐       │
│   │ DataFrame API │   │   SQL API    │   │ Streaming    │       │
│   │              │   │              │   │              │       │
│   │ spark.read   │   │ spark.sql    │   │ spark.       │       │
│   │ df.select    │   │ ("SELECT..") │   │ readStream   │       │
│   └──────────────┘   └──────────────┘   └──────────────┘       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Builder 패턴

SparkSession은 **Builder 패턴**으로 생성합니다:
1. `SparkSession.builder` → 빌더 시작
2. `.appName()` → 애플리케이션 이름 설정
3. `.config()` → 다양한 설정 추가
4. `.getOrCreate()` → 세션 생성 또는 기존 세션 반환

In [ ]:
# -----------------------------------------------------------------------------
# SparkSession 생성: 기본 형태
# -----------------------------------------------------------------------------

# SparkSession.builder: SparkSession을 만들기 위한 빌더 객체 반환
# .appName("이름"): Spark UI에 표시될 애플리케이션 이름 설정
# .getOrCreate(): 새 세션 생성, 이미 있으면 기존 세션 반환
spark = SparkSession.builder \
    .appName("PySpark-API-Basics") \
    .getOrCreate()

# 생성된 세션 정보 확인
print("SparkSession 생성 완료!")
print(f"  버전: {spark.version}")                          # Spark 버전
print(f"  앱 이름: {spark.sparkContext.appName}")          # 앱 이름
print(f"  마스터: {spark.sparkContext.master}")            # 실행 모드 (local/cluster)

### 설정 옵션 추가하기

실무에서는 메모리, 코어 수, 셔플 파티션 등을 설정합니다.

| 설정 | 설명 | 기본값 |
|------|------|--------|
| `spark.executor.memory` | Executor 메모리 | 1g |
| `spark.executor.cores` | Executor당 코어 수 | 1 |
| `spark.sql.shuffle.partitions` | 셔플 시 파티션 수 | 200 |
| `spark.driver.memory` | Driver 메모리 | 1g |

In [ ]:
# -----------------------------------------------------------------------------
# SparkSession 생성: 설정 포함 (실무 패턴)
# -----------------------------------------------------------------------------

# 기존 세션 종료 (설정 변경을 위해)
spark.stop()

# 새 세션 생성 (설정 포함)
spark = (
    SparkSession.builder
    # 애플리케이션 이름: Spark UI에서 식별용
    .appName("PySpark-API-Basics-Configured")

    # 실행 환경 설정
    # - "local[*]": 로컬 모드, 모든 코어 사용
    # - "spark://master:7077": 클러스터 모드
    .master("local[*]")

    # Executor 메모리: 각 워커가 사용할 메모리
    .config("spark.executor.memory", "2g")

    # 셔플 파티션 수: groupBy, join 등에서 사용
    # 기본값 200은 작은 데이터에 과도함 → 줄이면 성능 향상
    .config("spark.sql.shuffle.partitions", 50)

    # 세션 생성 또는 기존 세션 반환
    .getOrCreate()
)

print("설정된 SparkSession 생성 완료!")
print(f"  셔플 파티션: {spark.conf.get('spark.sql.shuffle.partitions')}")

---

## Part 3: 데이터 읽기 (Read)

### 지원 포맷과 특징

| 포맷 | 특징 | 사용 상황 |
|------|------|----------|
| **CSV** | 범용, 사람이 읽기 쉬움 | 데이터 교환, 작은 파일 |
| **Parquet** | 컬럼 기반, 압축률 높음, 빠름 | 실무 표준, 데이터 레이크 |
| **JSON** | 중첩 구조 지원 | 로그 데이터, API 응답 |

### 읽기 패턴

```python
spark.read
    .format("csv")           # 포맷 지정 (생략 가능)
    .option("header", True)  # 옵션 설정
    .load("path/to/file")    # 파일 경로

# 또는 단축형
spark.read.csv("path", header=True)
```

In [ ]:
# -----------------------------------------------------------------------------
# 테스트 데이터 생성 (Pandas로 먼저 생성 후 파일 저장)
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import os

# 재현 가능한 랜덤 시드 설정
np.random.seed(42)

# 샘플 데이터 생성: 직원 정보
sample_data = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 101)],          # E001 ~ E100
    "name": [f"Employee_{i}" for i in range(1, 101)],        # 이름
    "department": np.random.choice(                           # 부서 (랜덤)
        ["Engineering", "Sales", "Marketing", "HR", "Finance"],
        100
    ),
    "salary": np.random.randint(40000, 120000, 100),         # 연봉 (랜덤)
    "hire_date": pd.date_range("2020-01-01", periods=100, freq="7D"),  # 입사일
    "age": np.random.randint(25, 55, 100),                   # 나이 (랜덤)
    "is_manager": np.random.choice([True, False], 100, p=[0.2, 0.8]),  # 매니저 여부
})

# 결측치 일부 추가 (실무 데이터처럼)
sample_data.loc[5:10, "salary"] = None
sample_data.loc[15:18, "department"] = None

# 데이터 디렉토리 생성
os.makedirs("/tmp/spark_tutorial", exist_ok=True)

# 다양한 포맷으로 저장
sample_data.to_csv("/tmp/spark_tutorial/employees.csv", index=False)
sample_data.to_parquet("/tmp/spark_tutorial/employees.parquet", index=False)
sample_data.to_json("/tmp/spark_tutorial/employees.json", orient="records", lines=True)

print("테스트 데이터 생성 완료!")
print(f"  행 수: {len(sample_data)}")
print(f"  컬럼: {list(sample_data.columns)}")

### 3-1. CSV 파일 읽기

In [ ]:
# -----------------------------------------------------------------------------
# CSV 파일 읽기
# -----------------------------------------------------------------------------

# 방법 1: 기본 읽기 (헤더 있음, 스키마 자동 추론)
# - header=True: 첫 번째 줄을 컬럼명으로 사용
# - inferSchema=True: 데이터 타입 자동 추론 (숫자, 문자열 등)
#   주의: inferSchema는 데이터를 한 번 더 읽어서 느림 (대용량에서)
df_csv = spark.read.csv(
    "/tmp/spark_tutorial/employees.csv",  # 파일 경로
    header=True,                           # 첫 줄이 헤더인지
    inferSchema=True                       # 타입 자동 추론
)

# 스키마(구조) 확인
print("=== CSV 스키마 ===")
df_csv.printSchema()

In [ ]:
# -----------------------------------------------------------------------------
# CSV 읽기: option() 메서드 사용 (세밀한 제어)
# -----------------------------------------------------------------------------

# 방법 2: option() 메서드로 설정 (더 유연함)
df_csv2 = (
    spark.read
    .format("csv")                              # 포맷 명시
    .option("header", "true")                   # 헤더 사용
    .option("inferSchema", "true")              # 타입 추론
    .option("nullValue", "NA")                  # "NA"를 null로 처리
    .option("dateFormat", "yyyy-MM-dd")         # 날짜 포맷
    .load("/tmp/spark_tutorial/employees.csv")  # 파일 로드
)

print("=== option() 사용 CSV 읽기 ===")
df_csv2.show(5)

### 3-2. Parquet 파일 읽기 (실무 표준)

**Parquet을 사용해야 하는 이유**:
- 컬럼 기반 저장 → 필요한 컬럼만 읽어서 빠름
- 내장 압축 → 파일 크기 작음 (CSV의 1/5~1/10)
- 스키마 내장 → inferSchema 불필요
- Spark 최적화 지원 → Predicate Pushdown 등

In [ ]:
# -----------------------------------------------------------------------------
# Parquet 파일 읽기
# -----------------------------------------------------------------------------

# Parquet은 스키마가 파일에 포함되어 있어 옵션이 거의 필요 없음
# inferSchema도 불필요 (자동으로 스키마 읽음)
df_parquet = spark.read.parquet("/tmp/spark_tutorial/employees.parquet")

print("=== Parquet 스키마 (자동 추론됨) ===")
df_parquet.printSchema()

### 3-3. JSON 파일 읽기

In [ ]:
# -----------------------------------------------------------------------------
# JSON 파일 읽기
# -----------------------------------------------------------------------------

# JSON Lines 형식 (한 줄에 하나의 JSON 객체)
# - 기본 JSON: [{"a":1}, {"a":2}]  → 배열
# - JSON Lines: {"a":1}\n{"a":2}   → 줄바꿈으로 구분
df_json = spark.read.json("/tmp/spark_tutorial/employees.json")

print("=== JSON 스키마 ===")
df_json.printSchema()
df_json.show(3)

### 3-4. 스키마 직접 지정 (권장)

`inferSchema=True`는 편리하지만:
- 데이터를 한 번 더 읽어서 **느림**
- 타입 추론이 틀릴 수 있음 (예: "001" → 숫자로 추론)

**실무에서는 스키마를 직접 지정**하는 것이 좋습니다.

In [ ]:
# -----------------------------------------------------------------------------
# 스키마 직접 정의 후 읽기
# -----------------------------------------------------------------------------

# StructType: 전체 스키마 (테이블 구조)
# StructField(이름, 타입, nullable): 개별 컬럼 정의
#   - 이름: 컬럼명
#   - 타입: StringType(), IntegerType(), etc.
#   - nullable: NULL 허용 여부 (True/False)

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType, DateType
)

# 스키마 정의
employee_schema = StructType([
    StructField("emp_id", StringType(), False),       # 필수 (nullable=False)
    StructField("name", StringType(), True),          # NULL 허용
    StructField("department", StringType(), True),    # NULL 허용
    StructField("salary", IntegerType(), True),       # 정수형
    StructField("hire_date", DateType(), True),       # 날짜형
    StructField("age", IntegerType(), True),          # 정수형
    StructField("is_manager", BooleanType(), True),   # 불린형
])

# 정의한 스키마로 CSV 읽기 (inferSchema 대신)
df_with_schema = (
    spark.read
    .option("header", "true")
    .schema(employee_schema)                          # 스키마 지정
    .csv("/tmp/spark_tutorial/employees.csv")
)

print("=== 스키마 직접 지정 결과 ===")
df_with_schema.printSchema()

---

## Part 4: 데이터 쓰기 (Write)

### 쓰기 모드 (mode)

| 모드 | 설명 | SQL 비유 |
|------|------|----------|
| `overwrite` | 기존 데이터 삭제 후 새로 쓰기 | TRUNCATE + INSERT |
| `append` | 기존 데이터에 추가 | INSERT |
| `ignore` | 이미 있으면 무시 (에러 없음) | INSERT IGNORE |
| `error` (기본값) | 이미 있으면 에러 발생 | INSERT (에러 시 롤백) |

### 파티셔닝

```
파티셔닝 없이 저장:
output/
├── part-00000.parquet
├── part-00001.parquet
└── part-00002.parquet

partitionBy("year", "month") 사용:
output/
├── year=2024/
│   ├── month=01/
│   │   └── part-00000.parquet
│   └── month=02/
│       └── part-00000.parquet
└── year=2025/
    └── month=01/
        └── part-00000.parquet
```

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 쓰기: Parquet (실무 표준)
# -----------------------------------------------------------------------------

# mode("overwrite"): 기존 파일 덮어쓰기
# 주의: 실수로 데이터 날릴 수 있으니 경로 확인!
df_parquet.write \
    .mode("overwrite") \
    .parquet("/tmp/spark_tutorial/output/employees_output.parquet")

print("Parquet 저장 완료!")

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 쓰기: 파티셔닝 (대용량 데이터 필수)
# -----------------------------------------------------------------------------

# partitionBy(): 지정한 컬럼 값으로 폴더를 나눠서 저장
# 장점:
#   - 쿼리 시 필요한 파티션만 읽음 (Partition Pruning)
#   - department="Engineering" 조건 시 해당 폴더만 읽음
df_parquet.write \
    .mode("overwrite") \
    .partitionBy("department") \
    .parquet("/tmp/spark_tutorial/output/employees_partitioned")

print("파티셔닝 저장 완료!")
print("\n저장된 폴더 구조:")

# 폴더 구조 확인 (Bash 명령)
import subprocess
result = subprocess.run(
    ["find", "/tmp/spark_tutorial/output/employees_partitioned", "-type", "d"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 쓰기: 파일 개수 조절
# -----------------------------------------------------------------------------

# 기본적으로 파티션 수만큼 파일이 생성됨
# coalesce(n): 파일 개수를 n개로 줄임 (셔플 없음, 감소만 가능)
# repartition(n): 파일 개수를 n개로 변경 (셔플 발생, 증가/감소 가능)

# 작은 파일 여러 개 → 큰 파일 하나로 합치기
df_parquet.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet("/tmp/spark_tutorial/output/employees_single")

print("단일 파일로 저장 완료!")

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 쓰기: CSV (다른 시스템 연동용)
# -----------------------------------------------------------------------------

# CSV로 저장 (다른 도구와 연동 시)
df_parquet.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/tmp/spark_tutorial/output/employees_output.csv")

print("CSV 저장 완료!")

---

## Part 5: DataFrame 확인 (Inspection)

### 확인 메서드 요약

| 메서드 | 용도 | 반환 타입 |
|--------|------|----------|
| `printSchema()` | 스키마(구조) 확인 | None (출력) |
| `show()` | 데이터 미리보기 | None (출력) |
| `count()` | 행 수 | int |
| `columns` | 컬럼 목록 | list |
| `dtypes` | 컬럼별 타입 | list of tuple |
| `describe()` | 통계 요약 | DataFrame |

In [ ]:
# -----------------------------------------------------------------------------
# 스키마 확인: printSchema()
# -----------------------------------------------------------------------------

# printSchema(): 컬럼명, 데이터 타입, nullable 여부를 트리 형태로 출력
# - root: 최상위
# - |-- 컬럼명: 타입 (nullable = true/false)
print("=== 스키마 확인 ===")
df_csv.printSchema()

In [ ]:
# -----------------------------------------------------------------------------
# 데이터 미리보기: show()
# -----------------------------------------------------------------------------

# show(): 기본 20행 출력
# show(n): n행 출력
# show(n, truncate=False): 컬럼 내용 잘리지 않게 출력
# show(n, truncate=10): 10자까지만 출력
# show(vertical=True): 세로로 출력 (컬럼 많을 때)

print("=== 기본 show() ===")
df_csv.show(5)  # 5행만 출력

print("=== truncate=False (잘림 없이) ===")
df_csv.show(3, truncate=False)

print("=== vertical=True (세로 출력) ===")
df_csv.show(2, vertical=True)

In [ ]:
# -----------------------------------------------------------------------------
# 행/컬럼 수 확인
# -----------------------------------------------------------------------------

# count(): 전체 행 수 반환 (Action이라 실행됨)
row_count = df_csv.count()
print(f"행 수: {row_count:,}")

# columns: 컬럼명 리스트 반환 (속성)
col_list = df_csv.columns
print(f"컬럼 목록: {col_list}")
print(f"컬럼 수: {len(col_list)}")

# dtypes: (컬럼명, 타입) 튜플 리스트 반환
print(f"컬럼별 타입: {df_csv.dtypes}")

In [ ]:
# -----------------------------------------------------------------------------
# 통계 요약: describe()
# -----------------------------------------------------------------------------

# describe(): 숫자형 컬럼의 기본 통계 (count, mean, stddev, min, max)
# 반환값이 DataFrame이므로 show() 호출 필요
print("=== 통계 요약 ===")
df_csv.describe().show()

# 특정 컬럼만 통계
print("=== salary 컬럼 통계 ===")
df_csv.describe("salary", "age").show()

In [ ]:
# -----------------------------------------------------------------------------
# 유용한 추가 메서드들
# -----------------------------------------------------------------------------

# head(n): 처음 n행을 Row 객체 리스트로 반환
first_row = df_csv.head(1)
print(f"첫 번째 행: {first_row}")

# first(): 첫 번째 행을 Row 객체로 반환
first = df_csv.first()
print(f"first(): {first}")

# take(n): head(n)과 동일
top3 = df_csv.take(3)
print(f"take(3): {len(top3)}개 Row")

# collect(): 전체 데이터를 Driver로 가져옴 (주의: 대용량 시 OOM!)
# 작은 데이터에서만 사용
# all_rows = df_csv.collect()

# distinct(): 중복 제거된 행 수
dept_count = df_csv.select("department").distinct().count()
print(f"부서 종류: {dept_count}개")

---

## 퀴즈

Q1. SparkSession을 생성할 때 반드시 호출해야 하는 마지막 메서드는?

- A) `.build()`
- B) `.create()`
- C) `.getOrCreate()`
- D) `.start()`

<details>
<summary>정답 보기</summary>

**정답: C) `.getOrCreate()`**

`getOrCreate()`는 새 세션을 생성하거나, 이미 존재하는 세션을 반환합니다.
이렇게 하면 중복 세션 생성을 방지할 수 있습니다.

</details>

---

Q2. 다음 중 Parquet 포맷의 장점이 아닌 것은?

- A) 컬럼 기반 저장으로 필요한 컬럼만 읽을 수 있다
- B) 사람이 텍스트 에디터로 직접 읽고 수정할 수 있다
- C) 내장 압축으로 파일 크기가 작다
- D) 스키마가 파일에 포함되어 있다

<details>
<summary>정답 보기</summary>

**정답: B) 사람이 텍스트 에디터로 직접 읽고 수정할 수 있다**

Parquet은 바이너리 포맷이라 텍스트 에디터로 읽을 수 없습니다.
사람이 읽을 수 있는 포맷이 필요하면 CSV나 JSON을 사용합니다.

</details>

---

Q3. CSV 파일을 읽을 때 `inferSchema=True`의 단점은?

- A) 첫 번째 줄을 헤더로 사용할 수 없다
- B) 데이터를 한 번 더 읽어서 느리다
- C) NULL 값을 처리할 수 없다
- D) 날짜 타입을 지원하지 않는다

<details>
<summary>정답 보기</summary>

**정답: B) 데이터를 한 번 더 읽어서 느리다**

`inferSchema=True`는 타입 추론을 위해 데이터를 먼저 한 번 읽습니다.
대용량 데이터에서는 스키마를 직접 지정하는 것이 더 빠릅니다.

</details>

---

Q4. 다음 쓰기 모드 중 "기존 데이터가 있으면 무시하고, 없으면 새로 쓰기"에 해당하는 것은?

- A) `overwrite`
- B) `append`
- C) `ignore`
- D) `error`

<details>
<summary>정답 보기</summary>

**정답: C) `ignore`**

- `overwrite`: 기존 삭제 후 새로 쓰기
- `append`: 기존에 추가
- `ignore`: 이미 있으면 아무것도 안 함
- `error`: 이미 있으면 에러 발생 (기본값)

</details>

---

## 과제: 데이터 파이프라인 기초

### Step 1: SparkSession 생성

아래 조건으로 SparkSession을 생성하세요:
- 앱 이름: "MyFirstPipeline"
- 셔플 파티션: 10

<details>
<summary>힌트</summary>

`.config("spark.sql.shuffle.partitions", 10)` 사용

</details>

<details>
<summary>모범 답안</summary>

```python
spark = (
    SparkSession.builder
    .appName("MyFirstPipeline")
    .config("spark.sql.shuffle.partitions", 10)
    .getOrCreate()
)
```

</details>

### Step 2: 데이터 읽기

`/tmp/spark_tutorial/employees.csv` 파일을 읽되, 스키마를 직접 정의하세요.

<details>
<summary>힌트</summary>

`StructType([StructField(...), ...])` 사용
`.schema(my_schema).csv(path)` 형태

</details>

### Step 3: 데이터 저장

읽은 데이터를 `department` 컬럼으로 파티셔닝하여 Parquet으로 저장하세요.
경로: `/tmp/spark_tutorial/my_output`

<details>
<summary>힌트</summary>

`.partitionBy("department").parquet(path)` 사용

</details>

### 보너스: 파일 크기 비교

CSV와 Parquet 파일 크기를 비교해보세요. (du -sh 명령 사용)

---

## 핵심 요약

### 필수 임포트

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, count, sum, avg
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
```

### SparkSession 생성

```python
spark = SparkSession.builder \
    .appName("앱이름") \
    .config("설정키", "값") \
    .getOrCreate()
```

### 데이터 읽기

| 포맷 | 코드 |
|------|------|
| CSV | `spark.read.csv(path, header=True, inferSchema=True)` |
| Parquet | `spark.read.parquet(path)` |
| JSON | `spark.read.json(path)` |

### 데이터 쓰기

| 작업 | 코드 |
|------|------|
| 기본 저장 | `df.write.parquet(path)` |
| 덮어쓰기 | `df.write.mode("overwrite").parquet(path)` |
| 파티셔닝 | `df.write.partitionBy("col").parquet(path)` |

### DataFrame 확인

| 메서드 | 용도 |
|--------|------|
| `printSchema()` | 스키마 확인 |
| `show(n)` | 데이터 미리보기 |
| `count()` | 행 수 |
| `describe()` | 통계 요약 |

In [ ]:
# 세션 정리 (다음 교시를 위해 유지하거나 종료)
# spark.stop()
print("\n1교시 완료! 다음 교시: 컬럼 조작과 필터링")

---


# PySpark 기초 API - 2교시: 컬럼 조작과 필터링

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- `select()`로 필요한 컬럼만 선택할 수 있다
- `withColumn()`으로 새 컬럼을 추가하거나 수정할 수 있다
- `filter()`/`where()`로 데이터를 조건별로 필터링할 수 있다
- `when()`/`otherwise()`로 조건부 값을 설정할 수 있다
- 정렬, 중복 제거, 제한 등 기본 연산을 수행할 수 있다

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정: SparkSession 생성 및 데이터 로드
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, upper, lower, concat, substring
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DateType
import pandas as pd
import numpy as np
import os

# SparkSession 생성 (이전 교시에서 이미 있으면 재사용)
spark = SparkSession.builder \
    .appName("PySpark-Column-Filter") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

# 테스트 데이터 생성 (이전 교시 데이터 없으면 새로 생성)
np.random.seed(42)

os.makedirs("/tmp/spark_tutorial", exist_ok=True)

# 샘플 데이터
sample_data = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 101)],
    "name": [f"Employee_{i}" for i in range(1, 101)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", "HR", "Finance"], 100
    ),
    "salary": np.random.randint(40000, 120000, 100),
    "age": np.random.randint(25, 55, 100),
    "is_manager": np.random.choice([True, False], 100, p=[0.2, 0.8]),
})

# 결측치 추가
sample_data.loc[5:10, "salary"] = None
sample_data.loc[15:18, "department"] = None

sample_data.to_csv("/tmp/spark_tutorial/employees.csv", index=False)

# DataFrame 로드
df = spark.read.csv("/tmp/spark_tutorial/employees.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"행 수: {df.count()}, 컬럼: {df.columns}")
df.show(5)

---

## Part 1: 컬럼 참조 방법

### col() 함수 vs 문자열 vs df.컬럼명

Spark에서 컬럼을 참조하는 방법은 3가지가 있습니다:

| 방법 | 예시 | 사용 상황 |
|------|------|----------|
| 문자열 | `"name"` | select, groupBy 등에서 단순 참조 |
| col() 함수 | `col("name")` | 연산, 조건식에서 사용 (권장) |
| df.컬럼명 | `df.name` | 조인 시 테이블 구분 |

In [ ]:
# -----------------------------------------------------------------------------
# 컬럼 참조 방법 비교
# -----------------------------------------------------------------------------
from pyspark.sql.functions import col

# 방법 1: 문자열 (가장 단순)
# select, groupBy 등에서 컬럼명만 필요할 때
df.select("name", "salary").show(3)

# 방법 2: col() 함수 (가장 권장)
# 연산이나 조건식에서 사용
# col("컬럼명"): Column 객체 반환 → 연산 가능
df.select(col("name"), col("salary") * 1.1).show(3)

# 방법 3: df.컬럼명 (조인 시 유용)
# 여러 DataFrame을 다룰 때 어떤 테이블의 컬럼인지 명확히
df.select(df.name, df.salary).show(3)

In [ ]:
# -----------------------------------------------------------------------------
# col() vs 문자열: 연산 가능 여부
# -----------------------------------------------------------------------------

# 문자열은 연산 불가 (에러 발생)
# df.select("salary" * 1.1)  # TypeError!

# col()은 연산 가능
# col("salary"): salary 컬럼을 Column 객체로 반환
# * 1.1: 모든 값에 1.1 곱하기
df.select(
    col("name"),                    # 이름 그대로
    col("salary"),                  # 원래 급여
    col("salary") * 1.1             # 급여 10% 인상
).show(5)

---

## Part 2: 컬럼 선택 (select)

### select() 사용법

```python
# 기본 형태
df.select("col1", "col2", "col3")
df.select(col("col1"), col("col2"))

# 모든 컬럼
df.select("*")

# 연산과 함께
df.select("name", (col("salary") * 1.1).alias("new_salary"))
```

In [ ]:
# -----------------------------------------------------------------------------
# select(): 기본 사용법
# -----------------------------------------------------------------------------

# 단일 컬럼 선택
# 결과: 해당 컬럼만 포함된 새 DataFrame 반환
df.select("name").show(5)

# 여러 컬럼 선택
# 쉼표로 구분하여 나열
df.select("emp_id", "name", "department").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# select(): 컬럼 연산과 별칭(alias)
# -----------------------------------------------------------------------------

# alias(): 컬럼에 새 이름 부여
# (col("salary") * 1.1).alias("raised_salary"): 계산 결과에 이름 지정
result = df.select(
    col("name"),                                    # 이름 그대로
    col("salary"),                                  # 원래 급여
    (col("salary") * 1.1).alias("raised_salary"),  # 10% 인상 급여 (새 이름)
    (col("salary") / 12).alias("monthly_salary")   # 월급 (새 이름)
)

print("=== 급여 계산 ===")
result.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# select(): 컬럼 순서 변경, 특정 컬럼 제외
# -----------------------------------------------------------------------------

# 컬럼 순서 변경: select에 원하는 순서로 나열
reordered = df.select("department", "name", "salary", "emp_id")
print("=== 컬럼 순서 변경 ===")
reordered.show(3)

# 특정 컬럼 제외: drop() 사용 (select의 반대)
# drop(): 지정한 컬럼을 제외한 나머지 반환
without_manager = df.drop("is_manager")
print("=== is_manager 컬럼 제외 ===")
without_manager.show(3)

In [ ]:
# -----------------------------------------------------------------------------
# select(): 동적 컬럼 선택 (리스트 활용)
# -----------------------------------------------------------------------------

# 컬럼 목록을 변수로 관리
cols_to_select = ["emp_id", "name", "salary"]

# * 연산자로 리스트 언패킹
# *cols_to_select: ["a", "b"] → "a", "b"로 풀어줌
df.select(*cols_to_select).show(5)

# 조건에 따라 컬럼 선택
numeric_cols = ["salary", "age"]
df.select(*numeric_cols).describe().show()

---

## Part 3: 컬럼 추가/수정 (withColumn)

### Spark의 불변성 (Immutability)

```
┌─────────────────────────────────────────────────────────────────┐
│                    Spark DataFrame은 불변(Immutable)            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Pandas:                                                       │
│   df["new_col"] = df["old_col"] * 2  # 원본 df 변경             │
│                                                                 │
│   Spark:                                                        │
│   df.withColumn("new_col", col("old_col") * 2)  # 새 df 반환    │
│   df = df.withColumn(...)  # 결과를 다시 할당해야 유지           │
│                                                                 │
│   ★ 중요: withColumn() 결과를 변수에 저장하지 않으면 사라짐!     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 새 컬럼 추가
# -----------------------------------------------------------------------------

# withColumn(컬럼명, 표현식): 새 컬럼 추가 또는 기존 컬럼 덮어쓰기
# - 첫 번째 인자: 새 컬럼 이름 (문자열)
# - 두 번째 인자: 컬럼 값을 계산하는 표현식 (Column 객체)

# 새 컬럼 추가: 연봉을 월급으로 변환
# col("salary") / 12: salary 컬럼의 모든 값을 12로 나눔
df_with_monthly = df.withColumn(
    "monthly_salary",       # 새 컬럼 이름
    col("salary") / 12      # 계산식
)

print("=== 월급 컬럼 추가 ===")
df_with_monthly.select("name", "salary", "monthly_salary").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 상수값 추가 (lit 함수)
# -----------------------------------------------------------------------------

# lit(): 상수(리터럴) 값을 Column으로 변환
# - col("name") → 컬럼 참조
# - lit("Korea") → 상수값 "Korea"
#
# 모든 행에 같은 값을 넣을 때 사용

# 상수 컬럼 추가
df_with_country = df.withColumn(
    "country",      # 컬럼명
    lit("Korea")    # 모든 행에 "Korea" 값
)

print("=== 상수 컬럼 추가 ===")
df_with_country.select("name", "country").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 기존 컬럼 수정 (덮어쓰기)
# -----------------------------------------------------------------------------

# 같은 컬럼명을 사용하면 기존 컬럼을 덮어씀
# salary 컬럼을 10% 인상된 값으로 교체
df_raised = df.withColumn(
    "salary",           # 기존 컬럼명 (덮어쓰기)
    col("salary") * 1.1 # 10% 인상
)

print("=== 급여 10% 인상 (원본 비교) ===")
print("원본:")
df.select("name", "salary").show(3)
print("수정 후:")
df_raised.select("name", "salary").show(3)

In [ ]:
# -----------------------------------------------------------------------------
# withColumn(): 체이닝 (여러 컬럼 한번에)
# -----------------------------------------------------------------------------

# 여러 withColumn을 연속으로 호출 (메서드 체이닝)
# 각 withColumn은 새 DataFrame을 반환하므로 연결 가능
df_enhanced = (
    df
    # 연봉에서 월급 계산
    .withColumn("monthly_salary", col("salary") / 12)
    # 연봉에서 일급 계산 (연 250일 근무 가정)
    .withColumn("daily_salary", col("salary") / 250)
    # 국가 추가
    .withColumn("country", lit("Korea"))
    # 연도 추가
    .withColumn("year", lit(2024))
)

print("=== 여러 컬럼 추가 ===")
df_enhanced.show(5)

---

## Part 4: 컬럼 이름 변경과 삭제

| 메서드 | 용도 | 예시 |
|--------|------|------|
| `withColumnRenamed()` | 단일 컬럼 이름 변경 | `df.withColumnRenamed("old", "new")` |
| `toDF()` | 모든 컬럼 이름 변경 | `df.toDF("a", "b", "c")` |
| `drop()` | 컬럼 삭제 | `df.drop("col1", "col2")` |

In [ ]:
# -----------------------------------------------------------------------------
# withColumnRenamed(): 컬럼 이름 변경
# -----------------------------------------------------------------------------

# withColumnRenamed(기존이름, 새이름): 단일 컬럼 이름 변경
# 원본 DataFrame은 변경되지 않음 (새 DataFrame 반환)
df_renamed = df.withColumnRenamed("emp_id", "employee_id")

print("=== 컬럼명 변경: emp_id → employee_id ===")
df_renamed.printSchema()

In [ ]:
# -----------------------------------------------------------------------------
# 여러 컬럼 이름 변경
# -----------------------------------------------------------------------------

# 방법 1: withColumnRenamed 체이닝
df_multi_renamed = (
    df
    .withColumnRenamed("emp_id", "employee_id")
    .withColumnRenamed("department", "dept")
    .withColumnRenamed("is_manager", "manager_flag")
)

print("=== 여러 컬럼명 변경 ===")
df_multi_renamed.printSchema()

# 방법 2: toDF() - 모든 컬럼명 한번에 변경
# 주의: 컬럼 순서와 개수가 정확히 일치해야 함
# df.toDF("new_col1", "new_col2", ...) - 모든 컬럼에 새 이름 지정

In [ ]:
# -----------------------------------------------------------------------------
# drop(): 컬럼 삭제
# -----------------------------------------------------------------------------

# drop(컬럼명): 지정한 컬럼 제거
# 여러 컬럼 제거: drop("col1", "col2") 또는 drop("col1").drop("col2")

# 단일 컬럼 삭제
df_no_manager = df.drop("is_manager")

print("=== is_manager 컬럼 삭제 ===")
print(f"삭제 전 컬럼: {df.columns}")
print(f"삭제 후 컬럼: {df_no_manager.columns}")

# 여러 컬럼 삭제
df_minimal = df.drop("is_manager", "age")
print(f"여러 컬럼 삭제 후: {df_minimal.columns}")

---

## Part 5: 필터링 (filter / where)

### filter()와 where()는 동일

```python
df.filter(조건)   # 함수형 스타일
df.where(조건)    # SQL 스타일 (동일한 기능)
```

### 비교 연산자

| 연산자 | 의미 | 예시 |
|--------|------|------|
| `==` | 같음 | `col("dept") == "Sales"` |
| `!=` | 다름 | `col("dept") != "HR"` |
| `>`, `>=` | 크다, 크거나 같다 | `col("age") >= 30` |
| `<`, `<=` | 작다, 작거나 같다 | `col("salary") < 50000` |

### 논리 연산자

| 연산자 | 의미 | 주의사항 |
|--------|------|----------|
| `&` | AND | 괄호 필수! |
| `\|` | OR | 괄호 필수! |
| `~` | NOT | |

In [ ]:
# -----------------------------------------------------------------------------
# filter(): 기본 필터링
# -----------------------------------------------------------------------------

# 조건: col("컬럼") 연산자 값
# 결과: 조건을 만족하는 행만 포함된 새 DataFrame

# 급여가 80000 이상인 직원
high_salary = df.filter(col("salary") >= 80000)
print(f"=== 고연봉자 (80000 이상): {high_salary.count()}명 ===")
high_salary.show(5)

# 특정 부서 직원
engineers = df.filter(col("department") == "Engineering")
print(f"=== Engineering 부서: {engineers.count()}명 ===")
engineers.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# filter(): AND / OR 조건 (괄호 필수!)
# -----------------------------------------------------------------------------

# AND 조건: & 연산자 (각 조건을 괄호로 감싸야 함!)
# 이유: Python 연산자 우선순위 때문
# 틀린 예: col("age") > 30 & col("salary") > 70000  # 에러!
# 맞는 예: (col("age") > 30) & (col("salary") > 70000)

# AND: 30세 이상이면서 급여 70000 이상
senior_high = df.filter(
    (col("age") >= 30) & (col("salary") >= 70000)
)
print(f"=== 30세 이상 AND 고연봉: {senior_high.count()}명 ===")
senior_high.show(5)

# OR: Engineering이거나 Sales 부서
eng_or_sales = df.filter(
    (col("department") == "Engineering") | (col("department") == "Sales")
)
print(f"=== Engineering OR Sales: {eng_or_sales.count()}명 ===")

In [ ]:
# -----------------------------------------------------------------------------
# filter(): SQL 스타일 문자열 조건
# -----------------------------------------------------------------------------

# 문자열로 SQL WHERE 절처럼 작성 가능
# 더 직관적일 수 있음 (SQL에 익숙하면)

# SQL 스타일 필터링
df.filter("age >= 30 AND salary >= 70000").show(5)

# SQL 스타일: BETWEEN
df.filter("salary BETWEEN 50000 AND 80000").show(5)

# SQL 스타일: IN
df.filter("department IN ('Engineering', 'Sales', 'Marketing')").show(5)

# SQL 스타일: LIKE (문자열 패턴)
df.filter("name LIKE 'Employee_1%'").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# filter(): isin(), isNull(), isNotNull()
# -----------------------------------------------------------------------------

# isin(): 여러 값 중 하나인지 확인
# SQL의 IN 절과 동일
target_depts = ["Engineering", "Sales"]
df.filter(col("department").isin(target_depts)).show(5)

# isin()에 리스트 직접 전달
df.filter(col("department").isin("HR", "Finance")).show(5)

# isNull(): NULL인 행만
df.filter(col("salary").isNull()).show()

# isNotNull(): NULL이 아닌 행만
df.filter(col("salary").isNotNull()).count()

In [ ]:
# -----------------------------------------------------------------------------
# filter(): 문자열 조건 메서드
# -----------------------------------------------------------------------------

# startswith(): 특정 문자로 시작
df.filter(col("name").startswith("Employee_1")).show(5)

# endswith(): 특정 문자로 끝
df.filter(col("emp_id").endswith("5")).show(5)

# contains(): 특정 문자 포함
df.filter(col("department").contains("ing")).show(5)  # Engineering, Marketing

In [ ]:
# -----------------------------------------------------------------------------
# where(): filter()와 동일 (SQL 친화적 이름)
# -----------------------------------------------------------------------------

# where()는 filter()의 별칭 (alias)
# SQL에 익숙한 사람을 위한 이름

# filter()와 완전히 동일한 동작
df.where(col("age") >= 30).show(5)
df.where("salary > 60000").show(5)

---

## Part 6: 조건부 값 설정 (when / otherwise)

### when/otherwise = SQL의 CASE WHEN

```
SQL:
CASE
    WHEN age >= 50 THEN 'Senior'
    WHEN age >= 30 THEN 'Middle'
    ELSE 'Junior'
END

PySpark:
when(col("age") >= 50, "Senior")
.when(col("age") >= 30, "Middle")
.otherwise("Junior")
```

In [ ]:
# -----------------------------------------------------------------------------
# when(): 단일 조건
# -----------------------------------------------------------------------------

# when(조건, 참일때값): 조건이 참이면 지정값, 거짓이면 null
# otherwise(값): 모든 when 조건이 거짓일 때 값

# 성인 여부 판단
df_adult = df.withColumn(
    "is_adult",
    # when(조건, 참일때값).otherwise(거짓일때값)
    when(col("age") >= 18, "Yes").otherwise("No")
)

df_adult.select("name", "age", "is_adult").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# when(): 다중 조건 (if-elif-else)
# -----------------------------------------------------------------------------

# 여러 when을 체이닝: 첫 번째로 참인 조건의 값 반환
# otherwise: 모든 조건이 거짓일 때 값 (생략 시 null)

# 연령대 분류
df_age_group = df.withColumn(
    "age_group",
    # 첫 번째 참인 조건에서 멈춤
    when(col("age") >= 50, "50대 이상")      # 50 이상이면 "50대 이상"
    .when(col("age") >= 40, "40대")          # 40~49면 "40대"
    .when(col("age") >= 30, "30대")          # 30~39면 "30대"
    .otherwise("20대")                        # 나머지는 "20대"
)

print("=== 연령대 분류 ===")
df_age_group.select("name", "age", "age_group").show(10)

# 연령대별 인원 수
df_age_group.groupBy("age_group").count().show()

In [ ]:
# -----------------------------------------------------------------------------
# when(): 급여 등급 분류 (실무 예제)
# -----------------------------------------------------------------------------

# 급여 등급 분류
df_salary_grade = df.withColumn(
    "salary_grade",
    when(col("salary") >= 100000, "S")       # 10만 이상: S등급
    .when(col("salary") >= 80000, "A")       # 8만~10만: A등급
    .when(col("salary") >= 60000, "B")       # 6만~8만: B등급
    .when(col("salary") >= 40000, "C")       # 4만~6만: C등급
    .otherwise("D")                           # 4만 미만: D등급
)

print("=== 급여 등급 ===")
df_salary_grade.select("name", "salary", "salary_grade").show(10)

# 등급별 인원 분포
df_salary_grade.groupBy("salary_grade").count().orderBy("salary_grade").show()

In [ ]:
# -----------------------------------------------------------------------------
# when(): NULL 처리와 결합
# -----------------------------------------------------------------------------

# NULL을 특정 값으로 대체하면서 조건 분기
df_null_handled = df.withColumn(
    "salary_status",
    when(col("salary").isNull(), "미입력")           # NULL이면 "미입력"
    .when(col("salary") >= 80000, "고연봉")          # 8만 이상
    .when(col("salary") >= 50000, "중연봉")          # 5만~8만
    .otherwise("저연봉")                              # 5만 미만
)

print("=== NULL 처리 포함 급여 상태 ===")
df_null_handled.select("name", "salary", "salary_status").show(15)

---

## Part 7: 정렬, 중복 제거, 제한

| 메서드 | 용도 | 예시 |
|--------|------|------|
| `orderBy()` / `sort()` | 정렬 | `df.orderBy(col("salary").desc())` |
| `distinct()` | 전체 행 중복 제거 | `df.distinct()` |
| `dropDuplicates()` | 특정 컬럼 기준 중복 제거 | `df.dropDuplicates(["col"])` |
| `limit()` | 상위 N개 | `df.limit(10)` |

In [ ]:
# -----------------------------------------------------------------------------
# orderBy(): 정렬
# -----------------------------------------------------------------------------

# orderBy(): 지정 컬럼 기준 정렬
# - 기본: 오름차순 (ASC)
# - 내림차순: col("컬럼").desc()

# 급여 기준 오름차순 (기본)
df.orderBy("salary").select("name", "salary").show(5)

# 급여 기준 내림차순
df.orderBy(col("salary").desc()).select("name", "salary").show(5)

# 여러 컬럼 정렬: 부서 오름차순 → 급여 내림차순
df.orderBy(
    col("department").asc(),    # 부서 오름차순
    col("salary").desc()        # 급여 내림차순
).select("department", "name", "salary").show(10)

In [ ]:
# -----------------------------------------------------------------------------
# distinct(): 중복 제거
# -----------------------------------------------------------------------------

# distinct(): 전체 행이 동일한 중복 제거
# 반환: 고유한 행만 포함된 DataFrame

# 부서 목록 (고유값)
departments = df.select("department").distinct()
print("=== 부서 목록 ===")
departments.show()

# dropDuplicates(): 특정 컬럼 기준 중복 제거
# 해당 컬럼 값이 같은 행 중 첫 번째만 유지
df.dropDuplicates(["department"]).select("department", "name").show()

In [ ]:
# -----------------------------------------------------------------------------
# limit(): 상위 N개
# -----------------------------------------------------------------------------

# limit(n): 상위 n개 행만 반환
# 주의: 정렬 없이 limit만 쓰면 순서가 보장되지 않음

# 상위 5개
df.limit(5).show()

# 급여 상위 5명 (정렬 후 limit)
df.orderBy(col("salary").desc()).limit(5).select("name", "salary").show()

In [ ]:
# -----------------------------------------------------------------------------
# 실무 패턴: Top N 뽑기
# -----------------------------------------------------------------------------

# 부서별 최고 연봉자 1명씩 (간단 버전)
# 실제로는 Window 함수 사용이 더 정확함 (4교시에서 다룸)

# 정렬 후 부서별 첫 번째 행만
top_by_dept = (
    df
    .orderBy(col("salary").desc())
    .dropDuplicates(["department"])
    .select("department", "name", "salary")
    .orderBy("department")
)

print("=== 부서별 최고 연봉자 (간단 버전) ===")
top_by_dept.show()

---

## 퀴즈

Q1. PySpark에서 새 컬럼을 추가할 때 올바른 방법은?

- A) `df["new_col"] = df["old_col"] * 2`
- B) `df.withColumn("new_col", col("old_col") * 2)`
- C) `df.addColumn("new_col", "old_col * 2")`
- D) `df.new_col = df.old_col * 2`

<details>
<summary>정답 보기</summary>

**정답: B) `df.withColumn("new_col", col("old_col") * 2)`**

PySpark DataFrame은 불변(immutable)이라 Pandas처럼 직접 할당할 수 없습니다.
`withColumn()`으로 새 DataFrame을 생성해야 합니다.

</details>

---

Q2. 다음 코드의 문제점은?

```python
df.filter(col("age") > 30 & col("salary") > 50000)
```

- A) filter() 대신 where()를 써야 한다
- B) & 연산자 양쪽에 괄호가 필요하다
- C) col() 대신 문자열을 써야 한다
- D) 문제없이 잘 동작한다

<details>
<summary>정답 보기</summary>

**정답: B) & 연산자 양쪽에 괄호가 필요하다**

Python 연산자 우선순위 때문에 `col("age") > 30 & col("salary")`처럼 해석됩니다.
올바른 코드: `(col("age") > 30) & (col("salary") > 50000)`

</details>

---

Q3. 모든 행에 상수값 "Korea"를 추가하려면?

- A) `df.withColumn("country", "Korea")`
- B) `df.withColumn("country", col("Korea"))`
- C) `df.withColumn("country", lit("Korea"))`
- D) `df.addColumn("country", "Korea")`

<details>
<summary>정답 보기</summary>

**정답: C) `df.withColumn("country", lit("Korea"))`**

`lit()` 함수는 상수(리터럴) 값을 Column 객체로 변환합니다.
문자열 그대로 넣으면 컬럼명으로 해석되어 에러가 발생합니다.

</details>

---

Q4. when/otherwise 표현식에서 모든 조건이 거짓일 때 otherwise()를 생략하면?

- A) 에러가 발생한다
- B) 빈 문자열("")이 된다
- C) 0이 된다
- D) null이 된다

<details>
<summary>정답 보기</summary>

**정답: D) null이 된다**

otherwise()를 생략하면 모든 when 조건이 거짓일 때 null이 반환됩니다.
명시적으로 처리하려면 otherwise()를 항상 작성하는 것이 좋습니다.

</details>

---

## 과제: 직원 데이터 변환

### Step 1: 컬럼 선택 및 이름 변경

`emp_id`, `name`, `department`, `salary` 컬럼만 선택하고,
`emp_id`를 `employee_id`로, `department`를 `dept`로 변경하세요.

<details>
<summary>힌트</summary>

`select()` 후 `withColumnRenamed()` 체이닝

</details>

<details>
<summary>모범 답안</summary>

```python
result = (
    df.select("emp_id", "name", "department", "salary")
    .withColumnRenamed("emp_id", "employee_id")
    .withColumnRenamed("department", "dept")
)
```

</details>

### Step 2: 급여 등급 컬럼 추가

급여 기준으로 등급을 분류하는 `grade` 컬럼을 추가하세요:
- 100,000 이상: "A"
- 70,000 이상: "B"
- 그 외: "C"

<details>
<summary>힌트</summary>

`when().when().otherwise()` 사용

</details>

### Step 3: 필터링 및 정렬

B등급 이상(A 또는 B)인 직원만 필터링하고, 급여 내림차순으로 정렬하세요.

<details>
<summary>힌트</summary>

`filter(col("grade").isin("A", "B"))` 사용

</details>

### 보너스: 부서별 A등급 인원 수

부서별로 A등급 직원이 몇 명인지 집계하세요.

---

## 핵심 요약

### 컬럼 참조

```python
col("컬럼명")  # 가장 권장 (연산 가능)
"컬럼명"      # select, groupBy에서 단순 참조
df.컬럼명     # 조인 시 테이블 구분
```

### 컬럼 조작

| 작업 | 코드 |
|------|------|
| 선택 | `df.select("col1", "col2")` |
| 추가 | `df.withColumn("new", col("old") * 2)` |
| 상수 추가 | `df.withColumn("country", lit("Korea"))` |
| 이름 변경 | `df.withColumnRenamed("old", "new")` |
| 삭제 | `df.drop("col1", "col2")` |

### 필터링

```python
df.filter(col("age") >= 30)                          # 기본
df.filter((col("age") >= 30) & (col("salary") > 50000))  # AND (괄호 필수!)
df.filter(col("dept").isin("A", "B"))                # IN
df.filter("age >= 30 AND salary > 50000")            # SQL 스타일
```

### 조건부 값 (when/otherwise)

```python
when(col("age") >= 50, "Senior")
.when(col("age") >= 30, "Middle")
.otherwise("Junior")
```

### 정렬/중복/제한

```python
df.orderBy(col("salary").desc())     # 정렬
df.distinct()                         # 중복 제거
df.dropDuplicates(["col"])           # 특정 컬럼 기준 중복 제거
df.limit(10)                         # 상위 10개
```

In [ ]:
# 세션 유지 (다음 교시 계속)
print("\n2교시 완료! 다음 교시: 집계와 조인")

---


# PySpark 기초 API - 3교시: 집계와 조인

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- `groupBy()`와 `agg()`로 그룹별 집계를 수행할 수 있다
- `count()`, `sum()`, `avg()`, `min()`, `max()` 집계 함수를 활용할 수 있다
- 다양한 조인 타입(inner, left, right, outer)을 이해하고 적용할 수 있다
- `pivot()`으로 데이터를 피벗 테이블 형태로 변환할 수 있다
- Spark SQL을 DataFrame과 함께 활용할 수 있다

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정: SparkSession 및 테스트 데이터 준비
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, count, sum, avg, min, max,
    countDistinct, first, last, collect_list, collect_set,
    round as spark_round, expr
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
import numpy as np
import os

# SparkSession 생성
spark = SparkSession.builder \
    .appName("PySpark-Aggregation-Join") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

# 테스트 데이터 디렉토리
os.makedirs("/tmp/spark_tutorial", exist_ok=True)

np.random.seed(42)

# -----------------------------------------------------------------------------
# 테스트 데이터 1: 직원 정보
# -----------------------------------------------------------------------------
employees = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 51)],
    "name": [f"Employee_{i}" for i in range(1, 51)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", "HR", "Finance"], 50
    ),
    "salary": np.random.randint(40000, 120000, 50),
    "age": np.random.randint(25, 55, 50),
    "hire_year": np.random.choice([2020, 2021, 2022, 2023, 2024], 50),
})
employees.to_csv("/tmp/spark_tutorial/employees.csv", index=False)

# -----------------------------------------------------------------------------
# 테스트 데이터 2: 부서 정보 (조인용)
# -----------------------------------------------------------------------------
departments = pd.DataFrame({
    "dept_name": ["Engineering", "Sales", "Marketing", "HR", "Finance", "Legal"],
    "dept_head": ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "budget": [500000, 300000, 200000, 150000, 400000, 100000],
    "location": ["Seoul", "Busan", "Seoul", "Daegu", "Seoul", "Incheon"],
})
departments.to_csv("/tmp/spark_tutorial/departments.csv", index=False)

# -----------------------------------------------------------------------------
# 테스트 데이터 3: 매출 데이터 (시계열)
# -----------------------------------------------------------------------------
sales = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=100, freq="D").strftime("%Y-%m-%d"),
    "product": np.random.choice(["A", "B", "C"], 100),
    "region": np.random.choice(["Seoul", "Busan", "Daegu"], 100),
    "amount": np.random.randint(100, 1000, 100),
    "quantity": np.random.randint(1, 20, 100),
})
sales.to_csv("/tmp/spark_tutorial/sales.csv", index=False)

# DataFrame 로드
df_emp = spark.read.csv("/tmp/spark_tutorial/employees.csv", header=True, inferSchema=True)
df_dept = spark.read.csv("/tmp/spark_tutorial/departments.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("/tmp/spark_tutorial/sales.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"직원: {df_emp.count()}명, 부서: {df_dept.count()}개, 매출: {df_sales.count()}건")

---

## Part 1: 전체 집계 (agg)

### 집계 함수 종류

| 함수 | 설명 | 예시 |
|------|------|------|
| `count(col)` | 개수 (NULL 제외) | `count("*")` 또는 `count("col")` |
| `sum(col)` | 합계 | `sum("salary")` |
| `avg(col)` | 평균 | `avg("age")` |
| `min(col)` | 최소값 | `min("salary")` |
| `max(col)` | 최대값 | `max("salary")` |
| `countDistinct(col)` | 고유값 개수 | `countDistinct("department")` |

In [ ]:
# -----------------------------------------------------------------------------
# agg(): 전체 데이터 집계 (groupBy 없이)
# -----------------------------------------------------------------------------

# agg(): 집계 함수들을 적용하여 결과 DataFrame 반환
# 여러 집계를 한 번에 계산 가능

# 전체 직원 통계
total_stats = df_emp.agg(
    count("*").alias("총인원"),                    # 전체 행 수
    count("salary").alias("급여있는인원"),          # NULL 제외 카운트
    sum("salary").alias("총급여"),                 # 급여 합계
    avg("salary").alias("평균급여"),               # 급여 평균
    min("salary").alias("최소급여"),               # 최소 급여
    max("salary").alias("최대급여"),               # 최대 급여
    countDistinct("department").alias("부서수"),   # 고유 부서 수
)

print("=== 전체 직원 통계 ===")
total_stats.show()

In [ ]:
# -----------------------------------------------------------------------------
# 단일 집계 함수 사용 (shortcut)
# -----------------------------------------------------------------------------

# 간단한 집계는 agg() 없이 직접 호출 가능
# 결과는 단일 값이 아닌 DataFrame

# 전체 행 수
print(f"전체 직원 수: {df_emp.count()}")

# 고유 부서 수
unique_depts = df_emp.select("department").distinct().count()
print(f"부서 종류: {unique_depts}개")

---

## Part 2: 그룹별 집계 (groupBy + agg)

### groupBy 패턴

```python
df.groupBy("그룹컬럼")                    # 단일 컬럼
df.groupBy("컬럼1", "컬럼2")              # 여러 컬럼
df.groupBy(col("컬럼"))                   # col() 사용
```

### agg 내 집계 함수

```python
.agg(
    count("*").alias("별칭1"),           # alias로 결과 컬럼명 지정
    sum("컬럼").alias("별칭2"),
    avg("컬럼").alias("별칭3"),
)
```

In [ ]:
# -----------------------------------------------------------------------------
# groupBy() + agg(): 부서별 집계
# -----------------------------------------------------------------------------

# groupBy(컬럼): 해당 컬럼 값이 같은 행들을 그룹화
# agg(): 각 그룹에 대해 집계 함수 적용

# 부서별 통계
dept_stats = df_emp.groupBy("department").agg(
    count("*").alias("인원수"),                           # 부서별 인원
    spark_round(avg("salary"), 2).alias("평균급여"),      # 평균 급여 (소수 2자리)
    min("salary").alias("최소급여"),                      # 최소 급여
    max("salary").alias("최대급여"),                      # 최대 급여
    sum("salary").alias("총급여"),                        # 급여 합계
)

print("=== 부서별 통계 ===")
dept_stats.orderBy(col("인원수").desc()).show()

In [ ]:
# -----------------------------------------------------------------------------
# groupBy(): 여러 컬럼으로 그룹화
# -----------------------------------------------------------------------------

# 여러 컬럼을 쉼표로 나열하면 조합별로 그룹화
# (부서, 입사년도) 조합별 통계

dept_year_stats = df_emp.groupBy("department", "hire_year").agg(
    count("*").alias("인원수"),
    spark_round(avg("salary"), 0).alias("평균급여"),
)

print("=== 부서 + 입사년도별 통계 ===")
dept_year_stats.orderBy("department", "hire_year").show(15)

In [ ]:
# -----------------------------------------------------------------------------
# groupBy(): 간단한 집계 shortcut
# -----------------------------------------------------------------------------

# groupBy 후 바로 count(), sum() 등 호출 가능 (단일 집계)
# agg() 생략 가능

# 부서별 인원 수 (간단 버전)
df_emp.groupBy("department").count().show()

# 부서별 급여 합계 (간단 버전)
df_emp.groupBy("department").sum("salary").show()

# 부서별 평균 급여 (간단 버전)
df_emp.groupBy("department").avg("salary").show()

In [ ]:
# -----------------------------------------------------------------------------
# 조건부 집계: when과 결합
# -----------------------------------------------------------------------------

# when을 사용해 조건에 맞는 행만 집계
# SQL의 COUNT(CASE WHEN ... THEN 1 END)와 동일

# 부서별 고연봉자(8만 이상) 수
conditional_count = df_emp.groupBy("department").agg(
    count("*").alias("전체인원"),
    # when 조건이 참인 경우만 카운트
    count(when(col("salary") >= 80000, 1)).alias("고연봉자수"),
    count(when(col("salary") < 50000, 1)).alias("저연봉자수"),
    count(when(col("age") >= 40, 1)).alias("40대이상"),
)

print("=== 조건부 집계 ===")
conditional_count.show()

---

## Part 3: 특수 집계 함수

### 리스트/집합 수집

| 함수 | 설명 | 결과 타입 |
|------|------|----------|
| `collect_list()` | 그룹의 모든 값을 배열로 | Array (중복 포함) |
| `collect_set()` | 그룹의 고유 값을 집합으로 | Array (중복 제거) |
| `first()` | 첫 번째 값 | 단일 값 |
| `last()` | 마지막 값 | 단일 값 |

In [ ]:
# -----------------------------------------------------------------------------
# collect_list(), collect_set(): 값들을 배열로 모으기
# -----------------------------------------------------------------------------

# collect_list(): 그룹의 모든 값을 배열로 수집 (중복 포함)
# collect_set(): 그룹의 고유 값만 배열로 수집 (중복 제거)

# 부서별 직원 이름 목록
name_list = df_emp.groupBy("department").agg(
    collect_list("name").alias("직원목록"),      # 모든 이름
    collect_set("hire_year").alias("입사년도"),  # 고유 입사년도
    count("*").alias("인원수"),
)

print("=== 부서별 직원 목록 ===")
name_list.show(truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# first(), last(): 첫 번째/마지막 값
# -----------------------------------------------------------------------------

# first(): 그룹의 첫 번째 값 반환
# last(): 그룹의 마지막 값 반환
# 주의: 정렬 없이 사용하면 결과가 비결정적

# 부서별 첫 번째 직원 (정렬 없이는 순서 보장 안 됨)
first_emp = df_emp.groupBy("department").agg(
    first("name").alias("첫번째직원"),
    first("salary").alias("해당급여"),
)

print("=== 부서별 첫 번째 직원 ===")
first_emp.show()

---

## Part 4: 조인 (Join)

### 조인 타입 비교

```
df1 (직원)              df2 (부서)
┌────────────────┐     ┌────────────────┐
│ name │ dept    │     │ dept │ head    │
├────────────────┤     ├────────────────┤
│ Kim  │ Eng     │     │ Eng  │ Alice   │
│ Lee  │ Sales   │     │ Sales│ Bob     │
│ Park │ HR      │     │ Legal│ Charlie │  ← 직원 없음
│ Choi │ IT      │     └────────────────┘
└────────────────┘       ↑ IT 부서 없음
```

| 조인 타입 | 결과 | SQL 비유 |
|-----------|------|----------|
| `inner` | 양쪽에 모두 있는 것만 | INNER JOIN |
| `left` | 왼쪽 전체 + 오른쪽 매칭 | LEFT OUTER JOIN |
| `right` | 오른쪽 전체 + 왼쪽 매칭 | RIGHT OUTER JOIN |
| `outer` | 양쪽 전체 | FULL OUTER JOIN |
| `left_semi` | 왼쪽 중 매칭되는 것만 (왼쪽 컬럼만) | WHERE EXISTS |
| `left_anti` | 왼쪽 중 매칭 안 되는 것만 | WHERE NOT EXISTS |

In [ ]:
# -----------------------------------------------------------------------------
# 조인 데이터 확인
# -----------------------------------------------------------------------------

print("=== 직원 데이터 ===")
df_emp.select("emp_id", "name", "department").show(5)

print("=== 부서 데이터 ===")
df_dept.show()

In [ ]:
# -----------------------------------------------------------------------------
# Inner Join: 양쪽에 모두 있는 것만
# -----------------------------------------------------------------------------

# join(df2, 조건, how="inner")
# - df2: 조인할 DataFrame
# - 조건: 조인 키 (문자열 또는 조건식)
# - how: 조인 타입 (기본값 "inner")

# 직원과 부서 정보 조인
# department(직원) == dept_name(부서)인 행만 결합
df_joined = df_emp.join(
    df_dept,                                    # 조인할 DataFrame
    df_emp.department == df_dept.dept_name,    # 조인 조건
    "inner"                                     # 조인 타입
)

print("=== Inner Join 결과 ===")
df_joined.select(
    "name", "department", "salary", "dept_head", "location"
).show(10)

In [ ]:
# -----------------------------------------------------------------------------
# 조인 키가 같은 이름일 때: 문자열로 지정
# -----------------------------------------------------------------------------

# 조인 키 컬럼명이 같으면 문자열로 간단히 지정
# 결과에서 조인 키 컬럼이 하나만 남음 (중복 제거)

# 데이터 준비: 컬럼명 맞추기
df_dept_renamed = df_dept.withColumnRenamed("dept_name", "department")

# 같은 이름으로 조인
df_simple_join = df_emp.join(
    df_dept_renamed,
    "department",       # 양쪽에 같은 이름의 컬럼
    "inner"
)

print("=== 같은 컬럼명으로 조인 ===")
df_simple_join.select("name", "department", "salary", "dept_head").show(5)

In [ ]:
# -----------------------------------------------------------------------------
# Left Join: 왼쪽 전체 + 오른쪽 매칭
# -----------------------------------------------------------------------------

# left join: 왼쪽 DataFrame의 모든 행 유지
# 오른쪽에 매칭되는 행이 없으면 NULL

# IT 부서 직원 추가 (df_dept에는 IT 부서 없음)
df_emp_with_it = df_emp.union(
    spark.createDataFrame([
        ("E999", "IT_Person", "IT", 70000, 30, 2024)
    ], df_emp.columns)
)

# Left Join: 모든 직원 + 부서 정보 (없으면 NULL)
df_left = df_emp_with_it.join(
    df_dept,
    df_emp_with_it.department == df_dept.dept_name,
    "left"
)

print("=== Left Join 결과 (IT 부서 직원 포함) ===")
df_left.filter(col("dept_name").isNull()).select(
    "name", "department", "dept_name", "dept_head"
).show()

In [ ]:
# -----------------------------------------------------------------------------
# Left Semi Join: 존재 여부만 확인 (왼쪽 컬럼만 반환)
# -----------------------------------------------------------------------------

# left_semi: 오른쪽에 매칭되는 왼쪽 행만 반환
# 오른쪽 컬럼은 결과에 포함되지 않음
# SQL의 WHERE EXISTS와 동일

# 부서 정보가 있는 직원만 (부서 컬럼 없이)
df_exists = df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "left_semi"
)

print("=== Left Semi Join (존재하는 부서의 직원만) ===")
print(f"컬럼: {df_exists.columns}")
df_exists.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# Left Anti Join: 매칭 안 되는 것만 (NOT EXISTS)
# -----------------------------------------------------------------------------

# left_anti: 오른쪽에 매칭되지 않는 왼쪽 행만 반환
# SQL의 WHERE NOT EXISTS와 동일

# IT 직원 포함 데이터로 테스트
# 부서 정보가 없는 직원 찾기
df_not_exists = df_emp_with_it.join(
    df_dept,
    df_emp_with_it.department == df_dept.dept_name,
    "left_anti"
)

print("=== Left Anti Join (부서 정보 없는 직원) ===")
df_not_exists.show()

In [ ]:
# -----------------------------------------------------------------------------
# Broadcast Join: 작은 테이블 최적화
# -----------------------------------------------------------------------------

from pyspark.sql.functions import broadcast

# broadcast(): 작은 DataFrame을 모든 워커에 복사
# 대용량 + 소용량 조인 시 성능 향상
# 셔플 없이 각 워커에서 로컬 조인

# 부서 테이블은 작으므로 broadcast 적용
df_broadcast = df_emp.join(
    broadcast(df_dept),                        # 작은 테이블에 broadcast 적용
    df_emp.department == df_dept.dept_name,
    "inner"
)

print("=== Broadcast Join ===")
df_broadcast.select("name", "department", "dept_head").show(5)

---

## Part 5: 피벗 (Pivot)

### Pivot이란?

```
원본 데이터:                    피벗 후:
┌──────┬──────┬───────┐       ┌──────┬─────┬─────┬─────┐
│ year │ qtr  │ sales │       │ year │ Q1  │ Q2  │ Q3  │
├──────┼──────┼───────┤  →    ├──────┼─────┼─────┼─────┤
│ 2024 │ Q1   │ 100   │       │ 2024 │ 100 │ 150 │ 200 │
│ 2024 │ Q2   │ 150   │       │ 2025 │ 120 │ 180 │ ... │
│ 2024 │ Q3   │ 200   │       └──────┴─────┴─────┴─────┘
│ 2025 │ Q1   │ 120   │
│ 2025 │ Q2   │ 180   │       행 → 컬럼으로 변환
└──────┴──────┴───────┘
```

In [ ]:
# -----------------------------------------------------------------------------
# pivot(): 행을 컬럼으로 변환
# -----------------------------------------------------------------------------

# 매출 데이터로 피벗 테이블 생성
# 지역별, 제품별 매출 합계

# pivot(컬럼): 해당 컬럼의 고유값들이 새 컬럼명이 됨
# pivot(컬럼, [값목록]): 특정 값만 피벗 (성능 향상)

pivot_sales = df_sales.groupBy("region").pivot(
    "product",              # 피벗할 컬럼 (A, B, C가 컬럼명이 됨)
    ["A", "B", "C"]         # 피벗할 값 목록 (명시하면 성능 향상)
).sum("amount")             # 집계 함수

print("=== 지역별 제품별 매출 (피벗) ===")
pivot_sales.show()

In [ ]:
# -----------------------------------------------------------------------------
# pivot(): 여러 집계
# -----------------------------------------------------------------------------

# 피벗 + 여러 집계 함수
# agg() 안에 여러 집계 지정

pivot_multi = df_sales.groupBy("region").pivot(
    "product", ["A", "B", "C"]
).agg(
    sum("amount").alias("총매출"),
    avg("amount").alias("평균매출"),
)

print("=== 피벗 + 여러 집계 ===")
pivot_multi.show()

---

## Part 6: Spark SQL 연동

### DataFrame ↔ SQL

```python
# 1. DataFrame을 SQL 테이블로 등록
df.createOrReplaceTempView("테이블명")

# 2. SQL 쿼리 실행 → DataFrame 반환
result = spark.sql("SELECT * FROM 테이블명")
```

In [ ]:
# -----------------------------------------------------------------------------
# createOrReplaceTempView(): SQL 테이블 등록
# -----------------------------------------------------------------------------

# createOrReplaceTempView(이름): 임시 뷰로 등록
# - 해당 SparkSession 내에서만 유효
# - 세션 종료 시 자동 삭제

# DataFrame을 SQL 테이블로 등록
df_emp.createOrReplaceTempView("employees")
df_dept.createOrReplaceTempView("departments")

print("SQL 테이블 등록 완료: employees, departments")

In [ ]:
# -----------------------------------------------------------------------------
# spark.sql(): SQL 쿼리 실행
# -----------------------------------------------------------------------------

# spark.sql(쿼리문): SQL 실행 후 DataFrame 반환
# 복잡한 로직을 SQL로 작성 가능

# SQL로 부서별 통계
sql_result = spark.sql("""
    SELECT
        department,
        COUNT(*) as 인원수,
        ROUND(AVG(salary), 2) as 평균급여,
        MAX(salary) as 최고급여
    FROM employees
    GROUP BY department
    ORDER BY 인원수 DESC
""")

print("=== SQL 쿼리 결과 ===")
sql_result.show()

In [ ]:
# -----------------------------------------------------------------------------
# SQL: 조인
# -----------------------------------------------------------------------------

# SQL로 조인 쿼리
sql_join = spark.sql("""
    SELECT
        e.name,
        e.department,
        e.salary,
        d.dept_head,
        d.location
    FROM employees e
    JOIN departments d ON e.department = d.dept_name
    WHERE e.salary >= 70000
    ORDER BY e.salary DESC
""")

print("=== SQL 조인 결과 ===")
sql_join.show(10)

In [ ]:
# -----------------------------------------------------------------------------
# expr(): SQL 표현식을 DataFrame에서 사용
# -----------------------------------------------------------------------------

# expr(): SQL 표현식을 Column으로 변환
# select(), withColumn() 등에서 SQL 문법 사용 가능

# SQL 표현식으로 새 컬럼 추가
df_with_expr = df_emp.withColumn(
    "salary_grade",
    expr("CASE WHEN salary >= 80000 THEN 'High' ELSE 'Normal' END")
).withColumn(
    "bonus",
    expr("salary * 0.1")  # 급여의 10%
)

print("=== expr() 사용 예시 ===")
df_with_expr.select("name", "salary", "salary_grade", "bonus").show(5)

---

## 퀴즈

Q1. 다음 코드의 결과로 올바른 것은?

```python
df.groupBy("dept").agg(count("*").alias("cnt"))
```

- A) 전체 행 수를 계산한다
- B) 부서별 NULL이 아닌 행 수를 계산한다
- C) 부서별 전체 행 수를 계산한다
- D) 에러가 발생한다

<details>
<summary>정답 보기</summary>

**정답: C) 부서별 전체 행 수를 계산한다**

`count("*")`은 NULL 포함 모든 행을 카운트합니다.
`count("특정컬럼")`은 해당 컬럼이 NULL이 아닌 행만 카운트합니다.

</details>

---

Q2. 왼쪽 DataFrame의 모든 행을 유지하면서 오른쪽과 조인하려면?

- A) `df1.join(df2, 조건, "inner")`
- B) `df1.join(df2, 조건, "left")`
- C) `df1.join(df2, 조건, "right")`
- D) `df1.join(df2, 조건, "outer")`

<details>
<summary>정답 보기</summary>

**정답: B) `df1.join(df2, 조건, "left")`**

left join은 왼쪽 DataFrame의 모든 행을 유지하고,
매칭되지 않는 행은 오른쪽 컬럼이 NULL로 채워집니다.

</details>

---

Q3. 작은 테이블을 조인할 때 성능을 높이는 방법은?

- A) `cache()`로 캐싱한다
- B) `repartition()`으로 파티션을 늘린다
- C) `broadcast()`를 사용한다
- D) `coalesce()`로 파티션을 줄인다

<details>
<summary>정답 보기</summary>

**정답: C) `broadcast()`를 사용한다**

`broadcast(df)`는 작은 DataFrame을 모든 워커 노드에 복사하여
셔플 없이 로컬에서 조인할 수 있게 합니다.

</details>

---

Q4. pivot()의 역할은?

- A) 행과 열을 바꾼다
- B) 특정 컬럼의 고유값을 새 컬럼명으로 변환한다
- C) 데이터를 정렬한다
- D) 중복을 제거한다

<details>
<summary>정답 보기</summary>

**정답: B) 특정 컬럼의 고유값을 새 컬럼명으로 변환한다**

pivot은 행 데이터를 컬럼으로 변환하여 크로스탭 형태로 만듭니다.
예: product 컬럼의 [A, B, C] 값이 컬럼명 A, B, C가 됩니다.

</details>

---

## 과제: 매출 데이터 분석

### Step 1: 지역별 총매출 집계

지역(region)별 총매출(amount 합계)과 평균매출을 구하세요.
총매출 내림차순으로 정렬하세요.

<details>
<summary>힌트</summary>

`groupBy("region").agg(sum(...), avg(...)).orderBy(...)`

</details>

<details>
<summary>모범 답안</summary>

```python
df_sales.groupBy("region").agg(
    sum("amount").alias("총매출"),
    round(avg("amount"), 2).alias("평균매출"),
).orderBy(col("총매출").desc()).show()
```

</details>

### Step 2: 지역-제품 조합별 집계

지역과 제품 조합별로 판매 건수와 총매출을 구하세요.

<details>
<summary>힌트</summary>

`groupBy("region", "product")`

</details>

### Step 3: 피벗 테이블 만들기

지역(행) × 제품(열) 형태로 총매출 피벗 테이블을 만드세요.

<details>
<summary>힌트</summary>

`groupBy("region").pivot("product").sum("amount")`

</details>

### 보너스: SQL로 동일한 분석

SQL을 사용하여 Step 1과 동일한 결과를 얻어보세요.

---

## 핵심 요약

### 전체 집계

```python
df.agg(
    count("*").alias("총건수"),
    sum("amount").alias("총합"),
    avg("amount").alias("평균"),
)
```

### 그룹별 집계

```python
df.groupBy("컬럼1", "컬럼2").agg(
    count("*").alias("건수"),
    sum("금액").alias("합계"),
)
```

### 조인

| 조인 타입 | 코드 |
|-----------|------|
| Inner | `df1.join(df2, 조건, "inner")` |
| Left | `df1.join(df2, 조건, "left")` |
| Broadcast | `df1.join(broadcast(df2), 조건)` |

### 피벗

```python
df.groupBy("행").pivot("열", [값목록]).sum("집계컬럼")
```

### SQL 연동

```python
df.createOrReplaceTempView("테이블명")
spark.sql("SELECT * FROM 테이블명")
```

In [ ]:
# 세션 유지 (다음 교시 계속)
print("\n3교시 완료! 다음 교시: 실무 핵심 패턴")

---


# PySpark 기초 API - 4교시: 실무 핵심 패턴

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- NULL 값을 효과적으로 처리할 수 있다 (dropna, fillna, coalesce)
- 문자열/날짜 함수를 활용할 수 있다
- Window 함수로 순위, 누적합, 이동평균을 계산할 수 있다
- 실무에서 자주 사용하는 패턴들을 적용할 수 있다

---

## 환경 설정

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when,
    # NULL 처리
    coalesce, isnan,
    # 문자열 함수
    concat, concat_ws, substring, length, trim, ltrim, rtrim,
    upper, lower, initcap, regexp_replace, regexp_extract, split,
    lpad, rpad,
    # 날짜/시간 함수
    current_date, current_timestamp, to_date, to_timestamp, date_format,
    year, month, dayofmonth, dayofweek, hour, minute,
    date_add, date_sub, datediff, months_between, trunc,
    # 집계/윈도우
    count, sum, avg, min, max, first, last,
    row_number, rank, dense_rank, lag, lead,
    # 기타
    round as spark_round, expr,
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
import pandas as pd
import numpy as np
import os

# SparkSession 생성
spark = SparkSession.builder \
    .appName("PySpark-Advanced-Patterns") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

os.makedirs("/tmp/spark_tutorial", exist_ok=True)
np.random.seed(42)

# -----------------------------------------------------------------------------
# 테스트 데이터: 다양한 상황을 포함한 직원 데이터
# -----------------------------------------------------------------------------
employees = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 31)],
    "name": [f"  Employee {i}  " for i in range(1, 31)],  # 앞뒤 공백
    "email": [f"emp{i}@company.com" for i in range(1, 31)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", None], 30
    ),
    "salary": [
        50000, None, 70000, 80000, None,  # NULL 포함
        60000, 90000, 55000, None, 75000,
        65000, 85000, None, 72000, 68000,
        None, 95000, 62000, 78000, None,
        58000, 82000, 67000, None, 73000,
        69000, 88000, None, 76000, 71000
    ],
    "join_date": pd.date_range("2020-01-15", periods=30, freq="45D").strftime("%Y-%m-%d"),
})
employees.to_csv("/tmp/spark_tutorial/employees_advanced.csv", index=False)

# 매출 시계열 데이터
sales = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=90, freq="D").strftime("%Y-%m-%d"),
    "product": np.random.choice(["A", "B", "C"], 90),
    "region": np.random.choice(["Seoul", "Busan", "Daegu"], 90),
    "amount": np.random.randint(100, 1000, 90),
})
sales.to_csv("/tmp/spark_tutorial/sales_ts.csv", index=False)

# DataFrame 로드
df = spark.read.csv("/tmp/spark_tutorial/employees_advanced.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("/tmp/spark_tutorial/sales_ts.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"직원: {df.count()}명, 매출: {df_sales.count()}건")
df.show(10)

---

## Part 1: NULL 처리

### NULL 관련 메서드

| 메서드 | 설명 | 예시 |
|--------|------|------|
| `isNull()` | NULL 여부 확인 | `col("x").isNull()` |
| `isNotNull()` | NULL이 아닌지 확인 | `col("x").isNotNull()` |
| `dropna()` | NULL 행 제거 | `df.dropna()` |
| `fillna()` | NULL을 다른 값으로 채우기 | `df.fillna(0)` |
| `coalesce()` | 첫 번째 non-null 값 반환 | `coalesce(col1, col2)` |

In [ ]:
# -----------------------------------------------------------------------------
# NULL 확인
# -----------------------------------------------------------------------------

# isNull(), isNotNull(): NULL 여부 확인
print("=== NULL인 행 (salary) ===")
df.filter(col("salary").isNull()).select("emp_id", "name", "salary").show()

print("=== NULL이 아닌 행 (salary) ===")
df.filter(col("salary").isNotNull()).count()

# NULL 개수 세기
null_counts = df.agg(
    count(when(col("salary").isNull(), 1)).alias("salary_null"),
    count(when(col("department").isNull(), 1)).alias("dept_null"),
)
print("=== NULL 개수 ===")
null_counts.show()

In [ ]:
# -----------------------------------------------------------------------------
# dropna(): NULL 행 제거
# -----------------------------------------------------------------------------

# dropna(): NULL이 하나라도 있는 행 제거
# dropna(how="all"): 모든 컬럼이 NULL인 행만 제거
# dropna(subset=[]): 특정 컬럼에서만 NULL 검사

# 기본: 어떤 컬럼이든 NULL이면 제거
df_no_null = df.dropna()
print(f"=== dropna() 후 행 수: {df_no_null.count()} (원본: {df.count()}) ===")

# 특정 컬럼에 NULL이 있는 행만 제거
df_no_salary_null = df.dropna(subset=["salary"])
print(f"=== salary NULL 제거 후: {df_no_salary_null.count()} ===")

# 여러 컬럼 지정
df_clean = df.dropna(subset=["salary", "department"])
print(f"=== salary, department NULL 제거 후: {df_clean.count()} ===")

In [ ]:
# -----------------------------------------------------------------------------
# fillna(): NULL을 다른 값으로 채우기
# -----------------------------------------------------------------------------

# fillna(값): 모든 컬럼의 NULL을 해당 값으로 채움
# fillna(값, subset=[]): 특정 컬럼만 채움
# fillna({"컬럼": 값}): 컬럼별로 다른 값으로 채움

# 단일 값으로 채우기 (타입 일치 필요)
df_filled_salary = df.fillna(0, subset=["salary"])
print("=== salary NULL → 0 ===")
df_filled_salary.filter(col("emp_id").isin("E002", "E005")).show()

# 컬럼별로 다른 값 채우기 (딕셔너리)
df_filled = df.fillna({
    "salary": 0,                    # 급여 NULL → 0
    "department": "Unknown"         # 부서 NULL → Unknown
})

print("=== 컬럼별 다른 값으로 채우기 ===")
df_filled.filter(
    col("emp_id").isin("E002", "E005", "E009")
).show()

In [ ]:
# -----------------------------------------------------------------------------
# coalesce(): 첫 번째 non-null 값 반환
# -----------------------------------------------------------------------------

# coalesce(col1, col2, ...): 왼쪽부터 확인하여 첫 번째 non-null 값 반환
# SQL의 COALESCE와 동일
# 여러 컬럼 중 대체 값을 찾을 때 유용

# 예시: primary_phone → secondary_phone → "연락처없음" 순서로 채우기
df_contact = spark.createDataFrame([
    ("E001", "010-1234-5678", None),
    ("E002", None, "02-555-1234"),
    ("E003", None, None),
    ("E004", "010-9999-8888", "02-111-2222"),
], ["emp_id", "primary_phone", "secondary_phone"])

# coalesce: 첫 번째 non-null 값 선택
df_with_phone = df_contact.withColumn(
    "contact",
    coalesce(
        col("primary_phone"),       # 1순위: 휴대폰
        col("secondary_phone"),     # 2순위: 유선전화
        lit("연락처없음")             # 3순위: 기본값
    )
)

print("=== coalesce 예시 ===")
df_with_phone.show()

---

## Part 2: 문자열 함수

### 주요 문자열 함수

| 함수 | 설명 | 예시 |
|------|------|------|
| `concat()` | 문자열 합치기 | `concat(col1, col2)` |
| `concat_ws()` | 구분자로 합치기 | `concat_ws("-", col1, col2)` |
| `substring()` | 부분 문자열 | `substring(col, 1, 3)` |
| `trim()` | 앞뒤 공백 제거 | `trim(col)` |
| `upper()`/`lower()` | 대소문자 변환 | `upper(col)` |
| `split()` | 문자열 분리 → 배열 | `split(col, ",")` |
| `regexp_replace()` | 정규식 치환 | `regexp_replace(col, "패턴", "대체")` |

In [ ]:
# -----------------------------------------------------------------------------
# concat(), concat_ws(): 문자열 합치기
# -----------------------------------------------------------------------------

# concat(): 여러 문자열/컬럼을 연결
# concat_ws(구분자, ...): 구분자로 연결 (NULL 자동 건너뜀)

df_concat = df.select(
    col("emp_id"),
    col("name"),
    # concat: 단순 연결
    concat(col("emp_id"), lit("-"), col("name")).alias("id_name"),
    # concat_ws: 구분자로 연결 (NULL은 건너뜀)
    concat_ws("_", col("emp_id"), col("department")).alias("id_dept"),
)

print("=== 문자열 합치기 ===")
df_concat.show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# trim(), upper(), lower(): 공백/대소문자
# -----------------------------------------------------------------------------

# trim(): 앞뒤 공백 제거
# ltrim(): 왼쪽 공백 제거
# rtrim(): 오른쪽 공백 제거
# upper(): 대문자로
# lower(): 소문자로
# initcap(): 첫 글자만 대문자

# 이름에 앞뒤 공백이 있음 → trim으로 정리
df_cleaned = df.withColumn(
    "name_trimmed",
    trim(col("name"))                    # 앞뒤 공백 제거
).withColumn(
    "name_upper",
    upper(trim(col("name")))             # 대문자 변환
).withColumn(
    "name_lower",
    lower(trim(col("name")))             # 소문자 변환
)

print("=== 공백/대소문자 처리 ===")
df_cleaned.select("name", "name_trimmed", "name_upper", "name_lower").show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# substring(): 부분 문자열
# -----------------------------------------------------------------------------

# substring(컬럼, 시작위치, 길이)
# 주의: 시작 위치는 1부터 (0이 아님!)

df_substr = df.select(
    col("email"),
    # 이메일에서 @ 앞부분 추출 (간단 버전, 정확히는 split 사용)
    substring(col("email"), 1, 4).alias("first_4_chars"),
)

print("=== substring ===")
df_substr.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# split(): 문자열 분리 → 배열
# -----------------------------------------------------------------------------

# split(컬럼, 구분자): 문자열을 배열로 분리
# 결과[0], 결과[1] 등으로 인덱싱

df_split = df.select(
    col("email"),
    # @ 기준으로 분리
    split(col("email"), "@").alias("email_parts"),
    # 배열의 첫 번째 요소 (사용자명)
    split(col("email"), "@")[0].alias("username"),
    # 배열의 두 번째 요소 (도메인)
    split(col("email"), "@")[1].alias("domain"),
)

print("=== split ===")
df_split.show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# regexp_replace(), regexp_extract(): 정규식
# -----------------------------------------------------------------------------

# regexp_replace(컬럼, 패턴, 대체문자): 패턴 매칭 부분 치환
# regexp_extract(컬럼, 패턴, 그룹번호): 패턴 매칭 부분 추출

df_regex = df.select(
    col("name"),
    col("email"),
    # 숫자만 추출 (emp1 → 1)
    regexp_extract(col("email"), r"emp(\d+)", 1).alias("emp_number"),
    # 특수문자 제거
    regexp_replace(col("email"), r"@.*", "").alias("without_domain"),
)

print("=== 정규식 ===")
df_regex.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# lpad(), rpad(): 패딩 (자릿수 맞추기)
# -----------------------------------------------------------------------------

# lpad(컬럼, 총길이, 채울문자): 왼쪽 패딩
# rpad(컬럼, 총길이, 채울문자): 오른쪽 패딩

df_pad = df.select(
    col("emp_id"),
    # 사원번호를 10자리로 (앞에 0 채움)
    lpad(col("emp_id"), 10, "0").alias("padded_id"),
)

print("=== 패딩 ===")
df_pad.show(5)

---

## Part 3: 날짜/시간 함수

### 주요 날짜 함수

| 함수 | 설명 | 예시 |
|------|------|------|
| `current_date()` | 오늘 날짜 | - |
| `to_date()` | 문자열 → 날짜 | `to_date(col, "yyyy-MM-dd")` |
| `date_format()` | 날짜 포맷 변경 | `date_format(col, "yyyy/MM")` |
| `year()`, `month()` | 연/월 추출 | `year(col)` |
| `datediff()` | 날짜 차이 | `datediff(end, start)` |
| `date_add()` | 날짜 더하기 | `date_add(col, 7)` |

In [ ]:
# -----------------------------------------------------------------------------
# to_date(), to_timestamp(): 문자열 → 날짜/시간 변환
# -----------------------------------------------------------------------------

# to_date(컬럼, 포맷): 문자열을 날짜로 변환
# to_timestamp(컬럼, 포맷): 문자열을 타임스탬프로 변환

df_date = df.withColumn(
    "join_date_parsed",
    to_date(col("join_date"), "yyyy-MM-dd")   # 문자열 → DateType
)

print("=== 날짜 파싱 ===")
df_date.select("join_date", "join_date_parsed").show(5)
df_date.printSchema()

In [ ]:
# -----------------------------------------------------------------------------
# year(), month(), dayofmonth(): 날짜 부분 추출
# -----------------------------------------------------------------------------

# year(): 연도 추출
# month(): 월 추출
# dayofmonth(): 일 추출
# dayofweek(): 요일 (1=일요일, 7=토요일)
# hour(), minute(), second(): 시간 부분 추출

df_parts = df_date.select(
    col("join_date_parsed"),
    year(col("join_date_parsed")).alias("year"),             # 연도
    month(col("join_date_parsed")).alias("month"),           # 월
    dayofmonth(col("join_date_parsed")).alias("day"),        # 일
    dayofweek(col("join_date_parsed")).alias("dow"),         # 요일 (1=일)
)

print("=== 날짜 부분 추출 ===")
df_parts.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# date_format(): 날짜 포맷 변경
# -----------------------------------------------------------------------------

# date_format(컬럼, 포맷): 날짜를 지정한 포맷의 문자열로 변환
# 포맷 패턴: yyyy(년), MM(월), dd(일), HH(시), mm(분), ss(초)

df_formatted = df_date.select(
    col("join_date_parsed"),
    # 다양한 포맷으로 변환
    date_format(col("join_date_parsed"), "yyyy/MM/dd").alias("slash_format"),
    date_format(col("join_date_parsed"), "yyyy-MM").alias("year_month"),
    date_format(col("join_date_parsed"), "yyyy년 MM월 dd일").alias("korean"),
    date_format(col("join_date_parsed"), "EEEE").alias("day_name"),  # 요일 이름
)

print("=== 날짜 포맷 변경 ===")
df_formatted.show(5, truncate=False)

In [ ]:
# -----------------------------------------------------------------------------
# date_add(), date_sub(), datediff(): 날짜 연산
# -----------------------------------------------------------------------------

# date_add(컬럼, 일수): 날짜에 일 더하기
# date_sub(컬럼, 일수): 날짜에서 일 빼기
# datediff(end, start): 두 날짜 사이 일수

df_calc = df_date.select(
    col("emp_id"),
    col("join_date_parsed").alias("join_date"),
    # 7일 후
    date_add(col("join_date_parsed"), 7).alias("plus_7_days"),
    # 30일 전
    date_sub(col("join_date_parsed"), 30).alias("minus_30_days"),
    # 오늘까지 며칠?
    datediff(current_date(), col("join_date_parsed")).alias("days_since_join"),
)

print("=== 날짜 연산 ===")
df_calc.show(5)

In [ ]:
# -----------------------------------------------------------------------------
# trunc(): 날짜 자르기 (월초, 연초 등)
# -----------------------------------------------------------------------------

# trunc(컬럼, "단위"): 해당 단위의 시작점으로 자름
# "month" → 해당 월의 1일
# "year" → 해당 연도의 1월 1일
# "week" → 해당 주의 월요일

df_trunc = df_date.select(
    col("join_date_parsed"),
    trunc(col("join_date_parsed"), "month").alias("month_start"),   # 월초
    trunc(col("join_date_parsed"), "year").alias("year_start"),     # 연초
)

print("=== 날짜 자르기 ===")
df_trunc.show(5)

---

## Part 4: Window 함수

### Window 함수란?

```
┌─────────────────────────────────────────────────────────────────┐
│                      Window 함수 개념                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  일반 집계 (groupBy):                                          │
│  ┌─────┬────────┐    groupBy("dept")   ┌─────┬─────┐           │
│  │ dept│ salary │    ───────────────→  │ dept│ sum │           │
│  │ A   │ 100    │                      │ A   │ 300 │  행 감소  │
│  │ A   │ 200    │                      │ B   │ 500 │           │
│  │ B   │ 500    │                      └─────┴─────┘           │
│  └─────┴────────┘                                               │
│                                                                 │
│  Window 함수:                                                   │
│  ┌─────┬────────┐    Window("dept")    ┌─────┬────────┬─────┐  │
│  │ dept│ salary │    ───────────────→  │ dept│ salary │ sum │  │
│  │ A   │ 100    │                      │ A   │ 100    │ 300 │  │
│  │ A   │ 200    │                      │ A   │ 200    │ 300 │  │
│  │ B   │ 500    │                      │ B   │ 500    │ 500 │  │
│  └─────┴────────┘                      └─────┴────────┴─────┘  │
│                                                                 │
│  ★ 행 수 유지! 각 행에 그룹 집계값 추가                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Window 정의

```python
from pyspark.sql.window import Window

# 파티션(그룹) + 정렬
window_spec = Window.partitionBy("dept").orderBy("salary")

# 파티션만
window_spec = Window.partitionBy("dept")

# 범위 지정 (누적합 등)
window_spec = Window.partitionBy("dept").orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
```

In [ ]:
# -----------------------------------------------------------------------------
# Window 기본: 순위 함수
# -----------------------------------------------------------------------------

from pyspark.sql.window import Window

# Window 정의: 부서별로 그룹화, 급여 내림차순 정렬
window_rank = Window.partitionBy("department").orderBy(col("salary").desc())

# 순위 함수 적용
# row_number(): 동점이어도 순차적 번호 (1, 2, 3, 4...)
# rank(): 동점은 같은 순위, 다음 순위 건너뜀 (1, 1, 3, 4...)
# dense_rank(): 동점은 같은 순위, 순위 안 건너뜀 (1, 1, 2, 3...)
df_ranked = df.filter(col("salary").isNotNull()).withColumn(
    "row_num", row_number().over(window_rank)
).withColumn(
    "rank", rank().over(window_rank)
).withColumn(
    "dense_rank", dense_rank().over(window_rank)
)

print("=== 부서별 급여 순위 ===")
df_ranked.select(
    "department", "name", "salary", "row_num", "rank", "dense_rank"
).orderBy("department", "row_num").show(15)

In [ ]:
# -----------------------------------------------------------------------------
# lag(), lead(): 이전/다음 행 참조
# -----------------------------------------------------------------------------

# lag(컬럼, n): n행 이전 값
# lead(컬럼, n): n행 이후 값
# 시계열 분석, 전일 대비 비교 등에 유용

# 날짜순 정렬된 매출 데이터로 실습
df_sales_parsed = df_sales.withColumn("date_parsed", to_date(col("date")))

# Window: 제품별, 날짜순
window_sales = Window.partitionBy("product").orderBy("date_parsed")

df_with_prev = df_sales_parsed.withColumn(
    "prev_amount", lag("amount", 1).over(window_sales)      # 전일 매출
).withColumn(
    "next_amount", lead("amount", 1).over(window_sales)     # 익일 매출
).withColumn(
    # 전일 대비 증감
    "change", col("amount") - col("prev_amount")
)

print("=== lag/lead: 전일 대비 비교 ===")
df_with_prev.filter(col("product") == "A").select(
    "date_parsed", "product", "amount", "prev_amount", "change"
).show(10)

In [ ]:
# -----------------------------------------------------------------------------
# 누적 합계 / 이동 평균
# -----------------------------------------------------------------------------

# 범위 지정: rowsBetween(시작, 끝)
# unboundedPreceding: 파티션 시작부터
# currentRow: 현재 행까지
# unboundedFollowing: 파티션 끝까지
# 숫자: 현재 행 기준 상대 위치 (-3, 0, 2 등)

# 누적 합계: 시작부터 현재까지
window_cumsum = Window.partitionBy("product").orderBy("date_parsed") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 7일 이동 평균: 최근 7일
window_ma7 = Window.partitionBy("product").orderBy("date_parsed") \
    .rowsBetween(-6, Window.currentRow)  # 현재 포함 7일

df_cumsum = df_sales_parsed.withColumn(
    "cumsum", sum("amount").over(window_cumsum)
).withColumn(
    "ma7", spark_round(avg("amount").over(window_ma7), 2)
)

print("=== 누적합 / 7일 이동평균 ===")
df_cumsum.filter(col("product") == "A").select(
    "date_parsed", "product", "amount", "cumsum", "ma7"
).show(15)

In [ ]:
# -----------------------------------------------------------------------------
# 실무 패턴: 그룹별 Top N
# -----------------------------------------------------------------------------

# 부서별 급여 Top 3 뽑기
# 1. row_number()로 순위 매기기
# 2. filter로 순위 <= 3 필터링

window_top = Window.partitionBy("department").orderBy(col("salary").desc())

df_top3 = df.filter(col("salary").isNotNull()).withColumn(
    "rank", row_number().over(window_top)
).filter(
    col("rank") <= 3
)

print("=== 부서별 급여 Top 3 ===")
df_top3.select("department", "name", "salary", "rank") \
    .orderBy("department", "rank").show()

---

## Part 5: 자주 쓰는 실무 패턴

### 패턴 모음

1. 조건부 집계
2. 전체 대비 비율
3. 데이터 품질 체크
4. 컬럼명 일괄 변경
5. Forward Fill (이전 값으로 채우기)

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 1: 조건부 집계
# -----------------------------------------------------------------------------

# groupBy + when 조합으로 조건별 카운트/합계

conditional_agg = df.groupBy("department").agg(
    count("*").alias("total"),
    # 조건별 카운트
    count(when(col("salary") >= 80000, 1)).alias("high_salary_cnt"),
    count(when(col("salary") < 50000, 1)).alias("low_salary_cnt"),
    # 조건별 합계
    sum(when(col("salary") >= 80000, col("salary"))).alias("high_salary_sum"),
)

print("=== 패턴 1: 조건부 집계 ===")
conditional_agg.show()

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 2: 전체/그룹 대비 비율
# -----------------------------------------------------------------------------

# Window 함수로 그룹 전체 합계를 각 행에 추가
window_dept = Window.partitionBy("department")

df_ratio = df.filter(col("salary").isNotNull()).withColumn(
    "dept_total", sum("salary").over(window_dept)           # 부서 총급여
).withColumn(
    "pct_of_dept", spark_round(col("salary") / col("dept_total") * 100, 2)  # 부서 내 비율
)

print("=== 패턴 2: 부서 내 급여 비율 ===")
df_ratio.select("department", "name", "salary", "dept_total", "pct_of_dept") \
    .orderBy("department", col("salary").desc()).show(10)

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 3: 데이터 품질 체크
# -----------------------------------------------------------------------------

# 한 번에 여러 품질 지표 확인
quality_check = df.agg(
    count("*").alias("total_rows"),
    count("emp_id").alias("emp_id_non_null"),
    countDistinct("emp_id").alias("emp_id_unique"),
    count("salary").alias("salary_non_null"),
    sum(when(col("salary") < 0, 1).otherwise(0)).alias("negative_salary"),
    sum(when(col("department").isNull(), 1).otherwise(0)).alias("dept_null"),
)

print("=== 패턴 3: 데이터 품질 체크 ===")
quality_check.show()

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 4: 컬럼명 일괄 변경 (소문자, 공백→언더스코어)
# -----------------------------------------------------------------------------

# toDF()로 모든 컬럼명 한번에 변경
# 리스트 컴프리헨션으로 변환 규칙 적용

# 예시 DataFrame
df_messy_cols = spark.createDataFrame([
    (1, "A", 100),
    (2, "B", 200),
], ["User ID", "Product Name", "Total Amount"])

print("변경 전:", df_messy_cols.columns)

# 소문자 + 공백을 _로 변환
new_cols = [c.lower().replace(" ", "_") for c in df_messy_cols.columns]
df_clean_cols = df_messy_cols.toDF(*new_cols)

print("변경 후:", df_clean_cols.columns)

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 5: Forward Fill (이전 값으로 NULL 채우기)
# -----------------------------------------------------------------------------

# last(ignorenulls=True)를 Window와 함께 사용
# NULL을 직전 non-null 값으로 채움

df_with_null = spark.createDataFrame([
    (1, "2024-01-01", 100),
    (1, "2024-01-02", None),
    (1, "2024-01-03", None),
    (1, "2024-01-04", 200),
    (1, "2024-01-05", None),
], ["id", "date", "value"])

# Window: 시작~현재까지
window_ff = Window.partitionBy("id").orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_filled = df_with_null.withColumn(
    "value_filled",
    # ignorenulls=True: NULL을 무시하고 마지막 non-null 값 반환
    last("value", ignorenulls=True).over(window_ff)
)

print("=== 패턴 5: Forward Fill ===")
df_filled.show()

In [ ]:
# -----------------------------------------------------------------------------
# 패턴 6: 여러 컬럼에 같은 처리 일괄 적용
# -----------------------------------------------------------------------------

# 리스트 컴프리헨션 + agg로 여러 컬럼에 같은 집계 적용
cols_to_sum = ["salary", "age"]

# 방법: 리스트 컴프리헨션으로 집계 함수 생성
agg_exprs = [sum(c).alias(f"total_{c}") for c in cols_to_sum]
agg_exprs += [avg(c).alias(f"avg_{c}") for c in cols_to_sum]

result = df.agg(*agg_exprs)

print("=== 패턴 6: 여러 컬럼 일괄 집계 ===")
result.show()

---

## 퀴즈

Q1. fillna({"salary": 0, "department": "Unknown"})의 동작은?

- A) 모든 컬럼의 NULL을 0으로 채운다
- B) salary의 NULL은 0, department의 NULL은 Unknown으로 채운다
- C) salary와 department가 모두 NULL인 행만 채운다
- D) 에러가 발생한다

<details>
<summary>정답 보기</summary>

**정답: B) salary의 NULL은 0, department의 NULL은 Unknown으로 채운다**

딕셔너리로 컬럼별 다른 값을 지정할 수 있습니다.

</details>

---

Q2. row_number()와 rank()의 차이점은?

- A) row_number는 정렬 필요, rank는 정렬 불필요
- B) row_number는 동점도 순차 번호, rank는 동점이면 같은 순위
- C) row_number는 그룹별, rank는 전체
- D) 차이 없음

<details>
<summary>정답 보기</summary>

**정답: B) row_number는 동점도 순차 번호, rank는 동점이면 같은 순위**

동점 데이터 예시:
- row_number: 1, 2, 3, 4
- rank: 1, 1, 3, 4 (동점은 같은 순위, 다음 건너뜀)
- dense_rank: 1, 1, 2, 3 (동점은 같은 순위, 순위 안 건너뜀)

</details>

---

Q3. 7일 이동 평균을 구하려면 rowsBetween의 값은?

- A) `rowsBetween(0, 6)`
- B) `rowsBetween(-6, 0)`
- C) `rowsBetween(-7, -1)`
- D) `rowsBetween(1, 7)`

<details>
<summary>정답 보기</summary>

**정답: B) `rowsBetween(-6, 0)`**

-6은 현재 행 기준 6행 전, 0은 현재 행입니다.
현재 행 포함 최근 7일 = 현재행 + 이전 6행 = -6 ~ 0

</details>

---

Q4. coalesce(col1, col2, lit("default"))의 동작은?

- A) col1이 NULL이면 col2, col2도 NULL이면 "default"
- B) col1과 col2를 합친다
- C) col1, col2, "default" 중 가장 큰 값 반환
- D) col1, col2의 NULL 개수 반환

<details>
<summary>정답 보기</summary>

**정답: A) col1이 NULL이면 col2, col2도 NULL이면 "default"**

coalesce는 왼쪽부터 확인하여 첫 번째 non-null 값을 반환합니다.

</details>

---

## 과제: 종합 실습

### Step 1: 데이터 정리

직원 데이터에서:
1. name 컬럼의 앞뒤 공백 제거
2. salary가 NULL인 경우 0으로 채우기
3. department가 NULL인 경우 "Unassigned"로 채우기

<details>
<summary>힌트</summary>

`trim()`, `fillna({"salary": 0, "department": "Unassigned"})`

</details>

### Step 2: 날짜 파생 컬럼

join_date를 파싱하여:
1. 입사 연도(join_year) 추출
2. 입사 월(join_month) 추출
3. 근속일수(tenure_days) 계산

<details>
<summary>힌트</summary>

`to_date()`, `year()`, `month()`, `datediff(current_date(), ...)`

</details>

### Step 3: 부서별 급여 순위

Window 함수를 사용하여:
1. 부서별 급여 순위(rank) 추가
2. 부서별 급여 비율(%) 추가

<details>
<summary>힌트</summary>

`Window.partitionBy("department").orderBy(col("salary").desc())`

</details>

### 보너스: 부서별 Top 2 직원만 추출

부서별 급여 상위 2명만 필터링하세요.

---

## 핵심 요약

### NULL 처리

```python
df.dropna()                          # NULL 행 제거
df.dropna(subset=["col"])            # 특정 컬럼 NULL 제거
df.fillna(0)                         # NULL → 0
df.fillna({"col1": 0, "col2": "X"})  # 컬럼별 다른 값
coalesce(col1, col2, lit("기본값"))   # 첫 번째 non-null
```

### 문자열 함수

```python
trim(col)                            # 공백 제거
upper(col) / lower(col)              # 대소문자
concat_ws("_", col1, col2)           # 구분자로 합치기
split(col, "@")[0]                   # 분리 후 인덱싱
regexp_replace(col, 패턴, 대체)       # 정규식 치환
```

### 날짜 함수

```python
to_date(col, "yyyy-MM-dd")           # 문자열 → 날짜
date_format(col, "yyyy/MM")          # 포맷 변경
year(col), month(col)                # 부분 추출
datediff(end, start)                 # 날짜 차이
date_add(col, 7)                     # 날짜 더하기
```

### Window 함수

```python
window = Window.partitionBy("그룹").orderBy("정렬")

row_number().over(window)            # 순위 (1,2,3,4)
rank().over(window)                  # 순위 (동점 같은 순위)
lag("col", 1).over(window)           # 이전 행
lead("col", 1).over(window)          # 다음 행
sum("col").over(window_cumsum)       # 누적합
avg("col").over(window_ma7)          # 이동평균
```

In [ ]:
# 세션 종료
# spark.stop()
print("\n4교시 완료! PySpark 기초 API 학습 완료!")
print("다음 단계: Spark Structured Streaming")

---


# Day 11 - 5교시: Spark Structured Streaming

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- Spark Structured Streaming의 개념과 특징을 이해할 수 있다
- Kafka에서 실시간으로 데이터를 읽어올 수 있다
- 윈도우 기반 집계(시간 윈도우)를 구현할 수 있다
- 스트리밍 결과를 콘솔/파일/Kafka로 출력할 수 있다

---

## 핵심 개념 1: 배치 처리 vs 스트리밍 처리

### 두 가지 데이터 처리 방식

![](https://k21academy.com/wp-content/uploads/2020/11/BatchProcessingStreamProcessing_Diagram-02.png)

데이터 처리 방식은 크게 **배치(Batch)**와 **스트리밍(Streaming)**으로 나뉩니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                    배치 처리 (Batch Processing)                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   어제의 데이터  ─────────→  처리  ─────────→  결과              │
│   (파일/DB에 저장)          (한번에)           (다음날 확인)       │
│                                                                 │
│   예시:                                                         │
│   - 매일 밤 12시에 전날 판매 리포트 생성                          │
│   - 매주 월요일에 주간 사용자 통계 계산                           │
│   - 매월 1일에 월간 정산 처리                                     │
│                                                                 │
│   특징:                                                         │
│   - 데이터가 모두 준비된 후 처리                                  │
│   - 높은 처리량 (throughput) 가능                                │
│   - 지연 시간(latency)이 김 (시간~일)                             │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                 스트리밍 처리 (Stream Processing)                │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   실시간 데이터  ───→  처리  ───→  결과  ───→  처리  ───→  결과   │
│   (끊임없이 도착)    (즉시)       (즉시)     (즉시)       (즉시)   │
│                                                                 │
│   예시:                                                         │
│   - 실시간 대시보드 (트래픽 모니터링)                             │
│   - 이상 거래 탐지 (1초 내 알림)                                  │
│   - 실시간 추천 시스템                                            │
│                                                                 │
│   특징:                                                         │
│   - 데이터가 도착하는 즉시 처리                                   │
│   - 낮은 지연 시간(latency) (밀리초~초)                           │
│   - 24시간 계속 실행                                              │
└─────────────────────────────────────────────────────────────────┘
```

### 언제 어떤 방식을 사용하나요?

| 상황 | 배치 | 스트리밍 |
|------|------|----------|
| 데이터 분석 리포트 | O | |
| 실시간 대시보드 | | O |
| 월간 정산 | O | |
| 이상 거래 탐지 | | O |
| 머신러닝 모델 학습 | O | |
| 실시간 추천 | | O |
| ETL 파이프라인 | O | O (둘 다 가능) |

## 핵심 개념 2: Micro-batch 처리 방식

### Spark Structured Streaming의 동작 원리

Spark Structured Streaming은 **Micro-batch** 방식으로 스트리밍을 처리합니다.
데이터를 작은 배치 단위로 나누어 처리하는 방식입니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                        Micro-batch 처리                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  시간 →                                                         │
│                                                                 │
│  데이터 도착:  ●●●●●  ●●●●  ●●●●●●  ●●●  ●●●●●●●               │
│               ↓      ↓       ↓      ↓     ↓                   │
│  배치 단위:  [배치1] [배치2] [배치3] [배치4] [배치5]              │
│              ↓      ↓       ↓      ↓      ↓                   │
│  처리:      ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐                │
│            │처리1│ │처리2│ │처리3│ │처리4│ │처리5│                │
│            └────┘ └────┘ └────┘ └────┘ └────┘                │
│              ↓      ↓      ↓      ↓      ↓                   │
│  결과 출력:   결과1   결과2   결과3   결과4   결과5                 │
│                                                                 │
│  trigger: processingTime="5 seconds" → 5초마다 배치 처리         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Micro-batch의 장점

| 장점 | 설명 |
|------|------|
| **Exactly-once 보장** | 데이터 중복/손실 없이 정확히 한 번 처리 |
| **배치 코드 재사용** | DataFrame API를 그대로 사용 |
| **장애 복구 용이** | 체크포인트로 상태 저장/복구 |
| **간편한 개발** | 스트리밍 복잡성을 추상화 |

### 트리거(Trigger) 옵션

| 트리거 | 설명 | 사용 예시 |
|--------|------|----------|
| `processingTime="5 seconds"` | 5초마다 배치 처리 | 실시간 대시보드 |
| `processingTime="1 minute"` | 1분마다 배치 처리 | 준실시간 집계 |
| `once=True` | 한 번만 처리 | 배치 스타일 스트리밍 |
| `availableNow=True` | 현재까지 데이터 한 번 처리 | 백필(backfill) |

## 핵심 개념 3: Watermark (워터마크)

### 지연 데이터 문제

실시간 시스템에서는 네트워크 지연 등으로 데이터가 늦게 도착할 수 있습니다.
Watermark는 "얼마나 늦은 데이터까지 허용할 것인가"를 정의합니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                      Watermark 동작 원리                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  실제 시간:     10:00   10:05   10:10   10:15   10:20           │
│                   │       │       │       │       │             │
│  데이터 도착:    ●       ●       ●       ●       ●             │
│                 (A)     (B)     (C)     (D)     (E)             │
│                                                                 │
│  이벤트 시간:   10:00   10:04   10:08   10:12   10:03 (늦음!)   │
│                   │       │       │       │       │             │
│                                                                 │
│  Watermark = 10분 설정:                                         │
│  - 현재 시간 10:15일 때                                          │
│  - Watermark = 10:15 - 10분 = 10:05                             │
│  - 이벤트 시간 10:03인 (E)는 허용 (10:03 > 10:05? NO → 버림)    │
│                                                                 │
│  결론: 10분 이상 늦은 데이터는 무시됨                             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Watermark 설정 방법

```python
df.withWatermark("event_time", "10 minutes")
```

| 파라미터 | 의미 |
|----------|------|
| `"event_time"` | 이벤트 시간 컬럼명 |
| `"10 minutes"` | 허용할 최대 지연 시간 |

### Watermark 설정 기준

| 값 | 상황 |
|----|------|
| 작은 값 (1분) | 네트워크가 안정적, 빠른 결과 필요 |
| 큰 값 (1시간) | 네트워크가 불안정, 데이터 손실 최소화 |
| 중간 값 (10분) | 일반적인 상황 |

## 핵심 개념 4: 윈도우 집계 (Window Aggregation)

### 시간 윈도우란?

스트리밍 데이터를 시간 단위로 묶어서 집계하는 방법입니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                   Tumbling Window (텀블링 윈도우)                │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  시간 →                                                         │
│  |-------|-------|-------|-------|                              │
│  0분    5분    10분   15분   20분                               │
│                                                                 │
│  [Window 1]                                                     │
│  0~5분   ●●●●●  → 집계 결과 1                                   │
│                                                                 │
│          [Window 2]                                             │
│          5~10분  ●●●●  → 집계 결과 2                            │
│                                                                 │
│                   [Window 3]                                    │
│                   10~15분  ●●●●●●  → 집계 결과 3                │
│                                                                 │
│  특징:                                                          │
│  - 윈도우가 겹치지 않음                                          │
│  - 각 데이터는 하나의 윈도우에만 속함                              │
│  - 가장 일반적으로 사용                                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                    Sliding Window (슬라이딩 윈도우)              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  시간 →                                                         │
│  |-----|-----|-----|-----|-----|-----|                          │
│  0분  2분  4분  6분  8분  10분 12분                             │
│                                                                 │
│  [Window 1: 0~5분]                                              │
│      [Window 2: 2~7분]                                          │
│          [Window 3: 4~9분]                                      │
│              [Window 4: 6~11분]                                 │
│                                                                 │
│  특징:                                                          │
│  - 윈도우가 겹침 (overlap)                                       │
│  - 하나의 데이터가 여러 윈도우에 속할 수 있음                      │
│  - 더 세밀한 분석 가능                                           │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 윈도우 설정 방법

```python
# Tumbling Window (5분)
window(col("event_time"), "5 minutes")

# Sliding Window (5분 윈도우, 1분 슬라이드)
window(col("event_time"), "5 minutes", "1 minute")
```

## Structured Streaming이란?

### 개념

```
배치 처리 (기존 방식):
┌─────────────────────────────────────────┐
│ 데이터 전체 로드 → 처리 → 결과 출력     │
└─────────────────────────────────────────┘
                   ↓
스트리밍 처리 (Structured Streaming):
┌─────────────────────────────────────────┐
│ 데이터 조금씩 → 처리 → 결과 지속 업데이트│
│    ↓              ↓              ↓       │
│ micro-batch 1  micro-batch 2   ...      │
└─────────────────────────────────────────┘
```

### 핵심 아이디어: 무한 테이블

```
시간 →

[기존 데이터]  [새 데이터 도착]  [또 새 데이터]
    ↓              ↓                ↓
┌────────┐    ┌────────┐       ┌────────┐
│ row 1  │    │ row 1  │       │ row 1  │
│ row 2  │    │ row 2  │       │ row 2  │
│        │    │ row 3  │ ←new  │ row 3  │
│        │    │ row 4  │ ←new  │ row 4  │
│        │    │        │       │ row 5  │ ←new
└────────┘    └────────┘       └────────┘

"무한히 늘어나는 테이블"처럼 취급
→ 같은 DataFrame API로 배치/스트리밍 모두 처리
```

---

## Part 1: Kafka → Spark 연결 (30분)

### Step 1: 빈칸 채우기

In [ ]:
# Step 1: Kafka Streaming 빈칸 채우기
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)

# -----------------------------------------------------------------------------
# SparkSession 생성 (Kafka 연동을 위한 패키지 포함)
# -----------------------------------------------------------------------------
spark = (
    SparkSession.builder.appName("Day11-Streaming")
    .master("spark://spark-master:7077")
    # TODO 1: Kafka 연동을 위한 패키지 추가
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:_____")
    .getOrCreate()
)

# -----------------------------------------------------------------------------
# API 이벤트 스키마 정의
# -----------------------------------------------------------------------------
# TODO 2: 스키마 정의 (StructType 사용)
schema = StructType(
    [
        StructField("request_id", _____, True),
        StructField("user_id", _____, True),
        StructField("endpoint", _____, True),
        StructField("method", _____, True),
        StructField("status_code", _____, True),
        StructField("response_time_ms", _____, True),
        StructField("timestamp", _____, True),
    ]
)

# -----------------------------------------------------------------------------
# Kafka에서 스트림 읽기
# -----------------------------------------------------------------------------
# TODO 3: readStream 설정
df_raw = (
    spark._____.format("kafka")  # readStream 시작
    .option("kafka.bootstrap.servers", "_____")  # Kafka 주소
    .option("subscribe", "_____")  # 토픽 이름
    .option("startingOffsets", "latest")  # 최신 데이터부터 읽기
    .load()
)

# TODO 4: JSON 파싱
df_parsed = df_raw.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

df_parsed.printSchema()

<details>
<summary>TODO 1 힌트 보기 (Kafka 패키지 버전)</summary>

spark-sql-kafka 패키지 버전은 Spark 버전과 일치해야 합니다.
Spark 3.5.8을 사용하므로 패키지 버전도 3.5.8입니다.

```python
.config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")
```

버전 형식: `org.apache.spark:spark-sql-kafka-0-10_2.12:<spark버전>`

</details>

<details>
<summary>TODO 2 힌트 보기 (스키마 정의)</summary>

각 필드의 데이터 타입을 지정합니다:
- 문자열: `StringType()`
- 정수: `IntegerType()`

```python
StructField("request_id", StringType(), True)
StructField("status_code", IntegerType(), True)
```

</details>

<details>
<summary>TODO 3 힌트 보기 (readStream 설정)</summary>

스트림을 읽을 때는 `spark.readStream`을 사용합니다.
Kafka 서버 주소와 토픽 이름을 지정합니다.

```python
spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "api-events")
```

</details>

<details>
<summary>전체 정답 보기</summary>

```python
# TODO 1: 패키지 버전 3.5.8
.config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")

# TODO 2: 스키마 정의
schema = StructType([
    StructField("request_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("response_time_ms", IntegerType(), True),
    StructField("timestamp", StringType(), True),
])

# TODO 3: readStream 설정
df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "api-events")
    .option("startingOffsets", "latest")
    .load()
)
```

</details>

### Step 2: 완성된 Kafka → Spark 연결

In [ ]:
# Step 2: 완성된 Kafka → Spark 연결
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
)

# -----------------------------------------------------------------------------
# SparkSession 생성
# -----------------------------------------------------------------------------
spark = (
    SparkSession.builder.appName("Day11-Streaming")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")
    .config("spark.sql.streaming.checkpointLocation", "/data/checkpoints")
    .getOrCreate()
)

print("SparkSession 생성 완료!")
print(f"  App ID: {spark.sparkContext.applicationId}")

# -----------------------------------------------------------------------------
# API 이벤트 스키마 정의
# -----------------------------------------------------------------------------
schema = StructType(
    [
        StructField("request_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("endpoint", StringType(), True),
        StructField("method", StringType(), True),
        StructField("status_code", IntegerType(), True),
        StructField("response_time_ms", IntegerType(), True),
        StructField("timestamp", StringType(), True),
    ]
)

print("\n스키마 정의:")
print(schema)

# -----------------------------------------------------------------------------
# Kafka에서 스트림 읽기
# -----------------------------------------------------------------------------
df_raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "api-events")
    .option("startingOffsets", "latest")
    .load()
)

print("\nKafka 스트림 연결 완료!")
print(f"  소스 스키마: {df_raw.schema.simpleString()}")

# -----------------------------------------------------------------------------
# JSON 파싱 및 타입 변환
# -----------------------------------------------------------------------------
df_parsed = (
    df_raw.select(from_json(col("value").cast("string"), schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("timestamp")))
)

print("\n파싱된 스키마:")
df_parsed.printSchema()

**예상 출력**:

```
SparkSession 생성 완료!
  App ID: app-20260118-...

스키마 정의:
StructType([StructField('request_id', StringType(), True), ...])

Kafka 스트림 연결 완료!
  소스 스키마: struct<key:binary,value:binary,topic:string,...>

파싱된 스키마:
root
 |-- request_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- endpoint: string (nullable = true)
 |-- method: string (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- response_time_ms: integer (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
```

---

## Part 2: 콘솔 출력 스트리밍 (30분)

### 간단한 콘솔 출력

In [ ]:
# 콘솔 출력 스트리밍 (터미널에서 실행)
# 참고: Jupyter에서는 스트리밍 쿼리가 제한적으로 동작합니다.
#       실제 실행은 Python 스크립트로 하는 것이 좋습니다.

# -----------------------------------------------------------------------------
# 콘솔 출력 쿼리 (모든 메시지 출력)
# -----------------------------------------------------------------------------
"""
# 이 코드를 src/streaming_console.py로 저장 후 실행

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("Streaming-Console")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")
    .getOrCreate()
)

schema = StructType([
    StructField("request_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("response_time_ms", IntegerType(), True),
    StructField("timestamp", StringType(), True),
])

df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "api-events")
    .option("startingOffsets", "latest")
    .load()
)

df_parsed = (
    df_raw.select(from_json(col("value").cast("string"), schema).alias("data"))
    .select("data.*")
)

# 콘솔 출력
query = (
    df_parsed.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="5 seconds")  # 5초마다 배치 처리
    .start()
)

print("스트리밍 시작! (Ctrl+C로 종료)")
query.awaitTermination()
"""

### writeStream 옵션 설명

| 옵션 | 설명 | 값 예시 |
|------|------|---------|
| `outputMode` | 출력 모드 | `append`, `complete`, `update` |
| `format` | 출력 형식 | `console`, `parquet`, `kafka` |
| `trigger` | 실행 주기 | `processingTime="5 seconds"` |
| `checkpointLocation` | 체크포인트 저장 위치 | `/data/checkpoints` |

### outputMode 비교

| 모드 | 설명 | 사용 상황 |
|------|------|----------|
| `append` | 새 행만 출력 | 집계 없는 단순 처리 |
| `complete` | 전체 결과 출력 | 집계 결과 전체 출력 |
| `update` | 변경된 행만 출력 | 집계 결과 중 변경분만 |

---

## Part 3: 윈도우 집계 (40분)

### 시간 윈도우란?

```
시간 →
|-------|-------|-------|-------|
0분    5분    10분   15분   20분

5분 윈도우:
[Window 1: 0~5분]
        [Window 2: 5~10분]
                 [Window 3: 10~15분]
                          [Window 4: 15~20분]

각 윈도우별로 집계:
- 요청 수
- 평균 응답시간
- 에러율
```

### Step 2: 완성된 윈도우 집계

In [ ]:
# 5분 윈도우 집계 (터미널에서 실행)
"""
# 이 코드를 src/streaming_window.py로 저장 후 실행

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, window, count, avg, sum as spark_sum, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("Streaming-Window")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")
    .config("spark.sql.streaming.checkpointLocation", "/data/checkpoints/window")
    .getOrCreate()
)

schema = StructType([
    StructField("request_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("response_time_ms", IntegerType(), True),
    StructField("timestamp", StringType(), True),
])

df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "api-events")
    .option("startingOffsets", "latest")
    .load()
)

df_parsed = (
    df_raw.select(from_json(col("value").cast("string"), schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("timestamp")))
)

# 5분 윈도우 집계
df_windowed = (
    df_parsed
    .withWatermark("event_time", "10 minutes")  # 늦게 도착하는 데이터 허용
    .groupBy(
        window(col("event_time"), "5 minutes"),  # 5분 윈도우
        col("endpoint")
    )
    .agg(
        count("request_id").alias("request_count"),
        avg("response_time_ms").alias("avg_response_time"),
        spark_sum(when(col("status_code") >= 400, 1).otherwise(0)).alias("error_count"),
    )
    .withColumn("error_rate", col("error_count") / col("request_count") * 100)
)

# 콘솔 출력
query = (
    df_windowed.writeStream
    .outputMode("update")  # 윈도우 집계는 update 모드
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="10 seconds")
    .start()
)

print("5분 윈도우 집계 시작!")
query.awaitTermination()
"""

**예상 출력** (Producer가 메시지 전송 중일 때):

```
-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+---------------+-------------+-----------------+-----------+----------+
|window                                    |endpoint       |request_count|avg_response_time|error_count|error_rate|
+------------------------------------------+---------------+-------------+-----------------+-----------+----------+
|{2026-01-18 10:00:00, 2026-01-18 10:05:00}|/api/products  |          156|            245.3|         39|      25.0|
|{2026-01-18 10:00:00, 2026-01-18 10:05:00}|/api/users     |          142|            267.1|         35|      24.6|
|{2026-01-18 10:00:00, 2026-01-18 10:05:00}|/api/orders    |          138|            251.8|         33|      23.9|
+------------------------------------------+---------------+-------------+-----------------+-----------+----------+
```

---

## Step 3: 지시사항

### 과제 1: 윈도우 1분으로 변경

5분 윈도우를 1분 윈도우로 변경하고, 더 세밀한 모니터링을 구현하세요.

<details>
<summary>힌트 보기</summary>

`window()` 함수의 두 번째 인자를 변경합니다.

```python
window(col("event_time"), "1 minute")
```

</details>

<details>
<summary>정답 보기</summary>

```python
df_windowed = (
    df_parsed
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),  # 5 minutes → 1 minute
        col("endpoint")
    )
    .agg(
        count("request_id").alias("request_count"),
        avg("response_time_ms").alias("avg_response_time"),
    )
)
```

</details>

### 과제 2: Parquet 파일 저장

집계 결과를 Parquet 파일로 저장하세요.

<details>
<summary>힌트 보기</summary>

`writeStream`의 `format`을 `"parquet"`로 변경하고,
`path` 옵션으로 저장 경로를 지정합니다.
집계 결과를 파일로 저장할 때는 `append` 모드를 사용합니다.

</details>

<details>
<summary>정답 보기</summary>

```python
query = (
    df_windowed.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", "/data/output/api_stats")
    .option("checkpointLocation", "/data/checkpoints/parquet")
    .trigger(processingTime="1 minute")
    .start()
)
```

</details>

### 과제 3: 에러만 필터링 집계

status_code가 400 이상인 에러만 필터링하여 실시간 에러 모니터링을 구현하세요.

<details>
<summary>힌트 보기</summary>

`filter()` 메서드로 에러만 걸러낸 후 집계합니다.

```python
df_errors = df_parsed.filter(col("status_code") >= 400)
```

</details>

<details>
<summary>정답 보기</summary>

```python
df_errors = df_parsed.filter(col("status_code") >= 400)

df_error_stats = (
    df_errors
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("endpoint"),
        col("status_code")
    )
    .agg(count("request_id").alias("error_count"))
)
```

</details>

In [ ]:
# 모범 답안: 에러 모니터링 스트리밍
"""
# src/streaming_errors.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, window, count
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("Streaming-Errors")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")
    .config("spark.sql.streaming.checkpointLocation", "/data/checkpoints/errors")
    .getOrCreate()
)

schema = StructType([
    StructField("request_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("response_time_ms", IntegerType(), True),
    StructField("timestamp", StringType(), True),
])

df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "api-events")
    .option("startingOffsets", "latest")
    .load()
)

df_parsed = (
    df_raw.select(from_json(col("value").cast("string"), schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("timestamp")))
)

# 에러만 필터링
df_errors = df_parsed.filter(col("status_code") >= 400)

# 1분 윈도우 에러 집계
df_error_stats = (
    df_errors
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("endpoint"),
        col("status_code")
    )
    .agg(count("request_id").alias("error_count"))
    .orderBy(col("error_count").desc())
)

# 콘솔 출력
query = (
    df_error_stats.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .trigger(processingTime="10 seconds")
    .start()
)

print("에러 모니터링 시작!")
print("  - 1분 윈도우")
print("  - status_code >= 400 필터링")
query.awaitTermination()
"""

---

## 보너스: Kafka로 결과 전송

집계 결과를 다른 Kafka 토픽으로 전송하여 다운스트림 시스템에서 활용할 수 있습니다.

In [ ]:
# 보너스: Kafka Sink
"""
# src/streaming_kafka_sink.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, window, count, avg, to_json, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("Streaming-KafkaSink")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8")
    .config("spark.sql.streaming.checkpointLocation", "/data/checkpoints/kafka-sink")
    .getOrCreate()
)

schema = StructType([
    StructField("request_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("endpoint", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("response_time_ms", IntegerType(), True),
    StructField("timestamp", StringType(), True),
])

df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "api-events")
    .option("startingOffsets", "latest")
    .load()
)

df_parsed = (
    df_raw.select(from_json(col("value").cast("string"), schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("timestamp")))
)

# 1분 윈도우 집계
df_windowed = (
    df_parsed
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("endpoint")
    )
    .agg(
        count("request_id").alias("request_count"),
        avg("response_time_ms").alias("avg_response_time"),
    )
)

# Kafka로 출력 (JSON 형식)
df_output = (
    df_windowed
    .select(
        col("endpoint").alias("key"),  # endpoint를 key로
        to_json(struct("*")).alias("value")  # 전체를 JSON value로
    )
)

query = (
    df_output.writeStream
    .outputMode("update")
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "api-events-aggregated")  # 결과 토픽
    .trigger(processingTime="30 seconds")
    .start()
)

print("집계 결과를 Kafka로 전송 중!")
print("  소스 토픽: api-events")
print("  결과 토픽: api-events-aggregated")
query.awaitTermination()
"""

---

## 핵심 요약

### Structured Streaming 핵심 패턴

```python
# 1. Kafka에서 읽기
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "my-topic")
    .load()
)

# 2. 처리 (일반 DataFrame API 사용)
df_processed = df.select(...).filter(...).groupBy(...)

# 3. 출력
query = (
    df_processed.writeStream
    .outputMode("update")
    .format("console")  # 또는 "parquet", "kafka"
    .trigger(processingTime="10 seconds")
    .start()
)

query.awaitTermination()
```

### 윈도우 집계 패턴

```python
df.withWatermark("event_time", "10 minutes") \
  .groupBy(
      window(col("event_time"), "5 minutes"),  # 윈도우 크기
      col("group_column")
  ) \
  .agg(
      count("*").alias("count"),
      avg("value").alias("avg_value")
  )
```

### 출력 형식 비교

| 형식 | 용도 | 예시 |
|------|------|------|
| `console` | 개발/디버깅 | 실시간 로그 확인 |
| `parquet` | 데이터 저장 | 분석용 데이터 레이크 |
| `kafka` | 이벤트 전달 | 다운스트림 시스템 연동 |
| `memory` | 테스트 | 임시 테이블로 쿼리 |

### 버전 정보

| 컴포넌트 | 버전 |
|----------|------|
| Spark | 3.5.8 |
| spark-sql-kafka | 3.5.8 |
| Kafka | 4.1.1 |

---

## 다음 시간 예고

**6교시: 정리 및 보너스**

- 전체 아키텍처 리뷰
- 각 도구의 역할 정리
- 대안 도구 비교 (Polars, DuckDB, Redis Streams)
- 보너스 과제 안내

In [ ]:
# 세션 정리 (필요시)
# spark.stop()

---


# Day 11 - 6교시: 정리 및 보너스

---

## 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- 오늘 구축한 데이터 파이프라인의 전체 아키텍처를 설명할 수 있다
- Kafka, Spark의 역할과 선택 이유를 설명할 수 있다
- 대안 도구들(Polars, DuckDB, Redis Streams)의 트레이드오프를 이해할 수 있다
- 이후 커리큘럼(ELK, 데이터 웨어하우스)과의 연결점을 파악할 수 있다

---

## 사용 버전 정보

이 교안에서 사용한 도구들의 버전 정보입니다:

| 도구 | 버전 | 릴리스 | 비고 |
|------|------|--------|------|
| **Apache Kafka** | 4.1.1 | 2025년 11월 | KRaft 모드 기본 |
| **Apache Spark** | 3.5.8 | 2026년 1월 | 최신 안정 버전 |
| **PySpark** | 3.5.8 | 2026년 1월 | Spark와 버전 일치 |
| **spark-sql-kafka** | 3.5.8 | 2026년 1월 | Spark와 버전 일치 |
| **Python** | 3.12 | | slim 이미지 |
| **confluent-kafka** | latest | | Python Kafka 클라이언트 |
| **Kafka UI** | latest | | 웹 기반 모니터링 |

---

## Part 1: 전체 아키텍처 리뷰 (15분)

### 오늘 구축한 파이프라인

```
┌─────────────────────────────────────────────────────────────────────┐
│                        API Gateway 모니터링 시스템                    │
└─────────────────────────────────────────────────────────────────────┘

┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  API Events  │────→│    Kafka     │────→│    Spark     │
│  (Producer)  │     │              │     │  Streaming   │
└──────────────┘     └──────────────┘     └──────────────┘
                           │                     │
                           ↓                     ↓
                     ┌──────────┐         ┌──────────────┐
                     │ Kafka UI │         │  Parquet /   │
                     │ 모니터링 │         │  Kafka Sink  │
                     └──────────┘         └──────────────┘
```

### 각 컴포넌트의 역할

| 컴포넌트 | 역할 | 오늘 배운 것 |
|---------|------|-------------|
| **Producer** | 이벤트 생성 및 전송 | confluent-kafka, JSON 직렬화 |
| **Kafka** | 이벤트 버퍼링 및 분배 | KRaft 모드, 파티션, Consumer Group |
| **Spark** | 실시간/배치 데이터 처리 | DataFrame, Structured Streaming |
| **Kafka UI** | 클러스터 모니터링 | 토픽, 메시지, Consumer Lag |
| **Spark UI** | 작업 모니터링 | Job, Stage, Task |

### 데이터 흐름

```
1. 이벤트 발생
   API Gateway에서 요청/응답 정보 생성
   ↓
2. Kafka로 전송
   Producer가 JSON 직렬화 후 api-events 토픽에 전송
   파티션별로 분산 저장
   ↓
3. Spark에서 실시간 처리
   Structured Streaming으로 Kafka 구독
   윈도우 기반 집계 (5분/1분)
   ↓
4. 결과 출력
   콘솔: 실시간 모니터링
   Parquet: 분석용 저장
   Kafka: 다운스트림 시스템 연동
```

---

## Part 2: 도구별 역할 정리 (15분)

### Kafka: "왜 Kafka를 쓰나요?"

**문제 상황**:
```
[직접 연결 방식]
Producer → Consumer 직접 전송

문제점:
- Consumer가 죽으면? 데이터 유실
- Consumer가 느리면? Producer 대기
- 여러 Consumer가 필요하면? 복잡해짐
```

**Kafka 해결책**:
```
Producer → Kafka → Consumer

장점:
- Consumer가 죽어도 데이터 보존 (영속성)
- Producer/Consumer 독립적 확장 (디커플링)
- 여러 Consumer Group 지원 (팬아웃)
- 재처리 가능 (오프셋 관리)
```

### Spark: "왜 Spark를 쓰나요?"

**문제 상황**:
```
[Pandas 한계]
데이터 100만 건: 5초
데이터 1000만 건: 50초... 메모리 부족!
데이터 1억 건: Out of Memory
```

**Spark 해결책**:
```
[분산 처리]
데이터 100만 건: Worker 1대 → 3초
데이터 1000만 건: Worker 5대 → 6초
데이터 1억 건: Worker 50대 → 60초

스케일 아웃으로 처리량 선형 증가!
```

### 오늘 측정한 성능

| 지표 | 결과 | 의미 |
|------|------|------|
| Producer 처리량 | ~50,000 records/sec | 초당 5만 건 Kafka 전송 |
| Pandas vs Spark (100만건) | Spark가 1.5배 빠름 | 대용량에서 Spark 유리 |
| 파티션 1 vs 4 | 15% 처리량 향상 | 병렬화의 효과 |

---

## Part 3: 대안 도구 비교 (15분)

### "항상 Kafka + Spark가 정답인가요?"

아닙니다! 상황에 따라 더 적합한 도구가 있습니다.

### 대안 도구 비교표

| 도구 | vs | 장점 | 단점 | 언제 선택? |
|------|-----|------|------|-----------|
| **Polars** | Spark | 빠름, 설치 간단, API 직관적 | 분산 처리 X | 단일 머신으로 충분할 때 |
| **DuckDB** | Spark | SQL 친화적, 설치 불필요 | 분산 처리 X | 분석 쿼리 위주일 때 |
| **Redis Streams** | Kafka | 간단, 매우 빠름 | 영속성 제한, 기능 제한 | 간단한 실시간 처리 |

### Polars: "빠른 단일 머신 처리"

```python
import polars as pl

# Pandas보다 10~100배 빠름!
df = pl.read_csv("api_events_1m.csv")
result = df.group_by("endpoint").agg([
    pl.count("request_id").alias("count"),
    pl.mean("response_time_ms").alias("avg_time"),
])
```

**선택 기준**:
- 데이터가 단일 머신 메모리에 들어감 (< 100GB)
- 분산 환경 구축이 부담스러움
- 빠른 개발과 반복이 필요

### DuckDB: "SQL로 빠르게 분석"

```python
import duckdb

# SQL로 바로 분석!
result = duckdb.sql('''
    SELECT endpoint,
           COUNT(*) as count,
           AVG(response_time_ms) as avg_time
    FROM 'api_events_1m.csv'
    GROUP BY endpoint
''').df()
```

**선택 기준**:
- SQL에 익숙한 팀
- 분석 쿼리 위주 (OLAP)
- 설치/설정 최소화

### Redis Streams: "초경량 실시간 처리"

```python
import redis

r = redis.Redis()

# Producer
r.xadd("api-events", {"endpoint": "/api/users", "status": "200"})

# Consumer
messages = r.xread({"api-events": "0"}, count=100)
```

**선택 기준**:
- 아주 간단한 실시간 처리
- 이미 Redis 사용 중
- 영속성보다 속도가 중요

### 선택 가이드: 의사결정 트리

```
데이터 규모가 어느 정도인가?
│
├─ < 1GB → Pandas / Polars
│
├─ 1GB ~ 100GB
│   │
│   └─ 분산 환경 가능? ─┬─ Yes → Spark
│                       └─ No  → Polars / DuckDB
│
└─ > 100GB → Spark (분산 필수)

실시간 처리가 필요한가?
│
├─ Yes ─┬─ 복잡한 처리/조인 → Kafka + Spark Streaming
│       └─ 단순 처리      → Redis Streams
│
└─ No → 배치 처리 (Spark / Polars / DuckDB)
```

---

## Part 4: 이후 커리큘럼 연결 (10분)

### 다음에 배울 것들

| 주제 | 연결점 | 배우는 이유 |
|------|--------|------------|
| **ELK Stack** | Kafka → Logstash → Elasticsearch | 로그 검색/시각화 |
| **Data Warehouse** | Spark → Snowflake/BigQuery | 분석용 데이터 저장 |
| **Airflow** | Spark 작업 스케줄링 | 배치 파이프라인 자동화 |

### 오늘 배운 것의 확장

```
[오늘]
API Events → Kafka → Spark → 콘솔/Parquet

[확장 1: ELK]
API Events → Kafka → Logstash → Elasticsearch → Kibana
                                   (검색 가능)    (대시보드)

[확장 2: Data Warehouse]
API Events → Kafka → Spark → Snowflake → BI Tool
                              (SQL 분석)  (시각화)

[확장 3: 완전 자동화]
Airflow가 스케줄링
   ↓
Kafka → Spark (실시간)
Spark (배치) → Data Warehouse → 리포트 자동 생성
```

---

## Part 5: 보너스 과제 안내 (5분)

### 보너스 1: 3대 브로커 클러스터 구성

**목표**: 고가용성을 위한 Kafka 클러스터 구성

**파일**: `docker/multi-broker.yml`

**핵심 설정**:
```yaml
KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka-1:9093,2@kafka-2:9093,3@kafka-3:9093
KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
KAFKA_DEFAULT_REPLICATION_FACTOR: 3
KAFKA_MIN_INSYNC_REPLICAS: 2
```

**실행**:
```bash
docker compose -f multi-broker.yml up -d
```

**확인**:
- Kafka UI에서 3개 브로커 확인
- 토픽의 복제 팩터가 3인지 확인

### 보너스 2: 키 기반 파티셔닝

**목표**: user_id를 키로 사용하여 같은 유저의 메시지가 같은 파티션으로 가도록 설정

<details>
<summary>힌트</summary>

```python
producer.produce(
    topic="api-events",
    key=event["user_id"].encode("utf-8"),  # 키 추가
    value=json.dumps(event).encode("utf-8"),
)
```

</details>

<details>
<summary>모범 답안</summary>

```python
from confluent_kafka import Producer
import json
import random
from datetime import datetime

producer = Producer({"bootstrap.servers": "kafka:9092"})

# 같은 user_id는 같은 파티션으로
for i in range(100):
    event = {
        "request_id": f"REQ_{i:06d}",
        "user_id": f"U{random.randint(1, 10):04d}",  # 10명의 유저
        "endpoint": "/api/test",
        "timestamp": datetime.now().isoformat(),
    }

    producer.produce(
        topic="api-events",
        key=event["user_id"].encode("utf-8"),
        value=json.dumps(event).encode("utf-8"),
    )

producer.flush()
print("키 기반 파티셔닝 완료!")
print("Kafka UI에서 같은 user_id가 같은 파티션에 있는지 확인하세요.")
```

</details>

### 보너스 3: Streaming 결과를 Kafka 토픽으로 전송

**목표**: 집계 결과를 `api-events-aggregated` 토픽으로 전송

<details>
<summary>힌트</summary>

```python
query = (
    df_output.writeStream
    .outputMode("update")
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "api-events-aggregated")
    .start()
)
```

</details>

<details>
<summary>모범 답안</summary>

```python
# 전체 코드는 04_Spark_Streaming.py의 보너스 섹션 참고
from pyspark.sql.functions import to_json, struct

df_output = (
    df_windowed
    .select(
        col("endpoint").alias("key"),
        to_json(struct("*")).alias("value")
    )
)

query = (
    df_output.writeStream
    .outputMode("update")
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "api-events-aggregated")
    .option("checkpointLocation", "/data/checkpoints/kafka-sink")
    .trigger(processingTime="30 seconds")
    .start()
)
```

</details>

---

## 오늘 배운 것 총정리

### 핵심 명령어/코드

```bash
# Docker 환경
docker compose up -d
docker compose ps
docker compose logs -f kafka
docker compose exec python bash
```

```python
# Kafka Producer (confluent-kafka)
from confluent_kafka import Producer
producer = Producer({"bootstrap.servers": "kafka:9092"})
producer.produce(topic, value=json.dumps(data).encode())
producer.flush()

# Kafka Consumer (confluent-kafka)
from confluent_kafka import Consumer
consumer = Consumer({"bootstrap.servers": "kafka:9092", "group.id": "my-group"})
consumer.subscribe(["topic"])
msg = consumer.poll(1.0)

# Spark Batch (PySpark 3.5.8)
spark = SparkSession.builder.master("spark://spark-master:7077").getOrCreate()
df = spark.read.csv("file.csv", header=True)
df.groupBy("col").agg(count("*")).show()

# Spark Streaming (Kafka 연동 - spark-sql-kafka 3.5.8)
spark = SparkSession.builder \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8") \
    .getOrCreate()
df = spark.readStream.format("kafka").option("subscribe", "topic").load()
query = df.writeStream.format("console").start()
```

### 체크리스트

- [ ] Docker Compose로 Kafka + Spark 환경 구성
- [ ] Kafka Producer로 API 이벤트 전송
- [ ] Consumer로 메시지 수신 및 처리
- [ ] 파티션 수에 따른 처리량 변화 측정
- [ ] Pandas vs Spark 성능 비교
- [ ] Structured Streaming으로 실시간 처리
- [ ] 윈도우 집계 구현

---

## 마무리

### 오늘의 핵심 메시지

> "아, 이래서 이런 도구를 쓰는구나!"

- **Kafka**: 데이터 유실 방지, 시스템 간 디커플링
- **Spark**: 대용량 데이터 처리, 실시간 스트리밍
- **도구 선택**: 상황에 따라 적절한 도구를 선택하는 것이 중요

### 다음 시간

ELK Stack (Elasticsearch, Logstash, Kibana)
- 로그 수집 및 검색
- 대시보드 시각화
- Kafka와의 연동

수고하셨습니다!